In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import shutil
import glob
import logging
import math
import xml.etree.ElementTree as ET

from pathlib import Path
from io import StringIO
from typing import Dict, List, Optional, Union
from functools import reduce
import multiprocess as mp
from multiprocess import cpu_count


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from tqdm.contrib.logging import logging_redirect_tqdm
import logging

import parmed
from openbabel import openbabel

from rdkit import Chem
from rdkit.Chem import rdmolops, AllChem, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

from plip.structure.preparation import PDBComplex, logger as PLIP_LOGGER
from plip.exchange.report import StructureReport
from plip.basic import config as PLIP_CONFIG

PLIP_LOGGER.setLevel(logging.ERROR)
PLIP_LOGGER.propagate = False

# Variable Restoration Functions

In [2]:
dataset_dir = '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset'
subdirs = ['biolip2_opt/',
           'pdbbind_opt_poly/',
           'pdbbind_opt_sm/']

def generate_PDBIDs():
    fcsv = os.path.join(dataset_dir, 'PDBIDs.csv')
    df_PDBIDs = pd.read_csv(fcsv)
    PDBIDs = (df_PDBIDs.T).to_numpy()
    PDBIDs = PDBIDs.tolist()
    PDBIDs = [[x for x in row if x is not None and not (isinstance(x, float) and math.isnan(x))]for row in PDBIDs]
    return PDBIDs

def generate_args():
    args = []
    for index, subdir in enumerate(subdirs):
        for PDBID in PDBIDs[index]:
            args.append((
                PDBID,
                subdir,
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_ligand_refined.sdf'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_ligand_refined.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_protein_refined.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_complex.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/interaction.xml')
            ))
    return args

PDBIDs = generate_PDBIDs()
args = generate_args()

# Collect interactions for New Dataset

In [48]:
halogen_dir = os.getcwd()

dataset_dir = '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset'
subdirs = ['biolip2_opt/',
            'pdbbind_opt_poly/',
            'pdbbind_opt_sm/']

PDBIDs = []

os.chdir(dataset_dir)
for index, subdir in enumerate(subdirs):
    print(subdir)
    os.chdir(subdir)
    subdirStruct = []
    for PDBID in os.listdir(os.getcwd()):
        os.chdir(PDBID)
        structures = os.listdir(os.getcwd())
        if len(structures) > 1:
            subdirStruct.append(random.choice(structures))
        else:
            subdirStruct.append(structures[0])
        os.chdir('../')
    PDBIDs.append(subdirStruct)
    os.chdir('../')
    
PDBIDs[0].sort()
PDBIDs[1].sort()
PDBIDs[2].sort()

biolip2_opt/
pdbbind_opt_poly/
pdbbind_opt_sm/
3336
746
14846


In [70]:
df_PDBIDs = pd.DataFrame(PDBIDs)
df_PDBIDs = df_PDBIDs.T
df_PDBIDs.columns = ['biolip2_opt', 'pdbbind_opt_poly', 'pdbbind_opt_sm']
fcsv = os.path.join(dataset_dir, 'PDBIDs.csv')
# df_PDBIDs.to_csv(fcsv, index=False)

## Generate ligand.pdb and complex.pdb Files

In [167]:
def generate_complex_pdb(ligandsdf, ligandpdb, protein, complex, ligandConverter):    
    ligand = openbabel.OBMol()
    ligandConverter.ReadFile(ligand, ligandsdf)
    ligandConverter.WriteFile(ligand, ligandpdb)
    
    if os.path.isfile(complex):
        os.remove(complex)
        
    complex_em = openbabel.OBMol()
    ligand = parmed.load_file(ligandpdb)
    protein = parmed.load_file(protein)
    complex_em = ligand + protein
    complex_em.save(complex, 'pdb')

In [163]:
warnings.filterwarnings("ignore")

def wrapper_complex_pdb(args):
    errs = []

    ligandConverter = openbabel.OBConversion()
    ligandConverter.SetInFormat("sdf")
    ligandConverter.SetOutFormat("pdb")   
    
    try:
        generate_complex_pdb(args[2], args[3], args[4], args[5], ligandConverter)
    except Exception as e:
        errs.append((args[0], args[1], str(e)))
    
    return errs

if __name__ == '__main__':
    args = []
    for index, subdir in enumerate(subdirs):
        for PDBID in PDBIDs[index]:
            args.append((
                PDBID,
                subdir,
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_ligand_refined.sdf'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_ligand_refined.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_protein_refined.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/{PDBID}_complex.pdb'),
                os.path.join(dataset_dir, f'{subdir}{PDBID[:4]}/{PDBID}/interaction.xml')
            ))

    print(len(args))
    logging.basicConfig(level=logging.ERROR)

    # with logging_redirect_tqdm():
    with mp.Pool(cpu_count()) as pool:
        errs = list(tqdm(pool.imap(wrapper_complex_pdb, args), total=len(args)))

    if len(errs) > 0:
        print(len(errs))
        print(errs)

18928



  0%|                                                                                                                                                                                                                      | 0/18928 [19:42<?, ?it/s]

  0%|                                                                                                                                                                                                                      | 0/18928 [19:42<?, ?it/s]

  0%|▏                                                                                                                                                                                                            | 20/18928 [00:02<24:08, 13.06it/s]2024-11-15 13:44:35,794 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file
2024-11-15 13:44:36,033 [WARNING] [structure.py:935] parmed.s

18928
[[('182l_BZF_A_401', 'biolip2_opt/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/biolip2_opt/182l/182l_BZF_A_401/182l_BZF_A_401_complex.pdb exists; not overwriting')], [('183l_DEN_A_400', 'biolip2_opt/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/biolip2_opt/183l/183l_DEN_A_400/183l_DEN_A_400_complex.pdb exists; not overwriting')], [('1a29_TFP_A_153', 'biolip2_opt/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/biolip2_opt/1a29/1a29_TFP_A_153/1a29_TFP_A_153_complex.pdb exists; not overwriting')], [('1a8g_2Z4_A_100', 'biolip2_opt/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/biolip2_opt/1a8g/1a8g_2Z4_A_100/1a8g_2Z4_A_100_complex.pdb exists; not overwriting')], [('1a9r_HPA_A_290', 'biolip2_opt/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/biolip2_opt/1a9r/1a9r_HPA_A_290/1a9r_HPA_A_290_complex.pdb exists; not overwriting')], [('1am1_ADP_A_300', 'biolip2_opt/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/biolip2_opt/1am1/1am1_ADP_A_30

In [181]:
missingLigand = []
missingComplex = []

for arg in tqdm(args):
    if not os.path.isfile(arg[5]):
        missingComplex.append(arg[0])
    if not os.path.isfile(arg[3]):
        missingLigand.append(arg[0])

print(len(missingLigand))
print(len(missingComplex))
print(missingLigand)
print(missingComplex)



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18928/18928 [00:00<00:00, 43857.37it/s]

0
0
[]
[]


# Use PLIP to get protein-ligand interaction profile

In [101]:
def analyze_single_frame(
    pdbpath: os.PathLike, 
    xmlpath: Optional[os.PathLike] = None,
    add_hydrogen: bool = False,
) -> Dict[str, int]:
    """
    Analyze a single ligand-complex structure

    Parameters
    ----------
    pdbpath: os.PathLike
        Path to the pdbfile to be analyzed
    xmlpath: os.PathLike
        Path to the output XML file
    add_hydrogen: bool
        Whether to add hydrogen to the pdb structure. Default is False
    
    Return
    ------
    xmlstr: str
        XML string that contains interaction information
    """
    
    if add_hydrogen:
        PLIP_CONFIG.NOHYDRO = False
    else:
        PLIP_CONFIG.NOHYDRO = True
        
    pdb = PDBComplex()
    pdb.load_pdb(str(pdbpath))
    pdb.analyze()
    report = StructureReport(pdb)

    tmp = sys.stdout
    xmlstr = StringIO()
    sys.stdout = xmlstr
    report.write_xml(True)
    sys.stdout = tmp
    xmlstr.seek(0)
    
    xmlstr = xmlstr.read()
    if xmlpath is not None:
        with open(xmlpath, 'w') as f:
            f.write(xmlstr)
    return xmlstr

In [200]:
def wrapper_plip(args):
    return analyze_single_frame(args[5], args[6])

if __name__ == '__main__':
    warnings.filterwarnings("ignore")

    with logging_redirect_tqdm():
        with mp.Pool(cpu_count()) as pool:
            results = list(tqdm(pool.imap(wrapper_plip, args), total=len(args)))            


  0%|                                                                                                                                                                                                                   | 0/18928 [22:53:40<?, ?it/s]

  0%|                                                                                                                                                                                                                   | 0/18928 [22:53:41<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18928/18928 [10:33<00:00, 29.87it/s]


### Rerun Structures without interaction.xml

In [18]:
missingInteraction = []
logging.basicConfig(level=logging.ERROR)

def checkMissingInteraction(missingInteraction):
    missingInteraction.clear()
    for arg in tqdm(args):
        if not os.path.isfile(arg[6]):
            missingInteraction.append(arg[0])
    print(len(missingInteraction))

checkMissingInteraction(missingInteraction)
print(missingInteraction)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18928/18928 [00:01<00:00, 15797.08it/s]

0
[]


# Collect Halogen interactions

In [65]:
def extract_halogen(xmlpath):
    xmlobj = ET.parse(xmlpath)
    binding_sites = xmlobj.findall("./bindingsite")
    # print(binding_sites)
    bs = [bs for bs in binding_sites if bs.findall("identifiers/longname")[0].text == "UNL"][0]
    halogens = bs.findall("interactions/halogen_bonds/halogen_bond")
    infos = []
    for halogen in halogens:
        info = {}
        for attr in halogen:
            if attr.tag == 'ligcoo' or attr.tag == 'protcoo':
                for s in ['x', 'y', 'z']:
                    info[f'{attr.tag}_{s}'] = attr.find(s).text
            else:
                info[attr.tag] = attr.text
        infos.append(info)
    return infos

def make_identifier(code, resnr, restype, sidechain, donoridx, acceptoridx):
    ifsc = "SC" if sidechain else "BB"
    return f'{code}_{resnr}_{restype}_{ifsc}_{donoridx}_{acceptoridx}'

def make_identifier_from_row(row):
    return make_identifier(row['PDBID'], row['resnr'], row['restype'], eval(row['sidechain']), row['don_idx'], row['acc_idx'])

In [232]:
df = []

for arg in tqdm(args):
    try:
        infos = extract_halogen(arg[6])
        for info in infos:
            info['PDBID'] = arg[0][:4]
            info['full_PDBID'] = arg[0]
            info['subdir'] = arg[1]
            df.append(info)
    except Exception as e:
        errs.append((PDBID, e))

df = pd.DataFrame(df)
cols = ['PDBID'] + ['full_PDBID'] + ['subdir'] + list(df.columns[:-3])
df = df[cols]

# filter bad halogen-bonds: PLIP will identify N-H...F as halogenbonds but it is actually hydrogen bonds
bads = ['Ng+', 'Nam', 'N3+']
df = df.query('acceptortype not in @bads')
# filter terminal residues
df = df.query('not (sidechain == "False" & acceptortype == "O.co2")')

df.index = [make_identifier_from_row(row) for index, row in df.iterrows()]
fcsv = os.path.join(halogen_dir, 'halogen_bond.csv')
df.to_csv(fcsv)
df

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18928/18928 [00:07<00:00, 2388.84it/s]


,PDBID,full_PDBID,subdir,resnr,restype,reschain,resnr_lig,restype_lig,reschain_lig,sidechain,...,don_idx,donortype,acc_idx,acceptortype,ligcoo_x,ligcoo_y,ligcoo_z,protcoo_x,protcoo_y,protcoo_z
1e7c_323_ASP_BB_1_5046,1e7c,1e7c_HLT_A_4001,biolip2_opt/,323,ASP,A,1,UNL,Z,False,...,1,F,5046,O2,26.140,6.427,-7.627,27.030,4.668,-5.249
1fko_235_LEU_BB_1_3875,1fko,1fko_EFZ_A_999,biolip2_opt/,235,LEU,A,1,UNL,Z,False,...,1,Cl,3875,O2,-6.783,-30.162,26.466,-4.685,-32.725,25.222
1ict_109_SER_BB_24_1686,1ict,1ict_T44_C_128,biolip2_opt/,109,SER,A,1,UNL,Z,False,...,24,I,1686,O2,3.065,-40.080,38.719,0.620,-42.067,36.964
1ict_341_SER_BB_22_5226,1ict,1ict_T44_C_128,biolip2_opt/,341,SER,C,1,UNL,Z,False,...,22,I,5226,O2,4.081,-50.520,36.163,2.282,-47.788,36.612
1nav_74_PHE_BB_22_1186,1nav,1nav_IH5_A_600,biolip2_opt/,74,PHE,A,1,UNL,Z,False,...,22,Cl,1186,O2,48.427,21.572,14.983,47.779,18.625,17.120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6u80_251_TRP_BB_26_3877,6u80,6u80_Q1J_B_401,pdbbind_opt_sm/,251,TRP,B,1,UNL,Z,False,...,26,Br,3877,O2,-1.244,9.428,11.786,-2.747,7.260,9.812
6u8b_562_TRP_BB_13_8675,6u8b,6u8b_Q1M_B_401,pdbbind_opt_sm/,562,TRP,B,1,UNL,Z,False,...,13,Br,8675,O2,-31.876,24.189,-1.263,-29.200,23.366,-3.194
6u8l_244_TRP_BB_21_3767,6u8l,6u8l_Q1P_B_401,pdbbind_opt_sm/,244,TRP,B,1,UNL,Z,False,...,21,Br,3767,O2,-23.096,29.660,69.902,-24.680,26.937,68.589
6u8o_244_TRP_BB_19_3769,6u8o,6u8o_Q1G_A_401,pdbbind_opt_sm/,244,TRP,A,1,UNL,Z,False,...,19,Br,3769,O2,-7.576,-20.578,-0.122,-4.380,-20.168,-0.005


In [253]:
df_angle = df[df['don_angle'].astype(float) < 170]
print(len(df_angle))
df_angle = df[df['don_angle'].astype(float) < 160]
print(len(df_angle))
df_angle = df[df['don_angle'].astype(float) < 150]
print(len(df_angle))
df_angle = df[df['don_angle'].astype(float) < 140]
print(len(df_angle))

1164
888
580
205


### Check for Ligands with Residue Names

In [73]:
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

ligandErr = []

with logging_redirect_tqdm(): 
    for arg in tqdm(args):
        ligand = parmed.load_file(arg[3])
        for residue in ligand.residues:
            if not residue.name == 'UNL':
                ligandErr.append(arg + (residue.name,))
            break

2024-11-24 18:36:02,126 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file                                               
2024-11-24 18:36:02,152 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file                                               
2024-11-24 18:36:02,160 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file                                               
2024-11-24 18:36:02,319 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file                                               
2024-11-24 18:36

In [76]:
print(len(ligandErr))
# ligandErrName = [x[7] for x in ligandNameErr]
# print(ligandErrName)
# print(set(ligandErrName))
# ligandErr

1304


In [81]:
for index, arg in enumerate(ligandErr):
    ligandName = os.path.join(os.path.dirname(arg[2]), f'{arg[0]}_ligand_fixed.pdb')
    ligandErr[index] = arg + (ligandName,)

### Generate ligand_refined_UNL.pdb and complex_UNL.pdb with Fixed Names

In [99]:
def fix_ligand_complex(arg, ligandConverter):
    ligandOB = openbabel.OBMol()
    ligandConverter.ReadFile(ligandOB, arg[2])
    ligandConverter.WriteFile(ligandOB, arg[8])
    
    ligandPED = parmed.load_file(arg[8])
    for residue in ligandPED.residues:
        if not residue.name == 'UNL':
            residue.name = 'UNL'
        if not residue.number == 1:
            residue.number = 1
    ligandPED.save(arg[8], overwrite=True)
    
    complex = openbabel.OBMol()
    protein = parmed.load_file(arg[4])
    complex = ligandPED + protein
    complex.save(arg[5], 'pdb', overwrite=True)

In [100]:
def wrapper_fix_lig(ligandErr):
    warnings.filterwarnings("ignore")

    ligandConverter = openbabel.OBConversion()
    ligandConverter.SetInFormat("sdf")
    ligandConverter.SetOutFormat("pdb")
    
    return fix_ligand_complex(ligandErr, ligandConverter)

if __name__ == '__main__':
    warnings.filterwarnings("ignore")

    with logging_redirect_tqdm():
        with mp.Pool(cpu_count()) as pool:
            results = list(tqdm(pool.imap(wrapper_fix_lig, ligandErr), total=len(ligandErr)))  

2024-11-24 19:02:09,083 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file
2024-11-24 19:02:09,083 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file
2024-11-24 19:02:09,083 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file
2024-11-24 19:02:09,084 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file
2024-11-24 19:02:09,085 [WARNING] [structure.py:935] parmed.structure: Could not find the head atom of the next template! Bond pattern may be wrong, which could lead to extra TER cards in a PDB file
2024-

### Regenerate Interaction.xml Files for Fixed Ligands

In [102]:
def wrapper_plip_UNL(ligandErr):
    return analyze_single_frame(ligandErr[5], ligandErr[6])

if __name__ == '__main__':
    warnings.filterwarnings("ignore")

    with logging_redirect_tqdm():
        with mp.Pool(cpu_count()) as pool:
            results = list(tqdm(pool.imap(wrapper_plip_UNL, ligandErr), total=len(ligandErr)))   

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1304/1304 [00:49<00:00, 26.25it/s]


### Extract Halogen Bonds from ligandNameErr

In [115]:
df = []
plsworkpls = []

for arg in tqdm(args):
    try:
        infos = extract_halogen(arg[6])
        for info in infos:
            info['PDBID'] = arg[0][:4]
            info['full_PDBID'] = arg[0]
            info['subdir'] = arg[1]
            df.append(info)
    except Exception as e:
        plsworkpls.append(arg)

print(len(plsworkpls))
# print(LigNumberErr)

df = pd.DataFrame(df)
cols = ['PDBID'] + ['full_PDBID'] + ['subdir'] + list(df.columns[:-3])
df = df[cols]

# filter bad halogen-bonds: PLIP will identify N-H...F as halogenbonds but it is actually hydrogen bonds
bads = ['Ng+', 'Nam', 'N3+']
df = df.query('acceptortype not in @bads')
# filter terminal residues
df = df.query('not (sidechain == "False" & acceptortype == "O.co2")')

df.index = [make_identifier_from_row(row) for index, row in df.iterrows()]
fcsv = os.path.join(halogen_dir, 'halogen_bond.csv')
df.to_csv(fcsv)
df

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18928/18928 [00:07<00:00, 2565.89it/s]

0


,PDBID,full_PDBID,subdir,resnr,restype,reschain,resnr_lig,restype_lig,reschain_lig,sidechain,...,don_idx,donortype,acc_idx,acceptortype,ligcoo_x,ligcoo_y,ligcoo_z,protcoo_x,protcoo_y,protcoo_z
1e7c_323_ASP_BB_1_5046,1e7c,1e7c_HLT_A_4001,biolip2_opt/,323,ASP,A,1,UNL,Z,False,...,1,F,5046,O2,26.140,6.427,-7.627,27.030,4.668,-5.249
1fko_235_LEU_BB_1_3875,1fko,1fko_EFZ_A_999,biolip2_opt/,235,LEU,A,1,UNL,Z,False,...,1,Cl,3875,O2,-6.783,-30.162,26.466,-4.685,-32.725,25.222
1ict_109_SER_BB_24_1686,1ict,1ict_T44_C_128,biolip2_opt/,109,SER,A,1,UNL,Z,False,...,24,I,1686,O2,3.065,-40.080,38.719,0.620,-42.067,36.964
1ict_341_SER_BB_22_5226,1ict,1ict_T44_C_128,biolip2_opt/,341,SER,C,1,UNL,Z,False,...,22,I,5226,O2,4.081,-50.520,36.163,2.282,-47.788,36.612
1nav_74_PHE_BB_22_1186,1nav,1nav_IH5_A_600,biolip2_opt/,74,PHE,A,1,UNL,Z,False,...,22,Cl,1186,O2,48.427,21.572,14.983,47.779,18.625,17.120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6u80_251_TRP_BB_26_3877,6u80,6u80_Q1J_B_401,pdbbind_opt_sm/,251,TRP,B,1,UNL,Z,False,...,26,Br,3877,O2,-1.244,9.428,11.786,-2.747,7.260,9.812
6u8b_562_TRP_BB_13_8675,6u8b,6u8b_Q1M_B_401,pdbbind_opt_sm/,562,TRP,B,1,UNL,Z,False,...,13,Br,8675,O2,-31.876,24.189,-1.263,-29.200,23.366,-3.194
6u8l_244_TRP_BB_21_3767,6u8l,6u8l_Q1P_B_401,pdbbind_opt_sm/,244,TRP,B,1,UNL,Z,False,...,21,Br,3767,O2,-23.096,29.660,69.902,-24.680,26.937,68.589
6u8o_244_TRP_BB_19_3769,6u8o,6u8o_Q1G_A_401,pdbbind_opt_sm/,244,TRP,A,1,UNL,Z,False,...,19,Br,3769,O2,-7.576,-20.578,-0.122,-4.380,-20.168,-0.005


In [89]:
yikes = [('1awi_PRO-PRO_P_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1awi/1awi_PRO-PRO_P_1-10/1awi_PRO-PRO_P_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1awi/1awi_PRO-PRO_P_1-10/1awi_PRO-PRO_P_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1awi/1awi_PRO-PRO_P_1-10/1awi_PRO-PRO_P_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1awi/1awi_PRO-PRO_P_1-10/1awi_PRO-PRO_P_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1awi/1awi_PRO-PRO_P_1-10/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1awi/1awi_PRO-PRO_P_1-10/1awi_PRO-PRO_P_1-10_complex_UNL.pdb'), ('1azg_PRO-THR_A_91-104', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1azg/1azg_PRO-THR_A_91-104/1azg_PRO-THR_A_91-104_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1azg/1azg_PRO-THR_A_91-104/1azg_PRO-THR_A_91-104_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1azg/1azg_PRO-THR_A_91-104/1azg_PRO-THR_A_91-104_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1azg/1azg_PRO-THR_A_91-104/1azg_PRO-THR_A_91-104_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1azg/1azg_PRO-THR_A_91-104/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1azg/1azg_PRO-THR_A_91-104/1azg_PRO-THR_A_91-104_complex_UNL.pdb'), ('1bbz_ACE-PRO_D_0-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bbz/1bbz_ACE-PRO_D_0-10/1bbz_ACE-PRO_D_0-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bbz/1bbz_ACE-PRO_D_0-10/1bbz_ACE-PRO_D_0-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bbz/1bbz_ACE-PRO_D_0-10/1bbz_ACE-PRO_D_0-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bbz/1bbz_ACE-PRO_D_0-10/1bbz_ACE-PRO_D_0-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bbz/1bbz_ACE-PRO_D_0-10/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bbz/1bbz_ACE-PRO_D_0-10/1bbz_ACE-PRO_D_0-10_complex_UNL.pdb'), ('1bmb_LYS-PHE_I_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bmb/1bmb_LYS-PHE_I_1-9/1bmb_LYS-PHE_I_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bmb/1bmb_LYS-PHE_I_1-9/1bmb_LYS-PHE_I_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bmb/1bmb_LYS-PHE_I_1-9/1bmb_LYS-PHE_I_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bmb/1bmb_LYS-PHE_I_1-9/1bmb_LYS-PHE_I_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bmb/1bmb_LYS-PHE_I_1-9/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bmb/1bmb_LYS-PHE_I_1-9/1bmb_LYS-PHE_I_1-9_complex_UNL.pdb'), ('1br8_SER-ALA_P_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1br8/1br8_SER-ALA_P_1-12/1br8_SER-ALA_P_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1br8/1br8_SER-ALA_P_1-12/1br8_SER-ALA_P_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1br8/1br8_SER-ALA_P_1-12/1br8_SER-ALA_P_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1br8/1br8_SER-ALA_P_1-12/1br8_SER-ALA_P_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1br8/1br8_SER-ALA_P_1-12/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1br8/1br8_SER-ALA_P_1-12/1br8_SER-ALA_P_1-12_complex_UNL.pdb'), ('1bxl_GLY-ARG_B_572-587', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bxl/1bxl_GLY-ARG_B_572-587/1bxl_GLY-ARG_B_572-587_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bxl/1bxl_GLY-ARG_B_572-587/1bxl_GLY-ARG_B_572-587_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bxl/1bxl_GLY-ARG_B_572-587/1bxl_GLY-ARG_B_572-587_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bxl/1bxl_GLY-ARG_B_572-587/1bxl_GLY-ARG_B_572-587_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bxl/1bxl_GLY-ARG_B_572-587/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1bxl/1bxl_GLY-ARG_B_572-587/1bxl_GLY-ARG_B_572-587_complex_UNL.pdb'), ('1d4t_LYS-LYS_B_276-286', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1d4t/1d4t_LYS-LYS_B_276-286/1d4t_LYS-LYS_B_276-286_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1d4t/1d4t_LYS-LYS_B_276-286/1d4t_LYS-LYS_B_276-286_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1d4t/1d4t_LYS-LYS_B_276-286/1d4t_LYS-LYS_B_276-286_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1d4t/1d4t_LYS-LYS_B_276-286/1d4t_LYS-LYS_B_276-286_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1d4t/1d4t_LYS-LYS_B_276-286/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1d4t/1d4t_LYS-LYS_B_276-286/1d4t_LYS-LYS_B_276-286_complex_UNL.pdb'), ('1ddm_GLY-PRO_B_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ddm/1ddm_GLY-PRO_B_1-11/1ddm_GLY-PRO_B_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ddm/1ddm_GLY-PRO_B_1-11/1ddm_GLY-PRO_B_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ddm/1ddm_GLY-PRO_B_1-11/1ddm_GLY-PRO_B_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ddm/1ddm_GLY-PRO_B_1-11/1ddm_GLY-PRO_B_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ddm/1ddm_GLY-PRO_B_1-11/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ddm/1ddm_GLY-PRO_B_1-11/1ddm_GLY-PRO_B_1-11_complex_UNL.pdb'), ('1dpu_ARG-ARG_B_73-88', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1dpu/1dpu_ARG-ARG_B_73-88/1dpu_ARG-ARG_B_73-88_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1dpu/1dpu_ARG-ARG_B_73-88/1dpu_ARG-ARG_B_73-88_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1dpu/1dpu_ARG-ARG_B_73-88/1dpu_ARG-ARG_B_73-88_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1dpu/1dpu_ARG-ARG_B_73-88/1dpu_ARG-ARG_B_73-88_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1dpu/1dpu_ARG-ARG_B_73-88/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1dpu/1dpu_ARG-ARG_B_73-88/1dpu_ARG-ARG_B_73-88_complex_UNL.pdb'), ('1f3j_ALA-SER_Q_11-24', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f3j/1f3j_ALA-SER_Q_11-24/1f3j_ALA-SER_Q_11-24_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f3j/1f3j_ALA-SER_Q_11-24/1f3j_ALA-SER_Q_11-24_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f3j/1f3j_ALA-SER_Q_11-24/1f3j_ALA-SER_Q_11-24_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f3j/1f3j_ALA-SER_Q_11-24/1f3j_ALA-SER_Q_11-24_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f3j/1f3j_ALA-SER_Q_11-24/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f3j/1f3j_ALA-SER_Q_11-24/1f3j_ALA-SER_Q_11-24_complex_UNL.pdb'), ('1f90_LYS-LEU_E_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f90/1f90_LYS-LEU_E_1-9/1f90_LYS-LEU_E_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f90/1f90_LYS-LEU_E_1-9/1f90_LYS-LEU_E_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f90/1f90_LYS-LEU_E_1-9/1f90_LYS-LEU_E_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f90/1f90_LYS-LEU_E_1-9/1f90_LYS-LEU_E_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f90/1f90_LYS-LEU_E_1-9/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1f90/1f90_LYS-LEU_E_1-9/1f90_LYS-LEU_E_1-9_complex_UNL.pdb'), ('1fo0_ILE-ILE_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fo0/1fo0_ILE-ILE_P_1-8/1fo0_ILE-ILE_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fo0/1fo0_ILE-ILE_P_1-8/1fo0_ILE-ILE_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fo0/1fo0_ILE-ILE_P_1-8/1fo0_ILE-ILE_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fo0/1fo0_ILE-ILE_P_1-8/1fo0_ILE-ILE_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fo0/1fo0_ILE-ILE_P_1-8/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fo0/1fo0_ILE-ILE_P_1-8/1fo0_ILE-ILE_P_1-8_complex_UNL.pdb'), ('1fzj_ARG-LEU_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzj/1fzj_ARG-LEU_P_1-8/1fzj_ARG-LEU_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzj/1fzj_ARG-LEU_P_1-8/1fzj_ARG-LEU_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzj/1fzj_ARG-LEU_P_1-8/1fzj_ARG-LEU_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzj/1fzj_ARG-LEU_P_1-8/1fzj_ARG-LEU_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzj/1fzj_ARG-LEU_P_1-8/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzj/1fzj_ARG-LEU_P_1-8/1fzj_ARG-LEU_P_1-8_complex_UNL.pdb'), ('1fzk_PHE-LEU_P_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzk/1fzk_PHE-LEU_P_1-9/1fzk_PHE-LEU_P_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzk/1fzk_PHE-LEU_P_1-9/1fzk_PHE-LEU_P_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzk/1fzk_PHE-LEU_P_1-9/1fzk_PHE-LEU_P_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzk/1fzk_PHE-LEU_P_1-9/1fzk_PHE-LEU_P_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzk/1fzk_PHE-LEU_P_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzk/1fzk_PHE-LEU_P_1-9/1fzk_PHE-LEU_P_1-9_complex_UNL.pdb'), ('1fzm_ARG-LEU_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzm/1fzm_ARG-LEU_P_1-8/1fzm_ARG-LEU_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzm/1fzm_ARG-LEU_P_1-8/1fzm_ARG-LEU_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzm/1fzm_ARG-LEU_P_1-8/1fzm_ARG-LEU_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzm/1fzm_ARG-LEU_P_1-8/1fzm_ARG-LEU_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzm/1fzm_ARG-LEU_P_1-8/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzm/1fzm_ARG-LEU_P_1-8/1fzm_ARG-LEU_P_1-8_complex_UNL.pdb'), ('1fzo_PHE-LEU_P_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzo/1fzo_PHE-LEU_P_1-9/1fzo_PHE-LEU_P_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzo/1fzo_PHE-LEU_P_1-9/1fzo_PHE-LEU_P_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzo/1fzo_PHE-LEU_P_1-9/1fzo_PHE-LEU_P_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzo/1fzo_PHE-LEU_P_1-9/1fzo_PHE-LEU_P_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzo/1fzo_PHE-LEU_P_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1fzo/1fzo_PHE-LEU_P_1-9/1fzo_PHE-LEU_P_1-9_complex_UNL.pdb'), ('1g1e_ARG-ARG_A_6-21', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g1e/1g1e_ARG-ARG_A_6-21/1g1e_ARG-ARG_A_6-21_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g1e/1g1e_ARG-ARG_A_6-21/1g1e_ARG-ARG_A_6-21_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g1e/1g1e_ARG-ARG_A_6-21/1g1e_ARG-ARG_A_6-21_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g1e/1g1e_ARG-ARG_A_6-21/1g1e_ARG-ARG_A_6-21_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g1e/1g1e_ARG-ARG_A_6-21/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g1e/1g1e_ARG-ARG_A_6-21/1g1e_ARG-ARG_A_6-21_complex_UNL.pdb'), ('1g3f_ALA-GLU_B_901-909', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g3f/1g3f_ALA-GLU_B_901-909/1g3f_ALA-GLU_B_901-909_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g3f/1g3f_ALA-GLU_B_901-909/1g3f_ALA-GLU_B_901-909_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g3f/1g3f_ALA-GLU_B_901-909/1g3f_ALA-GLU_B_901-909_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g3f/1g3f_ALA-GLU_B_901-909/1g3f_ALA-GLU_B_901-909_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g3f/1g3f_ALA-GLU_B_901-909/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g3f/1g3f_ALA-GLU_B_901-909/1g3f_ALA-GLU_B_901-909_complex_UNL.pdb'), ('1g6r_SER-LEU_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g6r/1g6r_SER-LEU_P_1-8/1g6r_SER-LEU_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g6r/1g6r_SER-LEU_P_1-8/1g6r_SER-LEU_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g6r/1g6r_SER-LEU_P_1-8/1g6r_SER-LEU_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g6r/1g6r_SER-LEU_P_1-8/1g6r_SER-LEU_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g6r/1g6r_SER-LEU_P_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g6r/1g6r_SER-LEU_P_1-8/1g6r_SER-LEU_P_1-8_complex_UNL.pdb'), ('1g7p_SER-MET_P_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7p/1g7p_SER-MET_P_1-9/1g7p_SER-MET_P_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7p/1g7p_SER-MET_P_1-9/1g7p_SER-MET_P_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7p/1g7p_SER-MET_P_1-9/1g7p_SER-MET_P_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7p/1g7p_SER-MET_P_1-9/1g7p_SER-MET_P_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7p/1g7p_SER-MET_P_1-9/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7p/1g7p_SER-MET_P_1-9/1g7p_SER-MET_P_1-9_complex_UNL.pdb'), ('1g7q_SER-ALA_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7q/1g7q_SER-ALA_P_1-8/1g7q_SER-ALA_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7q/1g7q_SER-ALA_P_1-8/1g7q_SER-ALA_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7q/1g7q_SER-ALA_P_1-8/1g7q_SER-ALA_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7q/1g7q_SER-ALA_P_1-8/1g7q_SER-ALA_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7q/1g7q_SER-ALA_P_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1g7q/1g7q_SER-ALA_P_1-8/1g7q_SER-ALA_P_1-8_complex_UNL.pdb'), ('1guw_ALA-GLY_B_1-18', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1guw/1guw_ALA-GLY_B_1-18/1guw_ALA-GLY_B_1-18_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1guw/1guw_ALA-GLY_B_1-18/1guw_ALA-GLY_B_1-18_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1guw/1guw_ALA-GLY_B_1-18/1guw_ALA-GLY_B_1-18_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1guw/1guw_ALA-GLY_B_1-18/1guw_ALA-GLY_B_1-18_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1guw/1guw_ALA-GLY_B_1-18/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1guw/1guw_ALA-GLY_B_1-18/1guw_ALA-GLY_B_1-18_complex_UNL.pdb'), ('1gux_ASP-ASN_E_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1gux/1gux_ASP-ASN_E_1-9/1gux_ASP-ASN_E_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1gux/1gux_ASP-ASN_E_1-9/1gux_ASP-ASN_E_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1gux/1gux_ASP-ASN_E_1-9/1gux_ASP-ASN_E_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1gux/1gux_ASP-ASN_E_1-9/1gux_ASP-ASN_E_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1gux/1gux_ASP-ASN_E_1-9/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1gux/1gux_ASP-ASN_E_1-9/1gux_ASP-ASN_E_1-9_complex_UNL.pdb'), ('1h3h_ALA-ALA_B_76-86', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1h3h/1h3h_ALA-ALA_B_76-86/1h3h_ALA-ALA_B_76-86_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1h3h/1h3h_ALA-ALA_B_76-86/1h3h_ALA-ALA_B_76-86_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1h3h/1h3h_ALA-ALA_B_76-86/1h3h_ALA-ALA_B_76-86_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1h3h/1h3h_ALA-ALA_B_76-86/1h3h_ALA-ALA_B_76-86_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1h3h/1h3h_ALA-ALA_B_76-86/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1h3h/1h3h_ALA-ALA_B_76-86/1h3h_ALA-ALA_B_76-86_complex_UNL.pdb'), ('1haa_TRP-ASP_B_75-87', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1haa/1haa_TRP-ASP_B_75-87/1haa_TRP-ASP_B_75-87_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1haa/1haa_TRP-ASP_B_75-87/1haa_TRP-ASP_B_75-87_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1haa/1haa_TRP-ASP_B_75-87/1haa_TRP-ASP_B_75-87_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1haa/1haa_TRP-ASP_B_75-87/1haa_TRP-ASP_B_75-87_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1haa/1haa_TRP-ASP_B_75-87/interaction.xml', 'TRP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1haa/1haa_TRP-ASP_B_75-87/1haa_TRP-ASP_B_75-87_complex_UNL.pdb'), ('1him_TYR-NH2_R_100-109', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1him/1him_TYR-NH2_R_100-109/1him_TYR-NH2_R_100-109_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1him/1him_TYR-NH2_R_100-109/1him_TYR-NH2_R_100-109_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1him/1him_TYR-NH2_R_100-109/1him_TYR-NH2_R_100-109_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1him/1him_TYR-NH2_R_100-109/1him_TYR-NH2_R_100-109_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1him/1him_TYR-NH2_R_100-109/interaction.xml', 'TYR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1him/1him_TYR-NH2_R_100-109/1him_TYR-NH2_R_100-109_complex_UNL.pdb'), ('1i5h_GLY-LEU_B_605-621', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1i5h/1i5h_GLY-LEU_B_605-621/1i5h_GLY-LEU_B_605-621_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1i5h/1i5h_GLY-LEU_B_605-621/1i5h_GLY-LEU_B_605-621_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1i5h/1i5h_GLY-LEU_B_605-621/1i5h_GLY-LEU_B_605-621_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1i5h/1i5h_GLY-LEU_B_605-621/1i5h_GLY-LEU_B_605-621_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1i5h/1i5h_GLY-LEU_B_605-621/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1i5h/1i5h_GLY-LEU_B_605-621/1i5h_GLY-LEU_B_605-621_complex_UNL.pdb'), ('1inq_SER-LEU_C_2001-2009', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1inq/1inq_SER-LEU_C_2001-2009/1inq_SER-LEU_C_2001-2009_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1inq/1inq_SER-LEU_C_2001-2009/1inq_SER-LEU_C_2001-2009_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1inq/1inq_SER-LEU_C_2001-2009/1inq_SER-LEU_C_2001-2009_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1inq/1inq_SER-LEU_C_2001-2009/1inq_SER-LEU_C_2001-2009_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1inq/1inq_SER-LEU_C_2001-2009/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1inq/1inq_SER-LEU_C_2001-2009/1inq_SER-LEU_C_2001-2009_complex_UNL.pdb'), ('1irs_LEU-SER_B_489-499', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1irs/1irs_LEU-SER_B_489-499/1irs_LEU-SER_B_489-499_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1irs/1irs_LEU-SER_B_489-499/1irs_LEU-SER_B_489-499_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1irs/1irs_LEU-SER_B_489-499/1irs_LEU-SER_B_489-499_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1irs/1irs_LEU-SER_B_489-499/1irs_LEU-SER_B_489-499_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1irs/1irs_LEU-SER_B_489-499/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1irs/1irs_LEU-SER_B_489-499/1irs_LEU-SER_B_489-499_complex_UNL.pdb'), ('1j4p_LYS-GLU_B_165-177', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j4p/1j4p_LYS-GLU_B_165-177/1j4p_LYS-GLU_B_165-177_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j4p/1j4p_LYS-GLU_B_165-177/1j4p_LYS-GLU_B_165-177_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j4p/1j4p_LYS-GLU_B_165-177/1j4p_LYS-GLU_B_165-177_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j4p/1j4p_LYS-GLU_B_165-177/1j4p_LYS-GLU_B_165-177_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j4p/1j4p_LYS-GLU_B_165-177/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j4p/1j4p_LYS-GLU_B_165-177/1j4p_LYS-GLU_B_165-177_complex_UNL.pdb'), ('1j7z_LYS-NH2_A_1-16', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j7z/1j7z_LYS-NH2_A_1-16/1j7z_LYS-NH2_A_1-16_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j7z/1j7z_LYS-NH2_A_1-16/1j7z_LYS-NH2_A_1-16_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j7z/1j7z_LYS-NH2_A_1-16/1j7z_LYS-NH2_A_1-16_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j7z/1j7z_LYS-NH2_A_1-16/1j7z_LYS-NH2_A_1-16_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j7z/1j7z_LYS-NH2_A_1-16/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j7z/1j7z_LYS-NH2_A_1-16/1j7z_LYS-NH2_A_1-16_complex_UNL.pdb'), ('1j80_LYS-NH2_A_1-16', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j80/1j80_LYS-NH2_A_1-16/1j80_LYS-NH2_A_1-16_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j80/1j80_LYS-NH2_A_1-16/1j80_LYS-NH2_A_1-16_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j80/1j80_LYS-NH2_A_1-16/1j80_LYS-NH2_A_1-16_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j80/1j80_LYS-NH2_A_1-16/1j80_LYS-NH2_A_1-16_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j80/1j80_LYS-NH2_A_1-16/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j80/1j80_LYS-NH2_A_1-16/1j80_LYS-NH2_A_1-16_complex_UNL.pdb'), ('1j81_LYS-NH2_A_1-16', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j81/1j81_LYS-NH2_A_1-16/1j81_LYS-NH2_A_1-16_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j81/1j81_LYS-NH2_A_1-16/1j81_LYS-NH2_A_1-16_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j81/1j81_LYS-NH2_A_1-16/1j81_LYS-NH2_A_1-16_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j81/1j81_LYS-NH2_A_1-16/1j81_LYS-NH2_A_1-16_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j81/1j81_LYS-NH2_A_1-16/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1j81/1j81_LYS-NH2_A_1-16/1j81_LYS-NH2_A_1-16_complex_UNL.pdb'), ('1juf_SER-LEU_C_2001-2009', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1juf/1juf_SER-LEU_C_2001-2009/1juf_SER-LEU_C_2001-2009_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1juf/1juf_SER-LEU_C_2001-2009/1juf_SER-LEU_C_2001-2009_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1juf/1juf_SER-LEU_C_2001-2009/1juf_SER-LEU_C_2001-2009_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1juf/1juf_SER-LEU_C_2001-2009/1juf_SER-LEU_C_2001-2009_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1juf/1juf_SER-LEU_C_2001-2009/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1juf/1juf_SER-LEU_C_2001-2009/1juf_SER-LEU_C_2001-2009_complex_UNL.pdb'), ('1jwm_PRO-THR_C_306-318', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwm/1jwm_PRO-THR_C_306-318/1jwm_PRO-THR_C_306-318_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwm/1jwm_PRO-THR_C_306-318/1jwm_PRO-THR_C_306-318_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwm/1jwm_PRO-THR_C_306-318/1jwm_PRO-THR_C_306-318_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwm/1jwm_PRO-THR_C_306-318/1jwm_PRO-THR_C_306-318_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwm/1jwm_PRO-THR_C_306-318/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwm/1jwm_PRO-THR_C_306-318/1jwm_PRO-THR_C_306-318_complex_UNL.pdb'), ('1jws_PRO-THR_C_306-318', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jws/1jws_PRO-THR_C_306-318/1jws_PRO-THR_C_306-318_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jws/1jws_PRO-THR_C_306-318/1jws_PRO-THR_C_306-318_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jws/1jws_PRO-THR_C_306-318/1jws_PRO-THR_C_306-318_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jws/1jws_PRO-THR_C_306-318/1jws_PRO-THR_C_306-318_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jws/1jws_PRO-THR_C_306-318/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jws/1jws_PRO-THR_C_306-318/1jws_PRO-THR_C_306-318_complex_UNL.pdb'), ('1jwu_PRO-THR_C_306-318', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwu/1jwu_PRO-THR_C_306-318/1jwu_PRO-THR_C_306-318_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwu/1jwu_PRO-THR_C_306-318/1jwu_PRO-THR_C_306-318_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwu/1jwu_PRO-THR_C_306-318/1jwu_PRO-THR_C_306-318_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwu/1jwu_PRO-THR_C_306-318/1jwu_PRO-THR_C_306-318_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwu/1jwu_PRO-THR_C_306-318/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jwu/1jwu_PRO-THR_C_306-318/1jwu_PRO-THR_C_306-318_complex_UNL.pdb'), ('1jyc_ARG-SER_R_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyc/1jyc_ARG-SER_R_1-15/1jyc_ARG-SER_R_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyc/1jyc_ARG-SER_R_1-15/1jyc_ARG-SER_R_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyc/1jyc_ARG-SER_R_1-15/1jyc_ARG-SER_R_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyc/1jyc_ARG-SER_R_1-15/1jyc_ARG-SER_R_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyc/1jyc_ARG-SER_R_1-15/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyc/1jyc_ARG-SER_R_1-15/1jyc_ARG-SER_R_1-15_complex_UNL.pdb'), ('1jyi_ASP-SER_Q_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyi/1jyi_ASP-SER_Q_1-12/1jyi_ASP-SER_Q_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyi/1jyi_ASP-SER_Q_1-12/1jyi_ASP-SER_Q_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyi/1jyi_ASP-SER_Q_1-12/1jyi_ASP-SER_Q_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyi/1jyi_ASP-SER_Q_1-12/1jyi_ASP-SER_Q_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyi/1jyi_ASP-SER_Q_1-12/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1jyi/1jyi_ASP-SER_Q_1-12/1jyi_ASP-SER_Q_1-12_complex_UNL.pdb'), ('1k3n_LYS-GLU_B_165-177', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3n/1k3n_LYS-GLU_B_165-177/1k3n_LYS-GLU_B_165-177_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3n/1k3n_LYS-GLU_B_165-177/1k3n_LYS-GLU_B_165-177_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3n/1k3n_LYS-GLU_B_165-177/1k3n_LYS-GLU_B_165-177_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3n/1k3n_LYS-GLU_B_165-177/1k3n_LYS-GLU_B_165-177_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3n/1k3n_LYS-GLU_B_165-177/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3n/1k3n_LYS-GLU_B_165-177/1k3n_LYS-GLU_B_165-177_complex_UNL.pdb'), ('1k3q_SER-GLN_B_165-177', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3q/1k3q_SER-GLN_B_165-177/1k3q_SER-GLN_B_165-177_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3q/1k3q_SER-GLN_B_165-177/1k3q_SER-GLN_B_165-177_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3q/1k3q_SER-GLN_B_165-177/1k3q_SER-GLN_B_165-177_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3q/1k3q_SER-GLN_B_165-177/1k3q_SER-GLN_B_165-177_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3q/1k3q_SER-GLN_B_165-177/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1k3q/1k3q_SER-GLN_B_165-177/1k3q_SER-GLN_B_165-177_complex_UNL.pdb'), ('1ka7_ARG-LYS_B_275-286', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ka7/1ka7_ARG-LYS_B_275-286/1ka7_ARG-LYS_B_275-286_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ka7/1ka7_ARG-LYS_B_275-286/1ka7_ARG-LYS_B_275-286_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ka7/1ka7_ARG-LYS_B_275-286/1ka7_ARG-LYS_B_275-286_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ka7/1ka7_ARG-LYS_B_275-286/1ka7_ARG-LYS_B_275-286_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ka7/1ka7_ARG-LYS_B_275-286/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ka7/1ka7_ARG-LYS_B_275-286/1ka7_ARG-LYS_B_275-286_complex_UNL.pdb'), ('1klg_GLY-ASP_C_23-37', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klg/1klg_GLY-ASP_C_23-37/1klg_GLY-ASP_C_23-37_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klg/1klg_GLY-ASP_C_23-37/1klg_GLY-ASP_C_23-37_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klg/1klg_GLY-ASP_C_23-37/1klg_GLY-ASP_C_23-37_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klg/1klg_GLY-ASP_C_23-37/1klg_GLY-ASP_C_23-37_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klg/1klg_GLY-ASP_C_23-37/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klg/1klg_GLY-ASP_C_23-37/1klg_GLY-ASP_C_23-37_complex_UNL.pdb'), ('1klu_GLY-ASP_C_23-37', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klu/1klu_GLY-ASP_C_23-37/1klu_GLY-ASP_C_23-37_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klu/1klu_GLY-ASP_C_23-37/1klu_GLY-ASP_C_23-37_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klu/1klu_GLY-ASP_C_23-37/1klu_GLY-ASP_C_23-37_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klu/1klu_GLY-ASP_C_23-37/1klu_GLY-ASP_C_23-37_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klu/1klu_GLY-ASP_C_23-37/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1klu/1klu_GLY-ASP_C_23-37/1klu_GLY-ASP_C_23-37_complex_UNL.pdb'), ('1l0a_SER-LYS_B_1-18', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l0a/1l0a_SER-LYS_B_1-18/1l0a_SER-LYS_B_1-18_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l0a/1l0a_SER-LYS_B_1-18/1l0a_SER-LYS_B_1-18_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l0a/1l0a_SER-LYS_B_1-18/1l0a_SER-LYS_B_1-18_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l0a/1l0a_SER-LYS_B_1-18/1l0a_SER-LYS_B_1-18_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l0a/1l0a_SER-LYS_B_1-18/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l0a/1l0a_SER-LYS_B_1-18/1l0a_SER-LYS_B_1-18_complex_UNL.pdb'), ('1l2z_SER-VAL_B_63-73', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l2z/1l2z_SER-VAL_B_63-73/1l2z_SER-VAL_B_63-73_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l2z/1l2z_SER-VAL_B_63-73/1l2z_SER-VAL_B_63-73_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l2z/1l2z_SER-VAL_B_63-73/1l2z_SER-VAL_B_63-73_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l2z/1l2z_SER-VAL_B_63-73/1l2z_SER-VAL_B_63-73_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l2z/1l2z_SER-VAL_B_63-73/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1l2z/1l2z_SER-VAL_B_63-73/1l2z_SER-VAL_B_63-73_complex_UNL.pdb'), ('1lb6_LYS-PHE_B_601-609', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lb6/1lb6_LYS-PHE_B_601-609/1lb6_LYS-PHE_B_601-609_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lb6/1lb6_LYS-PHE_B_601-609/1lb6_LYS-PHE_B_601-609_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lb6/1lb6_LYS-PHE_B_601-609/1lb6_LYS-PHE_B_601-609_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lb6/1lb6_LYS-PHE_B_601-609/1lb6_LYS-PHE_B_601-609_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lb6/1lb6_LYS-PHE_B_601-609/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lb6/1lb6_LYS-PHE_B_601-609/1lb6_LYS-PHE_B_601-609_complex_UNL.pdb'), ('1lcj_GLU-LEU_B_201-211', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lcj/1lcj_GLU-LEU_B_201-211/1lcj_GLU-LEU_B_201-211_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lcj/1lcj_GLU-LEU_B_201-211/1lcj_GLU-LEU_B_201-211_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lcj/1lcj_GLU-LEU_B_201-211/1lcj_GLU-LEU_B_201-211_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lcj/1lcj_GLU-LEU_B_201-211/1lcj_GLU-LEU_B_201-211_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lcj/1lcj_GLU-LEU_B_201-211/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1lcj/1lcj_GLU-LEU_B_201-211/1lcj_GLU-LEU_B_201-211_complex_UNL.pdb'), ('1leg_GLU-VAL_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1leg/1leg_GLU-VAL_P_1-8/1leg_GLU-VAL_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1leg/1leg_GLU-VAL_P_1-8/1leg_GLU-VAL_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1leg/1leg_GLU-VAL_P_1-8/1leg_GLU-VAL_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1leg/1leg_GLU-VAL_P_1-8/1leg_GLU-VAL_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1leg/1leg_GLU-VAL_P_1-8/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1leg/1leg_GLU-VAL_P_1-8/1leg_GLU-VAL_P_1-8_complex_UNL.pdb'), ('1mfg_GLU-VAL_B_1247-1255', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mfg/1mfg_GLU-VAL_B_1247-1255/1mfg_GLU-VAL_B_1247-1255_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mfg/1mfg_GLU-VAL_B_1247-1255/1mfg_GLU-VAL_B_1247-1255_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mfg/1mfg_GLU-VAL_B_1247-1255/1mfg_GLU-VAL_B_1247-1255_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mfg/1mfg_GLU-VAL_B_1247-1255/1mfg_GLU-VAL_B_1247-1255_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mfg/1mfg_GLU-VAL_B_1247-1255/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mfg/1mfg_GLU-VAL_B_1247-1255/1mfg_GLU-VAL_B_1247-1255_complex_UNL.pdb'), ('1mq1_THR-SER_C_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mq1/1mq1_THR-SER_C_1-12/1mq1_THR-SER_C_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mq1/1mq1_THR-SER_C_1-12/1mq1_THR-SER_C_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mq1/1mq1_THR-SER_C_1-12/1mq1_THR-SER_C_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mq1/1mq1_THR-SER_C_1-12/1mq1_THR-SER_C_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mq1/1mq1_THR-SER_C_1-12/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mq1/1mq1_THR-SER_C_1-12/1mq1_THR-SER_C_1-12_complex_UNL.pdb'), ('1mwn_THR-SER_X_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mwn/1mwn_THR-SER_X_1-12/1mwn_THR-SER_X_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mwn/1mwn_THR-SER_X_1-12/1mwn_THR-SER_X_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mwn/1mwn_THR-SER_X_1-12/1mwn_THR-SER_X_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mwn/1mwn_THR-SER_X_1-12/1mwn_THR-SER_X_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mwn/1mwn_THR-SER_X_1-12/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mwn/1mwn_THR-SER_X_1-12/1mwn_THR-SER_X_1-12_complex_UNL.pdb'), ('1mxl_ARG-LYS_I_1-17', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mxl/1mxl_ARG-LYS_I_1-17/1mxl_ARG-LYS_I_1-17_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mxl/1mxl_ARG-LYS_I_1-17/1mxl_ARG-LYS_I_1-17_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mxl/1mxl_ARG-LYS_I_1-17/1mxl_ARG-LYS_I_1-17_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mxl/1mxl_ARG-LYS_I_1-17/1mxl_ARG-LYS_I_1-17_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mxl/1mxl_ARG-LYS_I_1-17/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1mxl/1mxl_ARG-LYS_I_1-17/1mxl_ARG-LYS_I_1-17_complex_UNL.pdb'), ('1ntv_ASN-THR_B_5-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ntv/1ntv_ASN-THR_B_5-14/1ntv_ASN-THR_B_5-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ntv/1ntv_ASN-THR_B_5-14/1ntv_ASN-THR_B_5-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ntv/1ntv_ASN-THR_B_5-14/1ntv_ASN-THR_B_5-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ntv/1ntv_ASN-THR_B_5-14/1ntv_ASN-THR_B_5-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ntv/1ntv_ASN-THR_B_5-14/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ntv/1ntv_ASN-THR_B_5-14/1ntv_ASN-THR_B_5-14_complex_UNL.pdb'), ('1nzl_PRO-ALA_C_498-505', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzl/1nzl_PRO-ALA_C_498-505/1nzl_PRO-ALA_C_498-505_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzl/1nzl_PRO-ALA_C_498-505/1nzl_PRO-ALA_C_498-505_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzl/1nzl_PRO-ALA_C_498-505/1nzl_PRO-ALA_C_498-505_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzl/1nzl_PRO-ALA_C_498-505/1nzl_PRO-ALA_C_498-505_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzl/1nzl_PRO-ALA_C_498-505/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzl/1nzl_PRO-ALA_C_498-505/1nzl_PRO-ALA_C_498-505_complex_UNL.pdb'), ('1nzv_PRO-ALA_C_498-505', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzv/1nzv_PRO-ALA_C_498-505/1nzv_PRO-ALA_C_498-505_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzv/1nzv_PRO-ALA_C_498-505/1nzv_PRO-ALA_C_498-505_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzv/1nzv_PRO-ALA_C_498-505/1nzv_PRO-ALA_C_498-505_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzv/1nzv_PRO-ALA_C_498-505/1nzv_PRO-ALA_C_498-505_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzv/1nzv_PRO-ALA_C_498-505/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1nzv/1nzv_PRO-ALA_C_498-505/1nzv_PRO-ALA_C_498-505_complex_UNL.pdb'), ('1oai_ASP-LYS_B_10-18', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oai/1oai_ASP-LYS_B_10-18/1oai_ASP-LYS_B_10-18_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oai/1oai_ASP-LYS_B_10-18/1oai_ASP-LYS_B_10-18_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oai/1oai_ASP-LYS_B_10-18/1oai_ASP-LYS_B_10-18_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oai/1oai_ASP-LYS_B_10-18/1oai_ASP-LYS_B_10-18_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oai/1oai_ASP-LYS_B_10-18/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oai/1oai_ASP-LYS_B_10-18/1oai_ASP-LYS_B_10-18_complex_UNL.pdb'), ('1om2_GLY-ALA_B_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1om2/1om2_GLY-ALA_B_1-11/1om2_GLY-ALA_B_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1om2/1om2_GLY-ALA_B_1-11/1om2_GLY-ALA_B_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1om2/1om2_GLY-ALA_B_1-11/1om2_GLY-ALA_B_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1om2/1om2_GLY-ALA_B_1-11/1om2_GLY-ALA_B_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1om2/1om2_GLY-ALA_B_1-11/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1om2/1om2_GLY-ALA_B_1-11/1om2_GLY-ALA_B_1-11_complex_UNL.pdb'), ('1opi_PRO-ASP_B_13-25', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1opi/1opi_PRO-ASP_B_13-25/1opi_PRO-ASP_B_13-25_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1opi/1opi_PRO-ASP_B_13-25/1opi_PRO-ASP_B_13-25_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1opi/1opi_PRO-ASP_B_13-25/1opi_PRO-ASP_B_13-25_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1opi/1opi_PRO-ASP_B_13-25/1opi_PRO-ASP_B_13-25_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1opi/1opi_PRO-ASP_B_13-25/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1opi/1opi_PRO-ASP_B_13-25/1opi_PRO-ASP_B_13-25_complex_UNL.pdb'), ('1oqp_LYS-LYS_B_239-257', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oqp/1oqp_LYS-LYS_B_239-257/1oqp_LYS-LYS_B_239-257_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oqp/1oqp_LYS-LYS_B_239-257/1oqp_LYS-LYS_B_239-257_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oqp/1oqp_LYS-LYS_B_239-257/1oqp_LYS-LYS_B_239-257_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oqp/1oqp_LYS-LYS_B_239-257/1oqp_LYS-LYS_B_239-257_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oqp/1oqp_LYS-LYS_B_239-257/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1oqp/1oqp_LYS-LYS_B_239-257/1oqp_LYS-LYS_B_239-257_complex_UNL.pdb'), ('1ox9_ALA-ALA_K_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ox9/1ox9_ALA-ALA_K_1-8/1ox9_ALA-ALA_K_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ox9/1ox9_ALA-ALA_K_1-8/1ox9_ALA-ALA_K_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ox9/1ox9_ALA-ALA_K_1-8/1ox9_ALA-ALA_K_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ox9/1ox9_ALA-ALA_K_1-8/1ox9_ALA-ALA_K_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ox9/1ox9_ALA-ALA_K_1-8/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ox9/1ox9_ALA-ALA_K_1-8/1ox9_ALA-ALA_K_1-8_complex_UNL.pdb'), ('1prl_ALA-ARG_A_71-79', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prl/1prl_ALA-ARG_A_71-79/1prl_ALA-ARG_A_71-79_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prl/1prl_ALA-ARG_A_71-79/1prl_ALA-ARG_A_71-79_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prl/1prl_ALA-ARG_A_71-79/1prl_ALA-ARG_A_71-79_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prl/1prl_ALA-ARG_A_71-79/1prl_ALA-ARG_A_71-79_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prl/1prl_ALA-ARG_A_71-79/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prl/1prl_ALA-ARG_A_71-79/1prl_ALA-ARG_A_71-79_complex_UNL.pdb'), ('1prm_ALA-ARG_A_71-79', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prm/1prm_ALA-ARG_A_71-79/1prm_ALA-ARG_A_71-79_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prm/1prm_ALA-ARG_A_71-79/1prm_ALA-ARG_A_71-79_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prm/1prm_ALA-ARG_A_71-79/1prm_ALA-ARG_A_71-79_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prm/1prm_ALA-ARG_A_71-79/1prm_ALA-ARG_A_71-79_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prm/1prm_ALA-ARG_A_71-79/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1prm/1prm_ALA-ARG_A_71-79/1prm_ALA-ARG_A_71-79_complex_UNL.pdb'), ('1pz5_MET-ALA_C_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1pz5/1pz5_MET-ALA_C_1-8/1pz5_MET-ALA_C_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1pz5/1pz5_MET-ALA_C_1-8/1pz5_MET-ALA_C_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1pz5/1pz5_MET-ALA_C_1-8/1pz5_MET-ALA_C_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1pz5/1pz5_MET-ALA_C_1-8/1pz5_MET-ALA_C_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1pz5/1pz5_MET-ALA_C_1-8/interaction.xml', 'MET', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1pz5/1pz5_MET-ALA_C_1-8/1pz5_MET-ALA_C_1-8_complex_UNL.pdb'), ('1qwe_ALA-LEU_B_71-82', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwe/1qwe_ALA-LEU_B_71-82/1qwe_ALA-LEU_B_71-82_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwe/1qwe_ALA-LEU_B_71-82/1qwe_ALA-LEU_B_71-82_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwe/1qwe_ALA-LEU_B_71-82/1qwe_ALA-LEU_B_71-82_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwe/1qwe_ALA-LEU_B_71-82/1qwe_ALA-LEU_B_71-82_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwe/1qwe_ALA-LEU_B_71-82/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwe/1qwe_ALA-LEU_B_71-82/1qwe_ALA-LEU_B_71-82_complex_UNL.pdb'), ('1qwf_VAL-PRO_B_71-82', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwf/1qwf_VAL-PRO_B_71-82/1qwf_VAL-PRO_B_71-82_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwf/1qwf_VAL-PRO_B_71-82/1qwf_VAL-PRO_B_71-82_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwf/1qwf_VAL-PRO_B_71-82/1qwf_VAL-PRO_B_71-82_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwf/1qwf_VAL-PRO_B_71-82/1qwf_VAL-PRO_B_71-82_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwf/1qwf_VAL-PRO_B_71-82/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1qwf/1qwf_VAL-PRO_B_71-82/1qwf_VAL-PRO_B_71-82_complex_UNL.pdb'), ('1r5v_ALA-PHE_E_2-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5v/1r5v_ALA-PHE_E_2-14/1r5v_ALA-PHE_E_2-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5v/1r5v_ALA-PHE_E_2-14/1r5v_ALA-PHE_E_2-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5v/1r5v_ALA-PHE_E_2-14/1r5v_ALA-PHE_E_2-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5v/1r5v_ALA-PHE_E_2-14/1r5v_ALA-PHE_E_2-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5v/1r5v_ALA-PHE_E_2-14/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5v/1r5v_ALA-PHE_E_2-14/1r5v_ALA-PHE_E_2-14_complex_UNL.pdb'), ('1r5w_ALA-PHE_E_2-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5w/1r5w_ALA-PHE_E_2-14/1r5w_ALA-PHE_E_2-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5w/1r5w_ALA-PHE_E_2-14/1r5w_ALA-PHE_E_2-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5w/1r5w_ALA-PHE_E_2-14/1r5w_ALA-PHE_E_2-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5w/1r5w_ALA-PHE_E_2-14/1r5w_ALA-PHE_E_2-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5w/1r5w_ALA-PHE_E_2-14/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1r5w/1r5w_ALA-PHE_E_2-14/1r5w_ALA-PHE_E_2-14_complex_UNL.pdb'), ('1rlp_ARG-TYR_R_71-79', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlp/1rlp_ARG-TYR_R_71-79/1rlp_ARG-TYR_R_71-79_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlp/1rlp_ARG-TYR_R_71-79/1rlp_ARG-TYR_R_71-79_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlp/1rlp_ARG-TYR_R_71-79/1rlp_ARG-TYR_R_71-79_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlp/1rlp_ARG-TYR_R_71-79/1rlp_ARG-TYR_R_71-79_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlp/1rlp_ARG-TYR_R_71-79/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlp/1rlp_ARG-TYR_R_71-79/1rlp_ARG-TYR_R_71-79_complex_UNL.pdb'), ('1rlq_ARG-TYR_R_71-79', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlq/1rlq_ARG-TYR_R_71-79/1rlq_ARG-TYR_R_71-79_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlq/1rlq_ARG-TYR_R_71-79/1rlq_ARG-TYR_R_71-79_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlq/1rlq_ARG-TYR_R_71-79/1rlq_ARG-TYR_R_71-79_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlq/1rlq_ARG-TYR_R_71-79/1rlq_ARG-TYR_R_71-79_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlq/1rlq_ARG-TYR_R_71-79/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rlq/1rlq_ARG-TYR_R_71-79/1rlq_ARG-TYR_R_71-79_complex_UNL.pdb'), ('1rst_ALA-GLY_P_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rst/1rst_ALA-GLY_P_1-9/1rst_ALA-GLY_P_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rst/1rst_ALA-GLY_P_1-9/1rst_ALA-GLY_P_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rst/1rst_ALA-GLY_P_1-9/1rst_ALA-GLY_P_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rst/1rst_ALA-GLY_P_1-9/1rst_ALA-GLY_P_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rst/1rst_ALA-GLY_P_1-9/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1rst/1rst_ALA-GLY_P_1-9/1rst_ALA-GLY_P_1-9_complex_UNL.pdb'), ('1s5q_ARG-ARG_A_6-21', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1s5q/1s5q_ARG-ARG_A_6-21/1s5q_ARG-ARG_A_6-21_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1s5q/1s5q_ARG-ARG_A_6-21/1s5q_ARG-ARG_A_6-21_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1s5q/1s5q_ARG-ARG_A_6-21/1s5q_ARG-ARG_A_6-21_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1s5q/1s5q_ARG-ARG_A_6-21/1s5q_ARG-ARG_A_6-21_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1s5q/1s5q_ARG-ARG_A_6-21/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1s5q/1s5q_ARG-ARG_A_6-21/1s5q_ARG-ARG_A_6-21_complex_UNL.pdb'), ('1sfi_GLY-ASP_I_1-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sfi/1sfi_GLY-ASP_I_1-14/1sfi_GLY-ASP_I_1-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sfi/1sfi_GLY-ASP_I_1-14/1sfi_GLY-ASP_I_1-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sfi/1sfi_GLY-ASP_I_1-14/1sfi_GLY-ASP_I_1-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sfi/1sfi_GLY-ASP_I_1-14/1sfi_GLY-ASP_I_1-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sfi/1sfi_GLY-ASP_I_1-14/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sfi/1sfi_GLY-ASP_I_1-14/1sfi_GLY-ASP_I_1-14_complex_UNL.pdb'), ('1shc_HIS-ALA_B_483-494', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1shc/1shc_HIS-ALA_B_483-494/1shc_HIS-ALA_B_483-494_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1shc/1shc_HIS-ALA_B_483-494/1shc_HIS-ALA_B_483-494_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1shc/1shc_HIS-ALA_B_483-494/1shc_HIS-ALA_B_483-494_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1shc/1shc_HIS-ALA_B_483-494/1shc_HIS-ALA_B_483-494_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1shc/1shc_HIS-ALA_B_483-494/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1shc/1shc_HIS-ALA_B_483-494/1shc_HIS-ALA_B_483-494_complex_UNL.pdb'), ('1sjh_PRO-GLY_C_34-46', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sjh/1sjh_PRO-GLY_C_34-46/1sjh_PRO-GLY_C_34-46_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sjh/1sjh_PRO-GLY_C_34-46/1sjh_PRO-GLY_C_34-46_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sjh/1sjh_PRO-GLY_C_34-46/1sjh_PRO-GLY_C_34-46_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sjh/1sjh_PRO-GLY_C_34-46/1sjh_PRO-GLY_C_34-46_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sjh/1sjh_PRO-GLY_C_34-46/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1sjh/1sjh_PRO-GLY_C_34-46/1sjh_PRO-GLY_C_34-46_complex_UNL.pdb'), ('1t08_ASP-ASP_C_1484-1498', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1t08/1t08_ASP-ASP_C_1484-1498/1t08_ASP-ASP_C_1484-1498_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1t08/1t08_ASP-ASP_C_1484-1498/1t08_ASP-ASP_C_1484-1498_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1t08/1t08_ASP-ASP_C_1484-1498/1t08_ASP-ASP_C_1484-1498_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1t08/1t08_ASP-ASP_C_1484-1498/1t08_ASP-ASP_C_1484-1498_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1t08/1t08_ASP-ASP_C_1484-1498/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1t08/1t08_ASP-ASP_C_1484-1498/1t08_ASP-ASP_C_1484-1498_complex_UNL.pdb'), ('1thr_SER-GLN_I_50-62', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1thr/1thr_SER-GLN_I_50-62/1thr_SER-GLN_I_50-62_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1thr/1thr_SER-GLN_I_50-62/1thr_SER-GLN_I_50-62_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1thr/1thr_SER-GLN_I_50-62/1thr_SER-GLN_I_50-62_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1thr/1thr_SER-GLN_I_50-62/1thr_SER-GLN_I_50-62_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1thr/1thr_SER-GLN_I_50-62/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1thr/1thr_SER-GLN_I_50-62/1thr_SER-GLN_I_50-62_complex_UNL.pdb'), ('1vyj_SER-LYS_L_1-16', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1vyj/1vyj_SER-LYS_L_1-16/1vyj_SER-LYS_L_1-16_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1vyj/1vyj_SER-LYS_L_1-16/1vyj_SER-LYS_L_1-16_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1vyj/1vyj_SER-LYS_L_1-16/1vyj_SER-LYS_L_1-16_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1vyj/1vyj_SER-LYS_L_1-16/1vyj_SER-LYS_L_1-16_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1vyj/1vyj_SER-LYS_L_1-16/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1vyj/1vyj_SER-LYS_L_1-16/1vyj_SER-LYS_L_1-16_complex_UNL.pdb'), ('1x7q_LYS-LYS_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1x7q/1x7q_LYS-LYS_C_1-9/1x7q_LYS-LYS_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1x7q/1x7q_LYS-LYS_C_1-9/1x7q_LYS-LYS_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1x7q/1x7q_LYS-LYS_C_1-9/1x7q_LYS-LYS_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1x7q/1x7q_LYS-LYS_C_1-9/1x7q_LYS-LYS_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1x7q/1x7q_LYS-LYS_C_1-9/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1x7q/1x7q_LYS-LYS_C_1-9/1x7q_LYS-LYS_C_1-9_complex_UNL.pdb'), ('1xh3_LEU-TYR_C_1-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xh3/1xh3_LEU-TYR_C_1-14/1xh3_LEU-TYR_C_1-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xh3/1xh3_LEU-TYR_C_1-14/1xh3_LEU-TYR_C_1-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xh3/1xh3_LEU-TYR_C_1-14/1xh3_LEU-TYR_C_1-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xh3/1xh3_LEU-TYR_C_1-14/1xh3_LEU-TYR_C_1-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xh3/1xh3_LEU-TYR_C_1-14/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xh3/1xh3_LEU-TYR_C_1-14/1xh3_LEU-TYR_C_1-14_complex_UNL.pdb'), ('1xr8_LEU-TYR_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr8/1xr8_LEU-TYR_C_1-9/1xr8_LEU-TYR_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr8/1xr8_LEU-TYR_C_1-9/1xr8_LEU-TYR_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr8/1xr8_LEU-TYR_C_1-9/1xr8_LEU-TYR_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr8/1xr8_LEU-TYR_C_1-9/1xr8_LEU-TYR_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr8/1xr8_LEU-TYR_C_1-9/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr8/1xr8_LEU-TYR_C_1-9/1xr8_LEU-TYR_C_1-9_complex_UNL.pdb'), ('1xr9_ILE-TYR_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr9/1xr9_ILE-TYR_C_1-9/1xr9_ILE-TYR_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr9/1xr9_ILE-TYR_C_1-9/1xr9_ILE-TYR_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr9/1xr9_ILE-TYR_C_1-9/1xr9_ILE-TYR_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr9/1xr9_ILE-TYR_C_1-9/1xr9_ILE-TYR_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr9/1xr9_ILE-TYR_C_1-9/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1xr9/1xr9_ILE-TYR_C_1-9/1xr9_ILE-TYR_C_1-9_complex_UNL.pdb'), ('1y2a_GLY-ILE_P_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y2a/1y2a_GLY-ILE_P_1-10/1y2a_GLY-ILE_P_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y2a/1y2a_GLY-ILE_P_1-10/1y2a_GLY-ILE_P_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y2a/1y2a_GLY-ILE_P_1-10/1y2a_GLY-ILE_P_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y2a/1y2a_GLY-ILE_P_1-10/1y2a_GLY-ILE_P_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y2a/1y2a_GLY-ILE_P_1-10/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y2a/1y2a_GLY-ILE_P_1-10/1y2a_GLY-ILE_P_1-10_complex_UNL.pdb'), ('1y98_PRO-THR_B_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y98/1y98_PRO-THR_B_1-12/1y98_PRO-THR_B_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y98/1y98_PRO-THR_B_1-12/1y98_PRO-THR_B_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y98/1y98_PRO-THR_B_1-12/1y98_PRO-THR_B_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y98/1y98_PRO-THR_B_1-12/1y98_PRO-THR_B_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y98/1y98_PRO-THR_B_1-12/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1y98/1y98_PRO-THR_B_1-12/1y98_PRO-THR_B_1-12_complex_UNL.pdb'), ('1ywh_LYS-LYS_B_501-513', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ywh/1ywh_LYS-LYS_B_501-513/1ywh_LYS-LYS_B_501-513_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ywh/1ywh_LYS-LYS_B_501-513/1ywh_LYS-LYS_B_501-513_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ywh/1ywh_LYS-LYS_B_501-513/1ywh_LYS-LYS_B_501-513_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ywh/1ywh_LYS-LYS_B_501-513/1ywh_LYS-LYS_B_501-513_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ywh/1ywh_LYS-LYS_B_501-513/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1ywh/1ywh_LYS-LYS_B_501-513/1ywh_LYS-LYS_B_501-513_complex_UNL.pdb'), ('1zea_DVA-DSN_A_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zea/1zea_DVA-DSN_A_1-9/1zea_DVA-DSN_A_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zea/1zea_DVA-DSN_A_1-9/1zea_DVA-DSN_A_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zea/1zea_DVA-DSN_A_1-9/1zea_DVA-DSN_A_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zea/1zea_DVA-DSN_A_1-9/1zea_DVA-DSN_A_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zea/1zea_DVA-DSN_A_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zea/1zea_DVA-DSN_A_1-9/1zea_DVA-DSN_A_1-9_complex_UNL.pdb'), ('1zh7_SER-SER_D_16-26', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zh7/1zh7_SER-SER_D_16-26/1zh7_SER-SER_D_16-26_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zh7/1zh7_SER-SER_D_16-26/1zh7_SER-SER_D_16-26_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zh7/1zh7_SER-SER_D_16-26/1zh7_SER-SER_D_16-26_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zh7/1zh7_SER-SER_D_16-26/1zh7_SER-SER_D_16-26_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zh7/1zh7_SER-SER_D_16-26/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zh7/1zh7_SER-SER_D_16-26/1zh7_SER-SER_D_16-26_complex_UNL.pdb'), ('1zhk_LEU-TYR_C_1-13', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhk/1zhk_LEU-TYR_C_1-13/1zhk_LEU-TYR_C_1-13_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhk/1zhk_LEU-TYR_C_1-13/1zhk_LEU-TYR_C_1-13_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhk/1zhk_LEU-TYR_C_1-13/1zhk_LEU-TYR_C_1-13_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhk/1zhk_LEU-TYR_C_1-13/1zhk_LEU-TYR_C_1-13_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhk/1zhk_LEU-TYR_C_1-13/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhk/1zhk_LEU-TYR_C_1-13/1zhk_LEU-TYR_C_1-13_complex_UNL.pdb'), ('1zhl_LEU-TYR_C_1-13', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhl/1zhl_LEU-TYR_C_1-13/1zhl_LEU-TYR_C_1-13_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhl/1zhl_LEU-TYR_C_1-13/1zhl_LEU-TYR_C_1-13_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhl/1zhl_LEU-TYR_C_1-13/1zhl_LEU-TYR_C_1-13_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhl/1zhl_LEU-TYR_C_1-13/1zhl_LEU-TYR_C_1-13_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhl/1zhl_LEU-TYR_C_1-13/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/1zhl/1zhl_LEU-TYR_C_1-13/1zhl_LEU-TYR_C_1-13_complex_UNL.pdb'), ('2a0t_ASN-SER_B_165-174', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a0t/2a0t_ASN-SER_B_165-174/2a0t_ASN-SER_B_165-174_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a0t/2a0t_ASN-SER_B_165-174/2a0t_ASN-SER_B_165-174_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a0t/2a0t_ASN-SER_B_165-174/2a0t_ASN-SER_B_165-174_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a0t/2a0t_ASN-SER_B_165-174/2a0t_ASN-SER_B_165-174_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a0t/2a0t_ASN-SER_B_165-174/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a0t/2a0t_ASN-SER_B_165-174/2a0t_ASN-SER_B_165-174_complex_UNL.pdb'), ('2a3i_GLN-GLU_B_1430-1441', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a3i/2a3i_GLN-GLU_B_1430-1441/2a3i_GLN-GLU_B_1430-1441_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a3i/2a3i_GLN-GLU_B_1430-1441/2a3i_GLN-GLU_B_1430-1441_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a3i/2a3i_GLN-GLU_B_1430-1441/2a3i_GLN-GLU_B_1430-1441_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a3i/2a3i_GLN-GLU_B_1430-1441/2a3i_GLN-GLU_B_1430-1441_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a3i/2a3i_GLN-GLU_B_1430-1441/interaction.xml', 'GLN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2a3i/2a3i_GLN-GLU_B_1430-1441/2a3i_GLN-GLU_B_1430-1441_complex_UNL.pdb'), ('2axi_PRO-DPR_B_20-29', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2axi/2axi_PRO-DPR_B_20-29/2axi_PRO-DPR_B_20-29_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2axi/2axi_PRO-DPR_B_20-29/2axi_PRO-DPR_B_20-29_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2axi/2axi_PRO-DPR_B_20-29/2axi_PRO-DPR_B_20-29_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2axi/2axi_PRO-DPR_B_20-29/2axi_PRO-DPR_B_20-29_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2axi/2axi_PRO-DPR_B_20-29/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2axi/2axi_PRO-DPR_B_20-29/2axi_PRO-DPR_B_20-29_complex_UNL.pdb'), ('2bj4_LEU-ALA_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bj4/2bj4_LEU-ALA_C_1-11/2bj4_LEU-ALA_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bj4/2bj4_LEU-ALA_C_1-11/2bj4_LEU-ALA_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bj4/2bj4_LEU-ALA_C_1-11/2bj4_LEU-ALA_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bj4/2bj4_LEU-ALA_C_1-11/2bj4_LEU-ALA_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bj4/2bj4_LEU-ALA_C_1-11/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bj4/2bj4_LEU-ALA_C_1-11/2bj4_LEU-ALA_C_1-11_complex_UNL.pdb'), ('2bz8_PRO-ARG_C_902-912', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bz8/2bz8_PRO-ARG_C_902-912/2bz8_PRO-ARG_C_902-912_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bz8/2bz8_PRO-ARG_C_902-912/2bz8_PRO-ARG_C_902-912_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bz8/2bz8_PRO-ARG_C_902-912/2bz8_PRO-ARG_C_902-912_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bz8/2bz8_PRO-ARG_C_902-912/2bz8_PRO-ARG_C_902-912_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bz8/2bz8_PRO-ARG_C_902-912/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2bz8/2bz8_PRO-ARG_C_902-912/2bz8_PRO-ARG_C_902-912_complex_UNL.pdb'), ('2c2l_ASP-ASP_G_501-509', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2c2l/2c2l_ASP-ASP_G_501-509/2c2l_ASP-ASP_G_501-509_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2c2l/2c2l_ASP-ASP_G_501-509/2c2l_ASP-ASP_G_501-509_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2c2l/2c2l_ASP-ASP_G_501-509/2c2l_ASP-ASP_G_501-509_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2c2l/2c2l_ASP-ASP_G_501-509/2c2l_ASP-ASP_G_501-509_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2c2l/2c2l_ASP-ASP_G_501-509/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2c2l/2c2l_ASP-ASP_G_501-509/2c2l_ASP-ASP_G_501-509_complex_UNL.pdb'), ('2clv_SER-LEU_C_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2clv/2clv_SER-LEU_C_1-8/2clv_SER-LEU_C_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2clv/2clv_SER-LEU_C_1-8/2clv_SER-LEU_C_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2clv/2clv_SER-LEU_C_1-8/2clv_SER-LEU_C_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2clv/2clv_SER-LEU_C_1-8/2clv_SER-LEU_C_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2clv/2clv_SER-LEU_C_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2clv/2clv_SER-LEU_C_1-8/2clv_SER-LEU_C_1-8_complex_UNL.pdb'), ('2cv3_THR-ASP_B_256-266', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2cv3/2cv3_THR-ASP_B_256-266/2cv3_THR-ASP_B_256-266_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2cv3/2cv3_THR-ASP_B_256-266/2cv3_THR-ASP_B_256-266_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2cv3/2cv3_THR-ASP_B_256-266/2cv3_THR-ASP_B_256-266_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2cv3/2cv3_THR-ASP_B_256-266/2cv3_THR-ASP_B_256-266_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2cv3/2cv3_THR-ASP_B_256-266/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2cv3/2cv3_THR-ASP_B_256-266/2cv3_THR-ASP_B_256-266_complex_UNL.pdb'), ('2fo4_SER-LEU_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fo4/2fo4_SER-LEU_P_1-8/2fo4_SER-LEU_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fo4/2fo4_SER-LEU_P_1-8/2fo4_SER-LEU_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fo4/2fo4_SER-LEU_P_1-8/2fo4_SER-LEU_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fo4/2fo4_SER-LEU_P_1-8/2fo4_SER-LEU_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fo4/2fo4_SER-LEU_P_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fo4/2fo4_SER-LEU_P_1-8/2fo4_SER-LEU_P_1-8_complex_UNL.pdb'), ('2fx7_ASN-LYS_P_671-686', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fx7/2fx7_ASN-LYS_P_671-686/2fx7_ASN-LYS_P_671-686_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fx7/2fx7_ASN-LYS_P_671-686/2fx7_ASN-LYS_P_671-686_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fx7/2fx7_ASN-LYS_P_671-686/2fx7_ASN-LYS_P_671-686_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fx7/2fx7_ASN-LYS_P_671-686/2fx7_ASN-LYS_P_671-686_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fx7/2fx7_ASN-LYS_P_671-686/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2fx7/2fx7_ASN-LYS_P_671-686/2fx7_ASN-LYS_P_671-686_complex_UNL.pdb'), ('2hug_ALA-SER_B_1-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2hug/2hug_ALA-SER_B_1-14/2hug_ALA-SER_B_1-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2hug/2hug_ALA-SER_B_1-14/2hug_ALA-SER_B_1-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2hug/2hug_ALA-SER_B_1-14/2hug_ALA-SER_B_1-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2hug/2hug_ALA-SER_B_1-14/2hug_ALA-SER_B_1-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2hug/2hug_ALA-SER_B_1-14/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2hug/2hug_ALA-SER_B_1-14/2hug_ALA-SER_B_1-14_complex_UNL.pdb'), ('2jo9_GLU-ASP_B_101-109', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jo9/2jo9_GLU-ASP_B_101-109/2jo9_GLU-ASP_B_101-109_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jo9/2jo9_GLU-ASP_B_101-109/2jo9_GLU-ASP_B_101-109_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jo9/2jo9_GLU-ASP_B_101-109/2jo9_GLU-ASP_B_101-109_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jo9/2jo9_GLU-ASP_B_101-109/2jo9_GLU-ASP_B_101-109_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jo9/2jo9_GLU-ASP_B_101-109/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jo9/2jo9_GLU-ASP_B_101-109/2jo9_GLU-ASP_B_101-109_complex_UNL.pdb'), ('2jqi_ASN-THR_B_3-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jqi/2jqi_ASN-THR_B_3-12/2jqi_ASN-THR_B_3-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jqi/2jqi_ASN-THR_B_3-12/2jqi_ASN-THR_B_3-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jqi/2jqi_ASN-THR_B_3-12/2jqi_ASN-THR_B_3-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jqi/2jqi_ASN-THR_B_3-12/2jqi_ASN-THR_B_3-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jqi/2jqi_ASN-THR_B_3-12/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jqi/2jqi_ASN-THR_B_3-12/2jqi_ASN-THR_B_3-12_complex_UNL.pdb'), ('2jql_ASN-THR_B_3-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jql/2jql_ASN-THR_B_3-12/2jql_ASN-THR_B_3-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jql/2jql_ASN-THR_B_3-12/2jql_ASN-THR_B_3-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jql/2jql_ASN-THR_B_3-12/2jql_ASN-THR_B_3-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jql/2jql_ASN-THR_B_3-12/2jql_ASN-THR_B_3-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jql/2jql_ASN-THR_B_3-12/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jql/2jql_ASN-THR_B_3-12/2jql_ASN-THR_B_3-12_complex_UNL.pdb'), ('2jup_GLY-PRO_P_101-109', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jup/2jup_GLY-PRO_P_101-109/2jup_GLY-PRO_P_101-109_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jup/2jup_GLY-PRO_P_101-109/2jup_GLY-PRO_P_101-109_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jup/2jup_GLY-PRO_P_101-109/2jup_GLY-PRO_P_101-109_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jup/2jup_GLY-PRO_P_101-109/2jup_GLY-PRO_P_101-109_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jup/2jup_GLY-PRO_P_101-109/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2jup/2jup_GLY-PRO_P_101-109/2jup_GLY-PRO_P_101-109_complex_UNL.pdb'), ('2k00_GLY-TYR_B_367-381', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k00/2k00_GLY-TYR_B_367-381/2k00_GLY-TYR_B_367-381_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k00/2k00_GLY-TYR_B_367-381/2k00_GLY-TYR_B_367-381_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k00/2k00_GLY-TYR_B_367-381/2k00_GLY-TYR_B_367-381_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k00/2k00_GLY-TYR_B_367-381/2k00_GLY-TYR_B_367-381_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k00/2k00_GLY-TYR_B_367-381/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k00/2k00_GLY-TYR_B_367-381/2k00_GLY-TYR_B_367-381_complex_UNL.pdb'), ('2k2r_ASP-GLU_B_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k2r/2k2r_ASP-GLU_B_1-10/2k2r_ASP-GLU_B_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k2r/2k2r_ASP-GLU_B_1-10/2k2r_ASP-GLU_B_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k2r/2k2r_ASP-GLU_B_1-10/2k2r_ASP-GLU_B_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k2r/2k2r_ASP-GLU_B_1-10/2k2r_ASP-GLU_B_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k2r/2k2r_ASP-GLU_B_1-10/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k2r/2k2r_ASP-GLU_B_1-10/2k2r_ASP-GLU_B_1-10_complex_UNL.pdb'), ('2k7l_GLU-VAL_B_582-600', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k7l/2k7l_GLU-VAL_B_582-600/2k7l_GLU-VAL_B_582-600_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k7l/2k7l_GLU-VAL_B_582-600/2k7l_GLU-VAL_B_582-600_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k7l/2k7l_GLU-VAL_B_582-600/2k7l_GLU-VAL_B_582-600_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k7l/2k7l_GLU-VAL_B_582-600/2k7l_GLU-VAL_B_582-600_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k7l/2k7l_GLU-VAL_B_582-600/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2k7l/2k7l_GLU-VAL_B_582-600/2k7l_GLU-VAL_B_582-600_complex_UNL.pdb'), ('2kbr_ASP-LYS_B_100-117', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbr/2kbr_ASP-LYS_B_100-117/2kbr_ASP-LYS_B_100-117_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbr/2kbr_ASP-LYS_B_100-117/2kbr_ASP-LYS_B_100-117_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbr/2kbr_ASP-LYS_B_100-117/2kbr_ASP-LYS_B_100-117_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbr/2kbr_ASP-LYS_B_100-117/2kbr_ASP-LYS_B_100-117_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbr/2kbr_ASP-LYS_B_100-117/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbr/2kbr_ASP-LYS_B_100-117/2kbr_ASP-LYS_B_100-117_complex_UNL.pdb'), ('2kbs_THR-LEU_B_111-118', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbs/2kbs_THR-LEU_B_111-118/2kbs_THR-LEU_B_111-118_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbs/2kbs_THR-LEU_B_111-118/2kbs_THR-LEU_B_111-118_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbs/2kbs_THR-LEU_B_111-118/2kbs_THR-LEU_B_111-118_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbs/2kbs_THR-LEU_B_111-118/2kbs_THR-LEU_B_111-118_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbs/2kbs_THR-LEU_B_111-118/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kbs/2kbs_THR-LEU_B_111-118/2kbs_THR-LEU_B_111-118_complex_UNL.pdb'), ('2ke1_ALA-SER_B_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ke1/2ke1_ALA-SER_B_1-10/2ke1_ALA-SER_B_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ke1/2ke1_ALA-SER_B_1-10/2ke1_ALA-SER_B_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ke1/2ke1_ALA-SER_B_1-10/2ke1_ALA-SER_B_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ke1/2ke1_ALA-SER_B_1-10/2ke1_ALA-SER_B_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ke1/2ke1_ALA-SER_B_1-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ke1/2ke1_ALA-SER_B_1-10/2ke1_ALA-SER_B_1-10_complex_UNL.pdb'), ('2kff_PHE-LYS_B_143-154', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kff/2kff_PHE-LYS_B_143-154/2kff_PHE-LYS_B_143-154_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kff/2kff_PHE-LYS_B_143-154/2kff_PHE-LYS_B_143-154_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kff/2kff_PHE-LYS_B_143-154/2kff_PHE-LYS_B_143-154_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kff/2kff_PHE-LYS_B_143-154/2kff_PHE-LYS_B_143-154_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kff/2kff_PHE-LYS_B_143-154/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kff/2kff_PHE-LYS_B_143-154/2kff_PHE-LYS_B_143-154_complex_UNL.pdb'), ('2kfg_PHE-LYS_B_143-154', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfg/2kfg_PHE-LYS_B_143-154/2kfg_PHE-LYS_B_143-154_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfg/2kfg_PHE-LYS_B_143-154/2kfg_PHE-LYS_B_143-154_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfg/2kfg_PHE-LYS_B_143-154/2kfg_PHE-LYS_B_143-154_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfg/2kfg_PHE-LYS_B_143-154/2kfg_PHE-LYS_B_143-154_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfg/2kfg_PHE-LYS_B_143-154/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfg/2kfg_PHE-LYS_B_143-154/2kfg_PHE-LYS_B_143-154_complex_UNL.pdb'), ('2kfh_PHE-LYS_B_143-154', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfh/2kfh_PHE-LYS_B_143-154/2kfh_PHE-LYS_B_143-154_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfh/2kfh_PHE-LYS_B_143-154/2kfh_PHE-LYS_B_143-154_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfh/2kfh_PHE-LYS_B_143-154/2kfh_PHE-LYS_B_143-154_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfh/2kfh_PHE-LYS_B_143-154/2kfh_PHE-LYS_B_143-154_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfh/2kfh_PHE-LYS_B_143-154/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kfh/2kfh_PHE-LYS_B_143-154/2kfh_PHE-LYS_B_143-154_complex_UNL.pdb'), ('2khh_ASP-LYS_B_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2khh/2khh_ASP-LYS_B_1-9/2khh_ASP-LYS_B_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2khh/2khh_ASP-LYS_B_1-9/2khh_ASP-LYS_B_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2khh/2khh_ASP-LYS_B_1-9/2khh_ASP-LYS_B_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2khh/2khh_ASP-LYS_B_1-9/2khh_ASP-LYS_B_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2khh/2khh_ASP-LYS_B_1-9/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2khh/2khh_ASP-LYS_B_1-9/2khh_ASP-LYS_B_1-9_complex_UNL.pdb'), ('2koh_MET-ILE_B_769-784', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2koh/2koh_MET-ILE_B_769-784/2koh_MET-ILE_B_769-784_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2koh/2koh_MET-ILE_B_769-784/2koh_MET-ILE_B_769-784_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2koh/2koh_MET-ILE_B_769-784/2koh_MET-ILE_B_769-784_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2koh/2koh_MET-ILE_B_769-784/2koh_MET-ILE_B_769-784_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2koh/2koh_MET-ILE_B_769-784/interaction.xml', 'MET', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2koh/2koh_MET-ILE_B_769-784/2koh_MET-ILE_B_769-784_complex_UNL.pdb'), ('2kpl_ARG-VAL_B_151-161', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kpl/2kpl_ARG-VAL_B_151-161/2kpl_ARG-VAL_B_151-161_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kpl/2kpl_ARG-VAL_B_151-161/2kpl_ARG-VAL_B_151-161_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kpl/2kpl_ARG-VAL_B_151-161/2kpl_ARG-VAL_B_151-161_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kpl/2kpl_ARG-VAL_B_151-161/2kpl_ARG-VAL_B_151-161_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kpl/2kpl_ARG-VAL_B_151-161/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kpl/2kpl_ARG-VAL_B_151-161/2kpl_ARG-VAL_B_151-161_complex_UNL.pdb'), ('2krd_ARG-LYS_I_147-163', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2krd/2krd_ARG-LYS_I_147-163/2krd_ARG-LYS_I_147-163_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2krd/2krd_ARG-LYS_I_147-163/2krd_ARG-LYS_I_147-163_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2krd/2krd_ARG-LYS_I_147-163/2krd_ARG-LYS_I_147-163_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2krd/2krd_ARG-LYS_I_147-163/2krd_ARG-LYS_I_147-163_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2krd/2krd_ARG-LYS_I_147-163/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2krd/2krd_ARG-LYS_I_147-163/2krd_ARG-LYS_I_147-163_complex_UNL.pdb'), ('2ks9_ARG-MET_B_365-375', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ks9/2ks9_ARG-MET_B_365-375/2ks9_ARG-MET_B_365-375_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ks9/2ks9_ARG-MET_B_365-375/2ks9_ARG-MET_B_365-375_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ks9/2ks9_ARG-MET_B_365-375/2ks9_ARG-MET_B_365-375_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ks9/2ks9_ARG-MET_B_365-375/2ks9_ARG-MET_B_365-375_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ks9/2ks9_ARG-MET_B_365-375/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ks9/2ks9_ARG-MET_B_365-375/2ks9_ARG-MET_B_365-375_complex_UNL.pdb'), ('2ksa_ARG-MET_B_365-375', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksa/2ksa_ARG-MET_B_365-375/2ksa_ARG-MET_B_365-375_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksa/2ksa_ARG-MET_B_365-375/2ksa_ARG-MET_B_365-375_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksa/2ksa_ARG-MET_B_365-375/2ksa_ARG-MET_B_365-375_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksa/2ksa_ARG-MET_B_365-375/2ksa_ARG-MET_B_365-375_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksa/2ksa_ARG-MET_B_365-375/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksa/2ksa_ARG-MET_B_365-375/2ksa_ARG-MET_B_365-375_complex_UNL.pdb'), ('2ksb_ARG-MET_B_365-375', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksb/2ksb_ARG-MET_B_365-375/2ksb_ARG-MET_B_365-375_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksb/2ksb_ARG-MET_B_365-375/2ksb_ARG-MET_B_365-375_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksb/2ksb_ARG-MET_B_365-375/2ksb_ARG-MET_B_365-375_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksb/2ksb_ARG-MET_B_365-375/2ksb_ARG-MET_B_365-375_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksb/2ksb_ARG-MET_B_365-375/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksb/2ksb_ARG-MET_B_365-375/2ksb_ARG-MET_B_365-375_complex_UNL.pdb'), ('2ksp_LEU-ASP_B_143-157', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksp/2ksp_LEU-ASP_B_143-157/2ksp_LEU-ASP_B_143-157_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksp/2ksp_LEU-ASP_B_143-157/2ksp_LEU-ASP_B_143-157_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksp/2ksp_LEU-ASP_B_143-157/2ksp_LEU-ASP_B_143-157_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksp/2ksp_LEU-ASP_B_143-157/2ksp_LEU-ASP_B_143-157_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksp/2ksp_LEU-ASP_B_143-157/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ksp/2ksp_LEU-ASP_B_143-157/2ksp_LEU-ASP_B_143-157_complex_UNL.pdb'), ('2kvm_ALA-PRO_B_15-30', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kvm/2kvm_ALA-PRO_B_15-30/2kvm_ALA-PRO_B_15-30_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kvm/2kvm_ALA-PRO_B_15-30/2kvm_ALA-PRO_B_15-30_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kvm/2kvm_ALA-PRO_B_15-30/2kvm_ALA-PRO_B_15-30_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kvm/2kvm_ALA-PRO_B_15-30/2kvm_ALA-PRO_B_15-30_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kvm/2kvm_ALA-PRO_B_15-30/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kvm/2kvm_ALA-PRO_B_15-30/2kvm_ALA-PRO_B_15-30_complex_UNL.pdb'), ('2kwn_GLY-ARG_B_9-23', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kwn/2kwn_GLY-ARG_B_9-23/2kwn_GLY-ARG_B_9-23_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kwn/2kwn_GLY-ARG_B_9-23/2kwn_GLY-ARG_B_9-23_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kwn/2kwn_GLY-ARG_B_9-23/2kwn_GLY-ARG_B_9-23_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kwn/2kwn_GLY-ARG_B_9-23/2kwn_GLY-ARG_B_9-23_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kwn/2kwn_GLY-ARG_B_9-23/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kwn/2kwn_GLY-ARG_B_9-23/2kwn_GLY-ARG_B_9-23_complex_UNL.pdb'), ('2kzu_GLY-TRP_B_22-39', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kzu/2kzu_GLY-TRP_B_22-39/2kzu_GLY-TRP_B_22-39_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kzu/2kzu_GLY-TRP_B_22-39/2kzu_GLY-TRP_B_22-39_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kzu/2kzu_GLY-TRP_B_22-39/2kzu_GLY-TRP_B_22-39_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kzu/2kzu_GLY-TRP_B_22-39/2kzu_GLY-TRP_B_22-39_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kzu/2kzu_GLY-TRP_B_22-39/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2kzu/2kzu_GLY-TRP_B_22-39/2kzu_GLY-TRP_B_22-39_complex_UNL.pdb'), ('2l11_ALA-ALA_B_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l11/2l11_ALA-ALA_B_1-15/2l11_ALA-ALA_B_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l11/2l11_ALA-ALA_B_1-15/2l11_ALA-ALA_B_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l11/2l11_ALA-ALA_B_1-15/2l11_ALA-ALA_B_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l11/2l11_ALA-ALA_B_1-15/2l11_ALA-ALA_B_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l11/2l11_ALA-ALA_B_1-15/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l11/2l11_ALA-ALA_B_1-15/2l11_ALA-ALA_B_1-15_complex_UNL.pdb'), ('2l12_ALA-ALA_B_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l12/2l12_ALA-ALA_B_1-15/2l12_ALA-ALA_B_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l12/2l12_ALA-ALA_B_1-15/2l12_ALA-ALA_B_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l12/2l12_ALA-ALA_B_1-15/2l12_ALA-ALA_B_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l12/2l12_ALA-ALA_B_1-15/2l12_ALA-ALA_B_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l12/2l12_ALA-ALA_B_1-15/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l12/2l12_ALA-ALA_B_1-15/2l12_ALA-ALA_B_1-15_complex_UNL.pdb'), ('2l1b_GLN-GLY_B_19-33', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l1b/2l1b_GLN-GLY_B_19-33/2l1b_GLN-GLY_B_19-33_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l1b/2l1b_GLN-GLY_B_19-33/2l1b_GLN-GLY_B_19-33_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l1b/2l1b_GLN-GLY_B_19-33/2l1b_GLN-GLY_B_19-33_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l1b/2l1b_GLN-GLY_B_19-33/2l1b_GLN-GLY_B_19-33_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l1b/2l1b_GLN-GLY_B_19-33/interaction.xml', 'GLN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l1b/2l1b_GLN-GLY_B_19-33/2l1b_GLN-GLY_B_19-33_complex_UNL.pdb'), ('2l3r_ALA-THR_B_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l3r/2l3r_ALA-THR_B_1-11/2l3r_ALA-THR_B_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l3r/2l3r_ALA-THR_B_1-11/2l3r_ALA-THR_B_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l3r/2l3r_ALA-THR_B_1-11/2l3r_ALA-THR_B_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l3r/2l3r_ALA-THR_B_1-11/2l3r_ALA-THR_B_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l3r/2l3r_ALA-THR_B_1-11/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2l3r/2l3r_ALA-THR_B_1-11/2l3r_ALA-THR_B_1-11_complex_UNL.pdb'), ('2las_ASP-LYS_B_2705-2717', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2las/2las_ASP-LYS_B_2705-2717/2las_ASP-LYS_B_2705-2717_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2las/2las_ASP-LYS_B_2705-2717/2las_ASP-LYS_B_2705-2717_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2las/2las_ASP-LYS_B_2705-2717/2las_ASP-LYS_B_2705-2717_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2las/2las_ASP-LYS_B_2705-2717/2las_ASP-LYS_B_2705-2717_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2las/2las_ASP-LYS_B_2705-2717/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2las/2las_ASP-LYS_B_2705-2717/2las_ASP-LYS_B_2705-2717_complex_UNL.pdb'), ('2lbm_ALA-ALA_C_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lbm/2lbm_ALA-ALA_C_1-15/2lbm_ALA-ALA_C_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lbm/2lbm_ALA-ALA_C_1-15/2lbm_ALA-ALA_C_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lbm/2lbm_ALA-ALA_C_1-15/2lbm_ALA-ALA_C_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lbm/2lbm_ALA-ALA_C_1-15/2lbm_ALA-ALA_C_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lbm/2lbm_ALA-ALA_C_1-15/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lbm/2lbm_ALA-ALA_C_1-15/2lbm_ALA-ALA_C_1-15_complex_UNL.pdb'), ('2lgf_ALA-LYS_B_349-363', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lgf/2lgf_ALA-LYS_B_349-363/2lgf_ALA-LYS_B_349-363_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lgf/2lgf_ALA-LYS_B_349-363/2lgf_ALA-LYS_B_349-363_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lgf/2lgf_ALA-LYS_B_349-363/2lgf_ALA-LYS_B_349-363_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lgf/2lgf_ALA-LYS_B_349-363/2lgf_ALA-LYS_B_349-363_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lgf/2lgf_ALA-LYS_B_349-363/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lgf/2lgf_ALA-LYS_B_349-363/2lgf_ALA-LYS_B_349-363_complex_UNL.pdb'), ('2ll6_LEU-LYS_B_149-165', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll6/2ll6_LEU-LYS_B_149-165/2ll6_LEU-LYS_B_149-165_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll6/2ll6_LEU-LYS_B_149-165/2ll6_LEU-LYS_B_149-165_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll6/2ll6_LEU-LYS_B_149-165/2ll6_LEU-LYS_B_149-165_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll6/2ll6_LEU-LYS_B_149-165/2ll6_LEU-LYS_B_149-165_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll6/2ll6_LEU-LYS_B_149-165/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll6/2ll6_LEU-LYS_B_149-165/2ll6_LEU-LYS_B_149-165_complex_UNL.pdb'), ('2ll7_LYS-LEU_B_149-165', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll7/2ll7_LYS-LEU_B_149-165/2ll7_LYS-LEU_B_149-165_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll7/2ll7_LYS-LEU_B_149-165/2ll7_LYS-LEU_B_149-165_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll7/2ll7_LYS-LEU_B_149-165/2ll7_LYS-LEU_B_149-165_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll7/2ll7_LYS-LEU_B_149-165/2ll7_LYS-LEU_B_149-165_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll7/2ll7_LYS-LEU_B_149-165/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ll7/2ll7_LYS-LEU_B_149-165/2ll7_LYS-LEU_B_149-165_complex_UNL.pdb'), ('2llo_ARG-SER_B_287-305', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2llo/2llo_ARG-SER_B_287-305/2llo_ARG-SER_B_287-305_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2llo/2llo_ARG-SER_B_287-305/2llo_ARG-SER_B_287-305_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2llo/2llo_ARG-SER_B_287-305/2llo_ARG-SER_B_287-305_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2llo/2llo_ARG-SER_B_287-305/2llo_ARG-SER_B_287-305_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2llo/2llo_ARG-SER_B_287-305/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2llo/2llo_ARG-SER_B_287-305/2llo_ARG-SER_B_287-305_complex_UNL.pdb'), ('2lnw_GLU-GLY_B_1404-1412', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lnw/2lnw_GLU-GLY_B_1404-1412/2lnw_GLU-GLY_B_1404-1412_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lnw/2lnw_GLU-GLY_B_1404-1412/2lnw_GLU-GLY_B_1404-1412_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lnw/2lnw_GLU-GLY_B_1404-1412/2lnw_GLU-GLY_B_1404-1412_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lnw/2lnw_GLU-GLY_B_1404-1412/2lnw_GLU-GLY_B_1404-1412_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lnw/2lnw_GLU-GLY_B_1404-1412/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lnw/2lnw_GLU-GLY_B_1404-1412/2lnw_GLU-GLY_B_1404-1412_complex_UNL.pdb'), ('2lo6_TYR-SER_B_155-168', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lo6/2lo6_TYR-SER_B_155-168/2lo6_TYR-SER_B_155-168_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lo6/2lo6_TYR-SER_B_155-168/2lo6_TYR-SER_B_155-168_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lo6/2lo6_TYR-SER_B_155-168/2lo6_TYR-SER_B_155-168_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lo6/2lo6_TYR-SER_B_155-168/2lo6_TYR-SER_B_155-168_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lo6/2lo6_TYR-SER_B_155-168/interaction.xml', 'TYR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lo6/2lo6_TYR-SER_B_155-168/2lo6_TYR-SER_B_155-168_complex_UNL.pdb'), ('2loz_GLU-ASN_B_367-380', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2loz/2loz_GLU-ASN_B_367-380/2loz_GLU-ASN_B_367-380_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2loz/2loz_GLU-ASN_B_367-380/2loz_GLU-ASN_B_367-380_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2loz/2loz_GLU-ASN_B_367-380/2loz_GLU-ASN_B_367-380_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2loz/2loz_GLU-ASN_B_367-380/2loz_GLU-ASN_B_367-380_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2loz/2loz_GLU-ASN_B_367-380/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2loz/2loz_GLU-ASN_B_367-380/2loz_GLU-ASN_B_367-380_complex_UNL.pdb'), ('2lsk_GLN-LEU_B_524-539', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsk/2lsk_GLN-LEU_B_524-539/2lsk_GLN-LEU_B_524-539_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsk/2lsk_GLN-LEU_B_524-539/2lsk_GLN-LEU_B_524-539_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsk/2lsk_GLN-LEU_B_524-539/2lsk_GLN-LEU_B_524-539_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsk/2lsk_GLN-LEU_B_524-539/2lsk_GLN-LEU_B_524-539_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsk/2lsk_GLN-LEU_B_524-539/interaction.xml', 'GLN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsk/2lsk_GLN-LEU_B_524-539/2lsk_GLN-LEU_B_524-539_complex_UNL.pdb'), ('2lsp_ARG-SER_A_1-13', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsp/2lsp_ARG-SER_A_1-13/2lsp_ARG-SER_A_1-13_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsp/2lsp_ARG-SER_A_1-13/2lsp_ARG-SER_A_1-13_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsp/2lsp_ARG-SER_A_1-13/2lsp_ARG-SER_A_1-13_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsp/2lsp_ARG-SER_A_1-13/2lsp_ARG-SER_A_1-13_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsp/2lsp_ARG-SER_A_1-13/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsp/2lsp_ARG-SER_A_1-13/2lsp_ARG-SER_A_1-13_complex_UNL.pdb'), ('2lsr_GLY-LYS_B_98-113', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsr/2lsr_GLY-LYS_B_98-113/2lsr_GLY-LYS_B_98-113_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsr/2lsr_GLY-LYS_B_98-113/2lsr_GLY-LYS_B_98-113_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsr/2lsr_GLY-LYS_B_98-113/2lsr_GLY-LYS_B_98-113_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsr/2lsr_GLY-LYS_B_98-113/2lsr_GLY-LYS_B_98-113_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsr/2lsr_GLY-LYS_B_98-113/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsr/2lsr_GLY-LYS_B_98-113/2lsr_GLY-LYS_B_98-113_complex_UNL.pdb'), ('2lsv_ALA-ASP_B_701-709', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsv/2lsv_ALA-ASP_B_701-709/2lsv_ALA-ASP_B_701-709_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsv/2lsv_ALA-ASP_B_701-709/2lsv_ALA-ASP_B_701-709_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsv/2lsv_ALA-ASP_B_701-709/2lsv_ALA-ASP_B_701-709_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsv/2lsv_ALA-ASP_B_701-709/2lsv_ALA-ASP_B_701-709_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsv/2lsv_ALA-ASP_B_701-709/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lsv/2lsv_ALA-ASP_B_701-709/2lsv_ALA-ASP_B_701-709_complex_UNL.pdb'), ('2lto_TYR-PRO_B_611-623', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lto/2lto_TYR-PRO_B_611-623/2lto_TYR-PRO_B_611-623_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lto/2lto_TYR-PRO_B_611-623/2lto_TYR-PRO_B_611-623_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lto/2lto_TYR-PRO_B_611-623/2lto_TYR-PRO_B_611-623_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lto/2lto_TYR-PRO_B_611-623/2lto_TYR-PRO_B_611-623_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lto/2lto_TYR-PRO_B_611-623/interaction.xml', 'TYR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lto/2lto_TYR-PRO_B_611-623/2lto_TYR-PRO_B_611-623_complex_UNL.pdb'), ('2ltv_SER-ASP_B_206-217', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltv/2ltv_SER-ASP_B_206-217/2ltv_SER-ASP_B_206-217_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltv/2ltv_SER-ASP_B_206-217/2ltv_SER-ASP_B_206-217_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltv/2ltv_SER-ASP_B_206-217/2ltv_SER-ASP_B_206-217_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltv/2ltv_SER-ASP_B_206-217/2ltv_SER-ASP_B_206-217_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltv/2ltv_SER-ASP_B_206-217/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltv/2ltv_SER-ASP_B_206-217/2ltv_SER-ASP_B_206-217_complex_UNL.pdb'), ('2ltw_GLY-ASP_B_204-217', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltw/2ltw_GLY-ASP_B_204-217/2ltw_GLY-ASP_B_204-217_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltw/2ltw_GLY-ASP_B_204-217/2ltw_GLY-ASP_B_204-217_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltw/2ltw_GLY-ASP_B_204-217/2ltw_GLY-ASP_B_204-217_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltw/2ltw_GLY-ASP_B_204-217/2ltw_GLY-ASP_B_204-217_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltw/2ltw_GLY-ASP_B_204-217/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltw/2ltw_GLY-ASP_B_204-217/2ltw_GLY-ASP_B_204-217_complex_UNL.pdb'), ('2ltx_GLU-ASP_B_203-217', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltx/2ltx_GLU-ASP_B_203-217/2ltx_GLU-ASP_B_203-217_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltx/2ltx_GLU-ASP_B_203-217/2ltx_GLU-ASP_B_203-217_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltx/2ltx_GLU-ASP_B_203-217/2ltx_GLU-ASP_B_203-217_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltx/2ltx_GLU-ASP_B_203-217/2ltx_GLU-ASP_B_203-217_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltx/2ltx_GLU-ASP_B_203-217/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltx/2ltx_GLU-ASP_B_203-217/2ltx_GLU-ASP_B_203-217_complex_UNL.pdb'), ('2lty_GLU-ASP_B_203-217', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lty/2lty_GLU-ASP_B_203-217/2lty_GLU-ASP_B_203-217_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lty/2lty_GLU-ASP_B_203-217/2lty_GLU-ASP_B_203-217_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lty/2lty_GLU-ASP_B_203-217/2lty_GLU-ASP_B_203-217_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lty/2lty_GLU-ASP_B_203-217/2lty_GLU-ASP_B_203-217_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lty/2lty_GLU-ASP_B_203-217/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lty/2lty_GLU-ASP_B_203-217/2lty_GLU-ASP_B_203-217_complex_UNL.pdb'), ('2ltz_GLU-ASP_B_203-217', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltz/2ltz_GLU-ASP_B_203-217/2ltz_GLU-ASP_B_203-217_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltz/2ltz_GLU-ASP_B_203-217/2ltz_GLU-ASP_B_203-217_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltz/2ltz_GLU-ASP_B_203-217/2ltz_GLU-ASP_B_203-217_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltz/2ltz_GLU-ASP_B_203-217/2ltz_GLU-ASP_B_203-217_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltz/2ltz_GLU-ASP_B_203-217/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ltz/2ltz_GLU-ASP_B_203-217/2ltz_GLU-ASP_B_203-217_complex_UNL.pdb'), ('2lyw_PCA-LEU_B_1-13', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lyw/2lyw_PCA-LEU_B_1-13/2lyw_PCA-LEU_B_1-13_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lyw/2lyw_PCA-LEU_B_1-13/2lyw_PCA-LEU_B_1-13_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lyw/2lyw_PCA-LEU_B_1-13/2lyw_PCA-LEU_B_1-13_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lyw/2lyw_PCA-LEU_B_1-13/2lyw_PCA-LEU_B_1-13_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lyw/2lyw_PCA-LEU_B_1-13/interaction.xml', 'GLN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2lyw/2lyw_PCA-LEU_B_1-13/2lyw_PCA-LEU_B_1-13_complex_UNL.pdb'), ('2m3m_PCA-VAL_B_141-151', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m3m/2m3m_PCA-VAL_B_141-151/2m3m_PCA-VAL_B_141-151_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m3m/2m3m_PCA-VAL_B_141-151/2m3m_PCA-VAL_B_141-151_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m3m/2m3m_PCA-VAL_B_141-151/2m3m_PCA-VAL_B_141-151_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m3m/2m3m_PCA-VAL_B_141-151/2m3m_PCA-VAL_B_141-151_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m3m/2m3m_PCA-VAL_B_141-151/interaction.xml', 'GLN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m3m/2m3m_PCA-VAL_B_141-151/2m3m_PCA-VAL_B_141-151_complex_UNL.pdb'), ('2m41_VAL-GLN_A_34-48', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m41/2m41_VAL-GLN_A_34-48/2m41_VAL-GLN_A_34-48_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m41/2m41_VAL-GLN_A_34-48/2m41_VAL-GLN_A_34-48_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m41/2m41_VAL-GLN_A_34-48/2m41_VAL-GLN_A_34-48_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m41/2m41_VAL-GLN_A_34-48/2m41_VAL-GLN_A_34-48_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m41/2m41_VAL-GLN_A_34-48/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2m41/2m41_VAL-GLN_A_34-48/2m41_VAL-GLN_A_34-48_complex_UNL.pdb'), ('2mc1_ASP-GLU_B_338-350', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mc1/2mc1_ASP-GLU_B_338-350/2mc1_ASP-GLU_B_338-350_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mc1/2mc1_ASP-GLU_B_338-350/2mc1_ASP-GLU_B_338-350_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mc1/2mc1_ASP-GLU_B_338-350/2mc1_ASP-GLU_B_338-350_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mc1/2mc1_ASP-GLU_B_338-350/2mc1_ASP-GLU_B_338-350_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mc1/2mc1_ASP-GLU_B_338-350/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mc1/2mc1_ASP-GLU_B_338-350/2mc1_ASP-GLU_B_338-350_complex_UNL.pdb'), ('2mnz_ALA-SER_B_363-372', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mnz/2mnz_ALA-SER_B_363-372/2mnz_ALA-SER_B_363-372_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mnz/2mnz_ALA-SER_B_363-372/2mnz_ALA-SER_B_363-372_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mnz/2mnz_ALA-SER_B_363-372/2mnz_ALA-SER_B_363-372_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mnz/2mnz_ALA-SER_B_363-372/2mnz_ALA-SER_B_363-372_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mnz/2mnz_ALA-SER_B_363-372/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mnz/2mnz_ALA-SER_B_363-372/2mnz_ALA-SER_B_363-372_complex_UNL.pdb'), ('2mpm_VAL-LEU_B_208-223', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mpm/2mpm_VAL-LEU_B_208-223/2mpm_VAL-LEU_B_208-223_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mpm/2mpm_VAL-LEU_B_208-223/2mpm_VAL-LEU_B_208-223_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mpm/2mpm_VAL-LEU_B_208-223/2mpm_VAL-LEU_B_208-223_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mpm/2mpm_VAL-LEU_B_208-223/2mpm_VAL-LEU_B_208-223_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mpm/2mpm_VAL-LEU_B_208-223/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mpm/2mpm_VAL-LEU_B_208-223/2mpm_VAL-LEU_B_208-223_complex_UNL.pdb'), ('2mps_ASP-ASP_B_301-316', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mps/2mps_ASP-ASP_B_301-316/2mps_ASP-ASP_B_301-316_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mps/2mps_ASP-ASP_B_301-316/2mps_ASP-ASP_B_301-316_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mps/2mps_ASP-ASP_B_301-316/2mps_ASP-ASP_B_301-316_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mps/2mps_ASP-ASP_B_301-316/2mps_ASP-ASP_B_301-316_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mps/2mps_ASP-ASP_B_301-316/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mps/2mps_ASP-ASP_B_301-316/2mps_ASP-ASP_B_301-316_complex_UNL.pdb'), ('2ms4_PRO-PRO_B_216-224', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ms4/2ms4_PRO-PRO_B_216-224/2ms4_PRO-PRO_B_216-224_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ms4/2ms4_PRO-PRO_B_216-224/2ms4_PRO-PRO_B_216-224_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ms4/2ms4_PRO-PRO_B_216-224/2ms4_PRO-PRO_B_216-224_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ms4/2ms4_PRO-PRO_B_216-224/2ms4_PRO-PRO_B_216-224_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ms4/2ms4_PRO-PRO_B_216-224/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ms4/2ms4_PRO-PRO_B_216-224/2ms4_PRO-PRO_B_216-224_complex_UNL.pdb'), ('2mwo_ARG-THR_B_363-377', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwo/2mwo_ARG-THR_B_363-377/2mwo_ARG-THR_B_363-377_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwo/2mwo_ARG-THR_B_363-377/2mwo_ARG-THR_B_363-377_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwo/2mwo_ARG-THR_B_363-377/2mwo_ARG-THR_B_363-377_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwo/2mwo_ARG-THR_B_363-377/2mwo_ARG-THR_B_363-377_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwo/2mwo_ARG-THR_B_363-377/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwo/2mwo_ARG-THR_B_363-377/2mwo_ARG-THR_B_363-377_complex_UNL.pdb'), ('2mwp_SER-THR_B_376-387', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwp/2mwp_SER-THR_B_376-387/2mwp_SER-THR_B_376-387_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwp/2mwp_SER-THR_B_376-387/2mwp_SER-THR_B_376-387_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwp/2mwp_SER-THR_B_376-387/2mwp_SER-THR_B_376-387_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwp/2mwp_SER-THR_B_376-387/2mwp_SER-THR_B_376-387_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwp/2mwp_SER-THR_B_376-387/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2mwp/2mwp_SER-THR_B_376-387/2mwp_SER-THR_B_376-387_complex_UNL.pdb'), ('2n1g_LYS-LYS_B_231-246', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n1g/2n1g_LYS-LYS_B_231-246/2n1g_LYS-LYS_B_231-246_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n1g/2n1g_LYS-LYS_B_231-246/2n1g_LYS-LYS_B_231-246_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n1g/2n1g_LYS-LYS_B_231-246/2n1g_LYS-LYS_B_231-246_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n1g/2n1g_LYS-LYS_B_231-246/2n1g_LYS-LYS_B_231-246_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n1g/2n1g_LYS-LYS_B_231-246/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n1g/2n1g_LYS-LYS_B_231-246/2n1g_LYS-LYS_B_231-246_complex_UNL.pdb'), ('2n8t_ALA-VAL_B_40-53', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n8t/2n8t_ALA-VAL_B_40-53/2n8t_ALA-VAL_B_40-53_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n8t/2n8t_ALA-VAL_B_40-53/2n8t_ALA-VAL_B_40-53_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n8t/2n8t_ALA-VAL_B_40-53/2n8t_ALA-VAL_B_40-53_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n8t/2n8t_ALA-VAL_B_40-53/2n8t_ALA-VAL_B_40-53_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n8t/2n8t_ALA-VAL_B_40-53/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n8t/2n8t_ALA-VAL_B_40-53/2n8t_ALA-VAL_B_40-53_complex_UNL.pdb'), ('2n9x_ASP-TYR_B_10-26', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n9x/2n9x_ASP-TYR_B_10-26/2n9x_ASP-TYR_B_10-26_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n9x/2n9x_ASP-TYR_B_10-26/2n9x_ASP-TYR_B_10-26_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n9x/2n9x_ASP-TYR_B_10-26/2n9x_ASP-TYR_B_10-26_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n9x/2n9x_ASP-TYR_B_10-26/2n9x_ASP-TYR_B_10-26_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n9x/2n9x_ASP-TYR_B_10-26/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2n9x/2n9x_ASP-TYR_B_10-26/2n9x_ASP-TYR_B_10-26_complex_UNL.pdb'), ('2ncz_GLU-ASN_B_201-212', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ncz/2ncz_GLU-ASN_B_201-212/2ncz_GLU-ASN_B_201-212_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ncz/2ncz_GLU-ASN_B_201-212/2ncz_GLU-ASN_B_201-212_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ncz/2ncz_GLU-ASN_B_201-212/2ncz_GLU-ASN_B_201-212_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ncz/2ncz_GLU-ASN_B_201-212/2ncz_GLU-ASN_B_201-212_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ncz/2ncz_GLU-ASN_B_201-212/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ncz/2ncz_GLU-ASN_B_201-212/2ncz_GLU-ASN_B_201-212_complex_UNL.pdb'), ('2nd0_ASN-GLY_B_199-217', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nd0/2nd0_ASN-GLY_B_199-217/2nd0_ASN-GLY_B_199-217_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nd0/2nd0_ASN-GLY_B_199-217/2nd0_ASN-GLY_B_199-217_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nd0/2nd0_ASN-GLY_B_199-217/2nd0_ASN-GLY_B_199-217_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nd0/2nd0_ASN-GLY_B_199-217/2nd0_ASN-GLY_B_199-217_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nd0/2nd0_ASN-GLY_B_199-217/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nd0/2nd0_ASN-GLY_B_199-217/2nd0_ASN-GLY_B_199-217_complex_UNL.pdb'), ('2ndf_GLY-ALA_B_212-224', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndf/2ndf_GLY-ALA_B_212-224/2ndf_GLY-ALA_B_212-224_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndf/2ndf_GLY-ALA_B_212-224/2ndf_GLY-ALA_B_212-224_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndf/2ndf_GLY-ALA_B_212-224/2ndf_GLY-ALA_B_212-224_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndf/2ndf_GLY-ALA_B_212-224/2ndf_GLY-ALA_B_212-224_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndf/2ndf_GLY-ALA_B_212-224/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndf/2ndf_GLY-ALA_B_212-224/2ndf_GLY-ALA_B_212-224_complex_UNL.pdb'), ('2ndg_GLY-ALA_B_212-224', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndg/2ndg_GLY-ALA_B_212-224/2ndg_GLY-ALA_B_212-224_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndg/2ndg_GLY-ALA_B_212-224/2ndg_GLY-ALA_B_212-224_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndg/2ndg_GLY-ALA_B_212-224/2ndg_GLY-ALA_B_212-224_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndg/2ndg_GLY-ALA_B_212-224/2ndg_GLY-ALA_B_212-224_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndg/2ndg_GLY-ALA_B_212-224/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ndg/2ndg_GLY-ALA_B_212-224/2ndg_GLY-ALA_B_212-224_complex_UNL.pdb'), ('2nm1_GLU-LYS_B_44-60', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nm1/2nm1_GLU-LYS_B_44-60/2nm1_GLU-LYS_B_44-60_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nm1/2nm1_GLU-LYS_B_44-60/2nm1_GLU-LYS_B_44-60_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nm1/2nm1_GLU-LYS_B_44-60/2nm1_GLU-LYS_B_44-60_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nm1/2nm1_GLU-LYS_B_44-60/2nm1_GLU-LYS_B_44-60_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nm1/2nm1_GLU-LYS_B_44-60/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2nm1/2nm1_GLU-LYS_B_44-60/2nm1_GLU-LYS_B_44-60_complex_UNL.pdb'), ('2oi3_ACE-NH2_B_87-100', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2oi3/2oi3_ACE-NH2_B_87-100/2oi3_ACE-NH2_B_87-100_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2oi3/2oi3_ACE-NH2_B_87-100/2oi3_ACE-NH2_B_87-100_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2oi3/2oi3_ACE-NH2_B_87-100/2oi3_ACE-NH2_B_87-100_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2oi3/2oi3_ACE-NH2_B_87-100/2oi3_ACE-NH2_B_87-100_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2oi3/2oi3_ACE-NH2_B_87-100/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2oi3/2oi3_ACE-NH2_B_87-100/2oi3_ACE-NH2_B_87-100_complex_UNL.pdb'), ('2or9_GLU-ASN_P_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2or9/2or9_GLU-ASN_P_1-11/2or9_GLU-ASN_P_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2or9/2or9_GLU-ASN_P_1-11/2or9_GLU-ASN_P_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2or9/2or9_GLU-ASN_P_1-11/2or9_GLU-ASN_P_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2or9/2or9_GLU-ASN_P_1-11/2or9_GLU-ASN_P_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2or9/2or9_GLU-ASN_P_1-11/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2or9/2or9_GLU-ASN_P_1-11/2or9_GLU-ASN_P_1-11_complex_UNL.pdb'), ('2ovq_LEU-GLY_C_374-385', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ovq/2ovq_LEU-GLY_C_374-385/2ovq_LEU-GLY_C_374-385_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ovq/2ovq_LEU-GLY_C_374-385/2ovq_LEU-GLY_C_374-385_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ovq/2ovq_LEU-GLY_C_374-385/2ovq_LEU-GLY_C_374-385_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ovq/2ovq_LEU-GLY_C_374-385/2ovq_LEU-GLY_C_374-385_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ovq/2ovq_LEU-GLY_C_374-385/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ovq/2ovq_LEU-GLY_C_374-385/2ovq_LEU-GLY_C_374-385_complex_UNL.pdb'), ('2pr9_ASP-ASP_P_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pr9/2pr9_ASP-ASP_P_1-10/2pr9_ASP-ASP_P_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pr9/2pr9_ASP-ASP_P_1-10/2pr9_ASP-ASP_P_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pr9/2pr9_ASP-ASP_P_1-10/2pr9_ASP-ASP_P_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pr9/2pr9_ASP-ASP_P_1-10/2pr9_ASP-ASP_P_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pr9/2pr9_ASP-ASP_P_1-10/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pr9/2pr9_ASP-ASP_P_1-10/2pr9_ASP-ASP_P_1-10_complex_UNL.pdb'), ('2puy_ALA-SER_E_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2puy/2puy_ALA-SER_E_1-10/2puy_ALA-SER_E_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2puy/2puy_ALA-SER_E_1-10/2puy_ALA-SER_E_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2puy/2puy_ALA-SER_E_1-10/2puy_ALA-SER_E_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2puy/2puy_ALA-SER_E_1-10/2puy_ALA-SER_E_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2puy/2puy_ALA-SER_E_1-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2puy/2puy_ALA-SER_E_1-10/2puy_ALA-SER_E_1-10_complex_UNL.pdb'), ('2pv2_ASN-LYS_F_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pv2/2pv2_ASN-LYS_F_1-12/2pv2_ASN-LYS_F_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pv2/2pv2_ASN-LYS_F_1-12/2pv2_ASN-LYS_F_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pv2/2pv2_ASN-LYS_F_1-12/2pv2_ASN-LYS_F_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pv2/2pv2_ASN-LYS_F_1-12/2pv2_ASN-LYS_F_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pv2/2pv2_ASN-LYS_F_1-12/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2pv2/2pv2_ASN-LYS_F_1-12/2pv2_ASN-LYS_F_1-12_complex_UNL.pdb'), ('2qbw_PRO-VAL_B_6-13', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qbw/2qbw_PRO-VAL_B_6-13/2qbw_PRO-VAL_B_6-13_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qbw/2qbw_PRO-VAL_B_6-13/2qbw_PRO-VAL_B_6-13_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qbw/2qbw_PRO-VAL_B_6-13/2qbw_PRO-VAL_B_6-13_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qbw/2qbw_PRO-VAL_B_6-13/2qbw_PRO-VAL_B_6-13_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qbw/2qbw_PRO-VAL_B_6-13/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qbw/2qbw_PRO-VAL_B_6-13/2qbw_PRO-VAL_B_6-13_complex_UNL.pdb'), ('2qhr_VAL-ASP_P_404-414', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qhr/2qhr_VAL-ASP_P_404-414/2qhr_VAL-ASP_P_404-414_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qhr/2qhr_VAL-ASP_P_404-414/2qhr_VAL-ASP_P_404-414_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qhr/2qhr_VAL-ASP_P_404-414/2qhr_VAL-ASP_P_404-414_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qhr/2qhr_VAL-ASP_P_404-414/2qhr_VAL-ASP_P_404-414_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qhr/2qhr_VAL-ASP_P_404-414/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2qhr/2qhr_VAL-ASP_P_404-414/2qhr_VAL-ASP_P_404-414_complex_UNL.pdb'), ('2rly_PRO-PRO_P_103-110', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rly/2rly_PRO-PRO_P_103-110/2rly_PRO-PRO_P_103-110_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rly/2rly_PRO-PRO_P_103-110/2rly_PRO-PRO_P_103-110_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rly/2rly_PRO-PRO_P_103-110/2rly_PRO-PRO_P_103-110_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rly/2rly_PRO-PRO_P_103-110/2rly_PRO-PRO_P_103-110_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rly/2rly_PRO-PRO_P_103-110/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rly/2rly_PRO-PRO_P_103-110/2rly_PRO-PRO_P_103-110_complex_UNL.pdb'), ('2rm0_PRO-PRO_P_101-109', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rm0/2rm0_PRO-PRO_P_101-109/2rm0_PRO-PRO_P_101-109_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rm0/2rm0_PRO-PRO_P_101-109/2rm0_PRO-PRO_P_101-109_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rm0/2rm0_PRO-PRO_P_101-109/2rm0_PRO-PRO_P_101-109_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rm0/2rm0_PRO-PRO_P_101-109/2rm0_PRO-PRO_P_101-109_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rm0/2rm0_PRO-PRO_P_101-109/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rm0/2rm0_PRO-PRO_P_101-109/2rm0_PRO-PRO_P_101-109_complex_UNL.pdb'), ('2rnw_ALA-ALA_B_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnw/2rnw_ALA-ALA_B_1-15/2rnw_ALA-ALA_B_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnw/2rnw_ALA-ALA_B_1-15/2rnw_ALA-ALA_B_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnw/2rnw_ALA-ALA_B_1-15/2rnw_ALA-ALA_B_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnw/2rnw_ALA-ALA_B_1-15/2rnw_ALA-ALA_B_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnw/2rnw_ALA-ALA_B_1-15/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnw/2rnw_ALA-ALA_B_1-15/2rnw_ALA-ALA_B_1-15_complex_UNL.pdb'), ('2rnx_SER-CYS_B_31-43', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnx/2rnx_SER-CYS_B_31-43/2rnx_SER-CYS_B_31-43_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnx/2rnx_SER-CYS_B_31-43/2rnx_SER-CYS_B_31-43_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnx/2rnx_SER-CYS_B_31-43/2rnx_SER-CYS_B_31-43_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnx/2rnx_SER-CYS_B_31-43/2rnx_SER-CYS_B_31-43_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnx/2rnx_SER-CYS_B_31-43/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rnx/2rnx_SER-CYS_B_31-43/2rnx_SER-CYS_B_31-43_complex_UNL.pdb'), ('2rny_GLY-GLN_B_13-27', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rny/2rny_GLY-GLN_B_13-27/2rny_GLY-GLN_B_13-27_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rny/2rny_GLY-GLN_B_13-27/2rny_GLY-GLN_B_13-27_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rny/2rny_GLY-GLN_B_13-27/2rny_GLY-GLN_B_13-27_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rny/2rny_GLY-GLN_B_13-27/2rny_GLY-GLN_B_13-27_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rny/2rny_GLY-GLN_B_13-27/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rny/2rny_GLY-GLN_B_13-27/2rny_GLY-GLN_B_13-27_complex_UNL.pdb'), ('2rqu_CYS-GLN_B_1578-1596', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rqu/2rqu_CYS-GLN_B_1578-1596/2rqu_CYS-GLN_B_1578-1596_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rqu/2rqu_CYS-GLN_B_1578-1596/2rqu_CYS-GLN_B_1578-1596_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rqu/2rqu_CYS-GLN_B_1578-1596/2rqu_CYS-GLN_B_1578-1596_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rqu/2rqu_CYS-GLN_B_1578-1596/2rqu_CYS-GLN_B_1578-1596_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rqu/2rqu_CYS-GLN_B_1578-1596/interaction.xml', 'CYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rqu/2rqu_CYS-GLN_B_1578-1596/2rqu_CYS-GLN_B_1578-1596_complex_UNL.pdb'), ('2rr4_ALA-SER_B_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rr4/2rr4_ALA-SER_B_1-10/2rr4_ALA-SER_B_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rr4/2rr4_ALA-SER_B_1-10/2rr4_ALA-SER_B_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rr4/2rr4_ALA-SER_B_1-10/2rr4_ALA-SER_B_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rr4/2rr4_ALA-SER_B_1-10/2rr4_ALA-SER_B_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rr4/2rr4_ALA-SER_B_1-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rr4/2rr4_ALA-SER_B_1-10/2rr4_ALA-SER_B_1-10_complex_UNL.pdb'), ('2rt5_TYR-GLU_B_2518-2525', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rt5/2rt5_TYR-GLU_B_2518-2525/2rt5_TYR-GLU_B_2518-2525_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rt5/2rt5_TYR-GLU_B_2518-2525/2rt5_TYR-GLU_B_2518-2525_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rt5/2rt5_TYR-GLU_B_2518-2525/2rt5_TYR-GLU_B_2518-2525_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rt5/2rt5_TYR-GLU_B_2518-2525/2rt5_TYR-GLU_B_2518-2525_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rt5/2rt5_TYR-GLU_B_2518-2525/interaction.xml', 'TYR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2rt5/2rt5_TYR-GLU_B_2518-2525/2rt5_TYR-GLU_B_2518-2525_complex_UNL.pdb'), ('2w85_SER-SER_B_101-112', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2w85/2w85_SER-SER_B_101-112/2w85_SER-SER_B_101-112_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2w85/2w85_SER-SER_B_101-112/2w85_SER-SER_B_101-112_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2w85/2w85_SER-SER_B_101-112/2w85_SER-SER_B_101-112_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2w85/2w85_SER-SER_B_101-112/2w85_SER-SER_B_101-112_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2w85/2w85_SER-SER_B_101-112/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2w85/2w85_SER-SER_B_101-112/2w85_SER-SER_B_101-112_complex_UNL.pdb'), ('2x4o_LYS-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4o/2x4o_LYS-LEU_C_1-9/2x4o_LYS-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4o/2x4o_LYS-LEU_C_1-9/2x4o_LYS-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4o/2x4o_LYS-LEU_C_1-9/2x4o_LYS-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4o/2x4o_LYS-LEU_C_1-9/2x4o_LYS-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4o/2x4o_LYS-LEU_C_1-9/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4o/2x4o_LYS-LEU_C_1-9/2x4o_LYS-LEU_C_1-9_complex_UNL.pdb'), ('2x4r_ASN-VAL_F_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4r/2x4r_ASN-VAL_F_1-9/2x4r_ASN-VAL_F_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4r/2x4r_ASN-VAL_F_1-9/2x4r_ASN-VAL_F_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4r/2x4r_ASN-VAL_F_1-9/2x4r_ASN-VAL_F_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4r/2x4r_ASN-VAL_F_1-9/2x4r_ASN-VAL_F_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4r/2x4r_ASN-VAL_F_1-9/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4r/2x4r_ASN-VAL_F_1-9/2x4r_ASN-VAL_F_1-9_complex_UNL.pdb'), ('2x4s_ALA-LEU_F_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4s/2x4s_ALA-LEU_F_1-9/2x4s_ALA-LEU_F_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4s/2x4s_ALA-LEU_F_1-9/2x4s_ALA-LEU_F_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4s/2x4s_ALA-LEU_F_1-9/2x4s_ALA-LEU_F_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4s/2x4s_ALA-LEU_F_1-9/2x4s_ALA-LEU_F_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4s/2x4s_ALA-LEU_F_1-9/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4s/2x4s_ALA-LEU_F_1-9/2x4s_ALA-LEU_F_1-9_complex_UNL.pdb'), ('2x4u_ILE-VAL_F_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4u/2x4u_ILE-VAL_F_1-9/2x4u_ILE-VAL_F_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4u/2x4u_ILE-VAL_F_1-9/2x4u_ILE-VAL_F_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4u/2x4u_ILE-VAL_F_1-9/2x4u_ILE-VAL_F_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4u/2x4u_ILE-VAL_F_1-9/2x4u_ILE-VAL_F_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4u/2x4u_ILE-VAL_F_1-9/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2x4u/2x4u_ILE-VAL_F_1-9/2x4u_ILE-VAL_F_1-9_complex_UNL.pdb'), ('2ynn_CYS-ASN_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ynn/2ynn_CYS-ASN_P_1-8/2ynn_CYS-ASN_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ynn/2ynn_CYS-ASN_P_1-8/2ynn_CYS-ASN_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ynn/2ynn_CYS-ASN_P_1-8/2ynn_CYS-ASN_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ynn/2ynn_CYS-ASN_P_1-8/2ynn_CYS-ASN_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ynn/2ynn_CYS-ASN_P_1-8/interaction.xml', 'CYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/2ynn/2ynn_CYS-ASN_P_1-8/2ynn_CYS-ASN_P_1-8_complex_UNL.pdb'), ('3av9_SER-ASP_X_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3av9/3av9_SER-ASP_X_1-8/3av9_SER-ASP_X_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3av9/3av9_SER-ASP_X_1-8/3av9_SER-ASP_X_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3av9/3av9_SER-ASP_X_1-8/3av9_SER-ASP_X_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3av9/3av9_SER-ASP_X_1-8/3av9_SER-ASP_X_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3av9/3av9_SER-ASP_X_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3av9/3av9_SER-ASP_X_1-8/3av9_SER-ASP_X_1-8_complex_UNL.pdb'), ('3ava_ALA-ASP_Y_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ava/3ava_ALA-ASP_Y_1-8/3ava_ALA-ASP_Y_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ava/3ava_ALA-ASP_Y_1-8/3ava_ALA-ASP_Y_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ava/3ava_ALA-ASP_Y_1-8/3ava_ALA-ASP_Y_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ava/3ava_ALA-ASP_Y_1-8/3ava_ALA-ASP_Y_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ava/3ava_ALA-ASP_Y_1-8/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ava/3ava_ALA-ASP_Y_1-8/3ava_ALA-ASP_Y_1-8_complex_UNL.pdb'), ('3avb_SER-ASP_Y_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avb/3avb_SER-ASP_Y_1-8/3avb_SER-ASP_Y_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avb/3avb_SER-ASP_Y_1-8/3avb_SER-ASP_Y_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avb/3avb_SER-ASP_Y_1-8/3avb_SER-ASP_Y_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avb/3avb_SER-ASP_Y_1-8/3avb_SER-ASP_Y_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avb/3avb_SER-ASP_Y_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avb/3avb_SER-ASP_Y_1-8/3avb_SER-ASP_Y_1-8_complex_UNL.pdb'), ('3avi_SER-ASP_F_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avi/3avi_SER-ASP_F_1-8/3avi_SER-ASP_F_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avi/3avi_SER-ASP_F_1-8/3avi_SER-ASP_F_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avi/3avi_SER-ASP_F_1-8/3avi_SER-ASP_F_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avi/3avi_SER-ASP_F_1-8/3avi_SER-ASP_F_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avi/3avi_SER-ASP_F_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avi/3avi_SER-ASP_F_1-8/3avi_SER-ASP_F_1-8_complex_UNL.pdb'), ('3avj_ALA-ASP_D_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avj/3avj_ALA-ASP_D_1-8/3avj_ALA-ASP_D_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avj/3avj_ALA-ASP_D_1-8/3avj_ALA-ASP_D_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avj/3avj_ALA-ASP_D_1-8/3avj_ALA-ASP_D_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avj/3avj_ALA-ASP_D_1-8/3avj_ALA-ASP_D_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avj/3avj_ALA-ASP_D_1-8/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avj/3avj_ALA-ASP_D_1-8/3avj_ALA-ASP_D_1-8_complex_UNL.pdb'), ('3avk_SER-ASP_D_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avk/3avk_SER-ASP_D_1-8/3avk_SER-ASP_D_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avk/3avk_SER-ASP_D_1-8/3avk_SER-ASP_D_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avk/3avk_SER-ASP_D_1-8/3avk_SER-ASP_D_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avk/3avk_SER-ASP_D_1-8/3avk_SER-ASP_D_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avk/3avk_SER-ASP_D_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avk/3avk_SER-ASP_D_1-8/3avk_SER-ASP_D_1-8_complex_UNL.pdb'), ('3avl_ALA-ASP_F_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avl/3avl_ALA-ASP_F_1-8/3avl_ALA-ASP_F_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avl/3avl_ALA-ASP_F_1-8/3avl_ALA-ASP_F_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avl/3avl_ALA-ASP_F_1-8/3avl_ALA-ASP_F_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avl/3avl_ALA-ASP_F_1-8/3avl_ALA-ASP_F_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avl/3avl_ALA-ASP_F_1-8/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avl/3avl_ALA-ASP_F_1-8/3avl_ALA-ASP_F_1-8_complex_UNL.pdb'), ('3avm_SER-ASP_F_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avm/3avm_SER-ASP_F_1-8/3avm_SER-ASP_F_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avm/3avm_SER-ASP_F_1-8/3avm_SER-ASP_F_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avm/3avm_SER-ASP_F_1-8/3avm_SER-ASP_F_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avm/3avm_SER-ASP_F_1-8/3avm_SER-ASP_F_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avm/3avm_SER-ASP_F_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avm/3avm_SER-ASP_F_1-8/3avm_SER-ASP_F_1-8_complex_UNL.pdb'), ('3avn_SER-ASP_H_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avn/3avn_SER-ASP_H_1-8/3avn_SER-ASP_H_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avn/3avn_SER-ASP_H_1-8/3avn_SER-ASP_H_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avn/3avn_SER-ASP_H_1-8/3avn_SER-ASP_H_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avn/3avn_SER-ASP_H_1-8/3avn_SER-ASP_H_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avn/3avn_SER-ASP_H_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3avn/3avn_SER-ASP_H_1-8/3avn_SER-ASP_H_1-8_complex_UNL.pdb'), ('3bgm_ARG-VAL_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bgm/3bgm_ARG-VAL_C_1-9/3bgm_ARG-VAL_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bgm/3bgm_ARG-VAL_C_1-9/3bgm_ARG-VAL_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bgm/3bgm_ARG-VAL_C_1-9/3bgm_ARG-VAL_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bgm/3bgm_ARG-VAL_C_1-9/3bgm_ARG-VAL_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bgm/3bgm_ARG-VAL_C_1-9/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bgm/3bgm_ARG-VAL_C_1-9/3bgm_ARG-VAL_C_1-9_complex_UNL.pdb'), ('3bh8_ARG-MET_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh8/3bh8_ARG-MET_C_1-10/3bh8_ARG-MET_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh8/3bh8_ARG-MET_C_1-10/3bh8_ARG-MET_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh8/3bh8_ARG-MET_C_1-10/3bh8_ARG-MET_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh8/3bh8_ARG-MET_C_1-10/3bh8_ARG-MET_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh8/3bh8_ARG-MET_C_1-10/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh8/3bh8_ARG-MET_C_1-10/3bh8_ARG-MET_C_1-10_complex_UNL.pdb'), ('3bh9_ARG-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh9/3bh9_ARG-VAL_C_1-10/3bh9_ARG-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh9/3bh9_ARG-VAL_C_1-10/3bh9_ARG-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh9/3bh9_ARG-VAL_C_1-10/3bh9_ARG-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh9/3bh9_ARG-VAL_C_1-10/3bh9_ARG-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh9/3bh9_ARG-VAL_C_1-10/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bh9/3bh9_ARG-VAL_C_1-10/3bh9_ARG-VAL_C_1-10_complex_UNL.pdb'), ('3bhb_LYS-LEU_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bhb/3bhb_LYS-LEU_C_1-10/3bhb_LYS-LEU_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bhb/3bhb_LYS-LEU_C_1-10/3bhb_LYS-LEU_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bhb/3bhb_LYS-LEU_C_1-10/3bhb_LYS-LEU_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bhb/3bhb_LYS-LEU_C_1-10/3bhb_LYS-LEU_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bhb/3bhb_LYS-LEU_C_1-10/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bhb/3bhb_LYS-LEU_C_1-10/3bhb_LYS-LEU_C_1-10_complex_UNL.pdb'), ('3bl2_THR-MET_C_106-124', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bl2/3bl2_THR-MET_C_106-124/3bl2_THR-MET_C_106-124_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bl2/3bl2_THR-MET_C_106-124/3bl2_THR-MET_C_106-124_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bl2/3bl2_THR-MET_C_106-124/3bl2_THR-MET_C_106-124_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bl2/3bl2_THR-MET_C_106-124/3bl2_THR-MET_C_106-124_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bl2/3bl2_THR-MET_C_106-124/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bl2/3bl2_THR-MET_C_106-124/3bl2_THR-MET_C_106-124_complex_UNL.pdb'), ('3bze_VAL-LEU_S_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bze/3bze_VAL-LEU_S_1-9/3bze_VAL-LEU_S_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bze/3bze_VAL-LEU_S_1-9/3bze_VAL-LEU_S_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bze/3bze_VAL-LEU_S_1-9/3bze_VAL-LEU_S_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bze/3bze_VAL-LEU_S_1-9/3bze_VAL-LEU_S_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bze/3bze_VAL-LEU_S_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bze/3bze_VAL-LEU_S_1-9/3bze_VAL-LEU_S_1-9_complex_UNL.pdb'), ('3bzf_VAL-LEU_Q_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bzf/3bzf_VAL-LEU_Q_1-9/3bzf_VAL-LEU_Q_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bzf/3bzf_VAL-LEU_Q_1-9/3bzf_VAL-LEU_Q_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bzf/3bzf_VAL-LEU_Q_1-9/3bzf_VAL-LEU_Q_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bzf/3bzf_VAL-LEU_Q_1-9/3bzf_VAL-LEU_Q_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bzf/3bzf_VAL-LEU_Q_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3bzf/3bzf_VAL-LEU_Q_1-9/3bzf_VAL-LEU_Q_1-9_complex_UNL.pdb'), ('3c3o_ASP-SER_B_210-222', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3o/3c3o_ASP-SER_B_210-222/3c3o_ASP-SER_B_210-222_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3o/3c3o_ASP-SER_B_210-222/3c3o_ASP-SER_B_210-222_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3o/3c3o_ASP-SER_B_210-222/3c3o_ASP-SER_B_210-222_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3o/3c3o_ASP-SER_B_210-222/3c3o_ASP-SER_B_210-222_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3o/3c3o_ASP-SER_B_210-222/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3o/3c3o_ASP-SER_B_210-222/3c3o_ASP-SER_B_210-222_complex_UNL.pdb'), ('3c3q_LYS-MET_B_207-224', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3q/3c3q_LYS-MET_B_207-224/3c3q_LYS-MET_B_207-224_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3q/3c3q_LYS-MET_B_207-224/3c3q_LYS-MET_B_207-224_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3q/3c3q_LYS-MET_B_207-224/3c3q_LYS-MET_B_207-224_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3q/3c3q_LYS-MET_B_207-224/3c3q_LYS-MET_B_207-224_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3q/3c3q_LYS-MET_B_207-224/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3q/3c3q_LYS-MET_B_207-224/3c3q_LYS-MET_B_207-224_complex_UNL.pdb'), ('3c3r_GLU-THR_B_221-233', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3r/3c3r_GLU-THR_B_221-233/3c3r_GLU-THR_B_221-233_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3r/3c3r_GLU-THR_B_221-233/3c3r_GLU-THR_B_221-233_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3r/3c3r_GLU-THR_B_221-233/3c3r_GLU-THR_B_221-233_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3r/3c3r_GLU-THR_B_221-233/3c3r_GLU-THR_B_221-233_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3r/3c3r_GLU-THR_B_221-233/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3c3r/3c3r_GLU-THR_B_221-233/3c3r_GLU-THR_B_221-233_complex_UNL.pdb'), ('3cii_VAL-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cii/3cii_VAL-LEU_C_1-9/3cii_VAL-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cii/3cii_VAL-LEU_C_1-9/3cii_VAL-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cii/3cii_VAL-LEU_C_1-9/3cii_VAL-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cii/3cii_VAL-LEU_C_1-9/3cii_VAL-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cii/3cii_VAL-LEU_C_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cii/3cii_VAL-LEU_C_1-9/3cii_VAL-LEU_C_1-9_complex_UNL.pdb'), ('3cyy_ARG-ILE_D_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cyy/3cyy_ARG-ILE_D_1-9/3cyy_ARG-ILE_D_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cyy/3cyy_ARG-ILE_D_1-9/3cyy_ARG-ILE_D_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cyy/3cyy_ARG-ILE_D_1-9/3cyy_ARG-ILE_D_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cyy/3cyy_ARG-ILE_D_1-9/3cyy_ARG-ILE_D_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cyy/3cyy_ARG-ILE_D_1-9/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3cyy/3cyy_ARG-ILE_D_1-9/3cyy_ARG-ILE_D_1-9_complex_UNL.pdb'), ('3d25_VAL-ALA_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3d25/3d25_VAL-ALA_C_1-9/3d25_VAL-ALA_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3d25/3d25_VAL-ALA_C_1-9/3d25_VAL-ALA_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3d25/3d25_VAL-ALA_C_1-9/3d25_VAL-ALA_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3d25/3d25_VAL-ALA_C_1-9/3d25_VAL-ALA_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3d25/3d25_VAL-ALA_C_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3d25/3d25_VAL-ALA_C_1-9/3d25_VAL-ALA_C_1-9_complex_UNL.pdb'), ('3dpc_HIS-ASP_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3dpc/3dpc_HIS-ASP_C_1-10/3dpc_HIS-ASP_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3dpc/3dpc_HIS-ASP_C_1-10/3dpc_HIS-ASP_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3dpc/3dpc_HIS-ASP_C_1-10/3dpc_HIS-ASP_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3dpc/3dpc_HIS-ASP_C_1-10/3dpc_HIS-ASP_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3dpc/3dpc_HIS-ASP_C_1-10/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3dpc/3dpc_HIS-ASP_C_1-10/3dpc_HIS-ASP_C_1-10_complex_UNL.pdb'), ('3ds0_ILE-PRO_T_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds0/3ds0_ILE-PRO_T_1-12/3ds0_ILE-PRO_T_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds0/3ds0_ILE-PRO_T_1-12/3ds0_ILE-PRO_T_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds0/3ds0_ILE-PRO_T_1-12/3ds0_ILE-PRO_T_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds0/3ds0_ILE-PRO_T_1-12/3ds0_ILE-PRO_T_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds0/3ds0_ILE-PRO_T_1-12/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds0/3ds0_ILE-PRO_T_1-12/3ds0_ILE-PRO_T_1-12_complex_UNL.pdb'), ('3ds3_ILE-PRO_D_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds3/3ds3_ILE-PRO_D_1-12/3ds3_ILE-PRO_D_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds3/3ds3_ILE-PRO_D_1-12/3ds3_ILE-PRO_D_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds3/3ds3_ILE-PRO_D_1-12/3ds3_ILE-PRO_D_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds3/3ds3_ILE-PRO_D_1-12/3ds3_ILE-PRO_D_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds3/3ds3_ILE-PRO_D_1-12/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds3/3ds3_ILE-PRO_D_1-12/3ds3_ILE-PRO_D_1-12_complex_UNL.pdb'), ('3ds4_ILE-PRO_T_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds4/3ds4_ILE-PRO_T_1-12/3ds4_ILE-PRO_T_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds4/3ds4_ILE-PRO_T_1-12/3ds4_ILE-PRO_T_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds4/3ds4_ILE-PRO_T_1-12/3ds4_ILE-PRO_T_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds4/3ds4_ILE-PRO_T_1-12/3ds4_ILE-PRO_T_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds4/3ds4_ILE-PRO_T_1-12/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ds4/3ds4_ILE-PRO_T_1-12/3ds4_ILE-PRO_T_1-12_complex_UNL.pdb'), ('3e1r_GLN-TYR_C_797-809', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3e1r/3e1r_GLN-TYR_C_797-809/3e1r_GLN-TYR_C_797-809_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3e1r/3e1r_GLN-TYR_C_797-809/3e1r_GLN-TYR_C_797-809_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3e1r/3e1r_GLN-TYR_C_797-809/3e1r_GLN-TYR_C_797-809_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3e1r/3e1r_GLN-TYR_C_797-809/3e1r_GLN-TYR_C_797-809_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3e1r/3e1r_GLN-TYR_C_797-809/interaction.xml', 'GLN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3e1r/3e1r_GLN-TYR_C_797-809/3e1r_GLN-TYR_C_797-809_complex_UNL.pdb'), ('3eqy_THR-PRO_C_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3eqy/3eqy_THR-PRO_C_1-12/3eqy_THR-PRO_C_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3eqy/3eqy_THR-PRO_C_1-12/3eqy_THR-PRO_C_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3eqy/3eqy_THR-PRO_C_1-12/3eqy_THR-PRO_C_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3eqy/3eqy_THR-PRO_C_1-12/3eqy_THR-PRO_C_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3eqy/3eqy_THR-PRO_C_1-12/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3eqy/3eqy_THR-PRO_C_1-12/3eqy_THR-PRO_C_1-12_complex_UNL.pdb'), ('3ft2_VAL-ALA_P_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft2/3ft2_VAL-ALA_P_1-9/3ft2_VAL-ALA_P_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft2/3ft2_VAL-ALA_P_1-9/3ft2_VAL-ALA_P_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft2/3ft2_VAL-ALA_P_1-9/3ft2_VAL-ALA_P_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft2/3ft2_VAL-ALA_P_1-9/3ft2_VAL-ALA_P_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft2/3ft2_VAL-ALA_P_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft2/3ft2_VAL-ALA_P_1-9/3ft2_VAL-ALA_P_1-9_complex_UNL.pdb'), ('3ft3_VAL-ALA_P_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft3/3ft3_VAL-ALA_P_1-9/3ft3_VAL-ALA_P_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft3/3ft3_VAL-ALA_P_1-9/3ft3_VAL-ALA_P_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft3/3ft3_VAL-ALA_P_1-9/3ft3_VAL-ALA_P_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft3/3ft3_VAL-ALA_P_1-9/3ft3_VAL-ALA_P_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft3/3ft3_VAL-ALA_P_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft3/3ft3_VAL-ALA_P_1-9/3ft3_VAL-ALA_P_1-9_complex_UNL.pdb'), ('3ft4_VAL-ALA_P_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft4/3ft4_VAL-ALA_P_1-9/3ft4_VAL-ALA_P_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft4/3ft4_VAL-ALA_P_1-9/3ft4_VAL-ALA_P_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft4/3ft4_VAL-ALA_P_1-9/3ft4_VAL-ALA_P_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft4/3ft4_VAL-ALA_P_1-9/3ft4_VAL-ALA_P_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft4/3ft4_VAL-ALA_P_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ft4/3ft4_VAL-ALA_P_1-9/3ft4_VAL-ALA_P_1-9_complex_UNL.pdb'), ('3h1z_TYR-HIS_P_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h1z/3h1z_TYR-HIS_P_1-15/3h1z_TYR-HIS_P_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h1z/3h1z_TYR-HIS_P_1-15/3h1z_TYR-HIS_P_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h1z/3h1z_TYR-HIS_P_1-15/3h1z_TYR-HIS_P_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h1z/3h1z_TYR-HIS_P_1-15/3h1z_TYR-HIS_P_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h1z/3h1z_TYR-HIS_P_1-15/interaction.xml', 'TYR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h1z/3h1z_TYR-HIS_P_1-15/3h1z_TYR-HIS_P_1-15_complex_UNL.pdb'), ('3h85_SER-HIS_P_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h85/3h85_SER-HIS_P_1-8/3h85_SER-HIS_P_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h85/3h85_SER-HIS_P_1-8/3h85_SER-HIS_P_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h85/3h85_SER-HIS_P_1-8/3h85_SER-HIS_P_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h85/3h85_SER-HIS_P_1-8/3h85_SER-HIS_P_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h85/3h85_SER-HIS_P_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3h85/3h85_SER-HIS_P_1-8/3h85_SER-HIS_P_1-8_complex_UNL.pdb'), ('3hg1_GLU-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3hg1/3hg1_GLU-VAL_C_1-10/3hg1_GLU-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3hg1/3hg1_GLU-VAL_C_1-10/3hg1_GLU-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3hg1/3hg1_GLU-VAL_C_1-10/3hg1_GLU-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3hg1/3hg1_GLU-VAL_C_1-10/3hg1_GLU-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3hg1/3hg1_GLU-VAL_C_1-10/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3hg1/3hg1_GLU-VAL_C_1-10/3hg1_GLU-VAL_C_1-10_complex_UNL.pdb'), ('3iwy_DAS-DAR_D_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3iwy/3iwy_DAS-DAR_D_1-12/3iwy_DAS-DAR_D_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3iwy/3iwy_DAS-DAR_D_1-12/3iwy_DAS-DAR_D_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3iwy/3iwy_DAS-DAR_D_1-12/3iwy_DAS-DAR_D_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3iwy/3iwy_DAS-DAR_D_1-12/3iwy_DAS-DAR_D_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3iwy/3iwy_DAS-DAR_D_1-12/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3iwy/3iwy_DAS-DAR_D_1-12/3iwy_DAS-DAR_D_1-12_complex_UNL.pdb'), ('3jzo_LEU-SER_P_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzo/3jzo_LEU-SER_P_1-12/3jzo_LEU-SER_P_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzo/3jzo_LEU-SER_P_1-12/3jzo_LEU-SER_P_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzo/3jzo_LEU-SER_P_1-12/3jzo_LEU-SER_P_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzo/3jzo_LEU-SER_P_1-12/3jzo_LEU-SER_P_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzo/3jzo_LEU-SER_P_1-12/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzo/3jzo_LEU-SER_P_1-12/3jzo_LEU-SER_P_1-12_complex_UNL.pdb'), ('3jzp_LEU-SER_P_17-28', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzp/3jzp_LEU-SER_P_17-28/3jzp_LEU-SER_P_17-28_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzp/3jzp_LEU-SER_P_17-28/3jzp_LEU-SER_P_17-28_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzp/3jzp_LEU-SER_P_17-28/3jzp_LEU-SER_P_17-28_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzp/3jzp_LEU-SER_P_17-28/3jzp_LEU-SER_P_17-28_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzp/3jzp_LEU-SER_P_17-28/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzp/3jzp_LEU-SER_P_17-28/3jzp_LEU-SER_P_17-28_complex_UNL.pdb'), ('3jzq_GLU-SER_P_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzq/3jzq_GLU-SER_P_1-12/3jzq_GLU-SER_P_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzq/3jzq_GLU-SER_P_1-12/3jzq_GLU-SER_P_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzq/3jzq_GLU-SER_P_1-12/3jzq_GLU-SER_P_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzq/3jzq_GLU-SER_P_1-12/3jzq_GLU-SER_P_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzq/3jzq_GLU-SER_P_1-12/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzq/3jzq_GLU-SER_P_1-12/3jzq_GLU-SER_P_1-12_complex_UNL.pdb'), ('3jzr_LEU-SER_P_17-28', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzr/3jzr_LEU-SER_P_17-28/3jzr_LEU-SER_P_17-28_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzr/3jzr_LEU-SER_P_17-28/3jzr_LEU-SER_P_17-28_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzr/3jzr_LEU-SER_P_17-28/3jzr_LEU-SER_P_17-28_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzr/3jzr_LEU-SER_P_17-28/3jzr_LEU-SER_P_17-28_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzr/3jzr_LEU-SER_P_17-28/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzr/3jzr_LEU-SER_P_17-28/3jzr_LEU-SER_P_17-28_complex_UNL.pdb'), ('3jzs_GLU-SER_P_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzs/3jzs_GLU-SER_P_1-12/3jzs_GLU-SER_P_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzs/3jzs_GLU-SER_P_1-12/3jzs_GLU-SER_P_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzs/3jzs_GLU-SER_P_1-12/3jzs_GLU-SER_P_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzs/3jzs_GLU-SER_P_1-12/3jzs_GLU-SER_P_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzs/3jzs_GLU-SER_P_1-12/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3jzs/3jzs_GLU-SER_P_1-12/3jzs_GLU-SER_P_1-12_complex_UNL.pdb'), ('3l6f_ALA-PRO_C_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3l6f/3l6f_ALA-PRO_C_1-15/3l6f_ALA-PRO_C_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3l6f/3l6f_ALA-PRO_C_1-15/3l6f_ALA-PRO_C_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3l6f/3l6f_ALA-PRO_C_1-15/3l6f_ALA-PRO_C_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3l6f/3l6f_ALA-PRO_C_1-15/3l6f_ALA-PRO_C_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3l6f/3l6f_ALA-PRO_C_1-15/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3l6f/3l6f_ALA-PRO_C_1-15/3l6f_ALA-PRO_C_1-15_complex_UNL.pdb'), ('3ll8_GLU-GLU_E_4-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ll8/3ll8_GLU-GLU_E_4-14/3ll8_GLU-GLU_E_4-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ll8/3ll8_GLU-GLU_E_4-14/3ll8_GLU-GLU_E_4-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ll8/3ll8_GLU-GLU_E_4-14/3ll8_GLU-GLU_E_4-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ll8/3ll8_GLU-GLU_E_4-14/3ll8_GLU-GLU_E_4-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ll8/3ll8_GLU-GLU_E_4-14/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3ll8/3ll8_GLU-GLU_E_4-14/3ll8_GLU-GLU_E_4-14_complex_UNL.pdb'), ('3lrh_GLU-GLN_F_5-18', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3lrh/3lrh_GLU-GLN_F_5-18/3lrh_GLU-GLN_F_5-18_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3lrh/3lrh_GLU-GLN_F_5-18/3lrh_GLU-GLN_F_5-18_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3lrh/3lrh_GLU-GLN_F_5-18/3lrh_GLU-GLN_F_5-18_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3lrh/3lrh_GLU-GLN_F_5-18/3lrh_GLU-GLN_F_5-18_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3lrh/3lrh_GLU-GLN_F_5-18/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3lrh/3lrh_GLU-GLN_F_5-18/3lrh_GLU-GLN_F_5-18_complex_UNL.pdb'), ('3mo8_PRO-TYR_B_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3mo8/3mo8_PRO-TYR_B_1-12/3mo8_PRO-TYR_B_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3mo8/3mo8_PRO-TYR_B_1-12/3mo8_PRO-TYR_B_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3mo8/3mo8_PRO-TYR_B_1-12/3mo8_PRO-TYR_B_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3mo8/3mo8_PRO-TYR_B_1-12/3mo8_PRO-TYR_B_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3mo8/3mo8_PRO-TYR_B_1-12/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3mo8/3mo8_PRO-TYR_B_1-12/3mo8_PRO-TYR_B_1-12_complex_UNL.pdb'), ('3o4l_GLY-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3o4l/3o4l_GLY-LEU_C_1-9/3o4l_GLY-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3o4l/3o4l_GLY-LEU_C_1-9/3o4l_GLY-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3o4l/3o4l_GLY-LEU_C_1-9/3o4l_GLY-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3o4l/3o4l_GLY-LEU_C_1-9/3o4l_GLY-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3o4l/3o4l_GLY-LEU_C_1-9/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3o4l/3o4l_GLY-LEU_C_1-9/3o4l_GLY-LEU_C_1-9_complex_UNL.pdb'), ('3oap_LYS-ASP_B_686-696', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oap/3oap_LYS-ASP_B_686-696/3oap_LYS-ASP_B_686-696_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oap/3oap_LYS-ASP_B_686-696/3oap_LYS-ASP_B_686-696_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oap/3oap_LYS-ASP_B_686-696/3oap_LYS-ASP_B_686-696_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oap/3oap_LYS-ASP_B_686-696/3oap_LYS-ASP_B_686-696_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oap/3oap_LYS-ASP_B_686-696/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oap/3oap_LYS-ASP_B_686-696/3oap_LYS-ASP_B_686-696_complex_UNL.pdb'), ('3obq_PRO-LEU_B_346-354', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obq/3obq_PRO-LEU_B_346-354/3obq_PRO-LEU_B_346-354_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obq/3obq_PRO-LEU_B_346-354/3obq_PRO-LEU_B_346-354_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obq/3obq_PRO-LEU_B_346-354/3obq_PRO-LEU_B_346-354_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obq/3obq_PRO-LEU_B_346-354/3obq_PRO-LEU_B_346-354_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obq/3obq_PRO-LEU_B_346-354/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obq/3obq_PRO-LEU_B_346-354/3obq_PRO-LEU_B_346-354_complex_UNL.pdb'), ('3obu_PRO-GLU_B_5-13', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obu/3obu_PRO-GLU_B_5-13/3obu_PRO-GLU_B_5-13_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obu/3obu_PRO-GLU_B_5-13/3obu_PRO-GLU_B_5-13_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obu/3obu_PRO-GLU_B_5-13/3obu_PRO-GLU_B_5-13_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obu/3obu_PRO-GLU_B_5-13/3obu_PRO-GLU_B_5-13_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obu/3obu_PRO-GLU_B_5-13/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obu/3obu_PRO-GLU_B_5-13/3obu_PRO-GLU_B_5-13_complex_UNL.pdb'), ('3obx_PRO-GLU_B_5-13', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obx/3obx_PRO-GLU_B_5-13/3obx_PRO-GLU_B_5-13_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obx/3obx_PRO-GLU_B_5-13/3obx_PRO-GLU_B_5-13_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obx/3obx_PRO-GLU_B_5-13/3obx_PRO-GLU_B_5-13_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obx/3obx_PRO-GLU_B_5-13/3obx_PRO-GLU_B_5-13_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obx/3obx_PRO-GLU_B_5-13/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3obx/3obx_PRO-GLU_B_5-13/3obx_PRO-GLU_B_5-13_complex_UNL.pdb'), ('3oe0_ARG-DPR_I_1-16', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oe0/3oe0_ARG-DPR_I_1-16/3oe0_ARG-DPR_I_1-16_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oe0/3oe0_ARG-DPR_I_1-16/3oe0_ARG-DPR_I_1-16_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oe0/3oe0_ARG-DPR_I_1-16/3oe0_ARG-DPR_I_1-16_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oe0/3oe0_ARG-DPR_I_1-16/3oe0_ARG-DPR_I_1-16_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oe0/3oe0_ARG-DPR_I_1-16/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3oe0/3oe0_ARG-DPR_I_1-16/3oe0_ARG-DPR_I_1-16_complex_UNL.pdb'), ('3p9l_SER-LEU_F_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9l/3p9l_SER-LEU_F_1-8/3p9l_SER-LEU_F_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9l/3p9l_SER-LEU_F_1-8/3p9l_SER-LEU_F_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9l/3p9l_SER-LEU_F_1-8/3p9l_SER-LEU_F_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9l/3p9l_SER-LEU_F_1-8/3p9l_SER-LEU_F_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9l/3p9l_SER-LEU_F_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9l/3p9l_SER-LEU_F_1-8/3p9l_SER-LEU_F_1-8_complex_UNL.pdb'), ('3p9m_SER-LEU_C_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9m/3p9m_SER-LEU_C_1-8/3p9m_SER-LEU_C_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9m/3p9m_SER-LEU_C_1-8/3p9m_SER-LEU_C_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9m/3p9m_SER-LEU_C_1-8/3p9m_SER-LEU_C_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9m/3p9m_SER-LEU_C_1-8/3p9m_SER-LEU_C_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9m/3p9m_SER-LEU_C_1-8/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3p9m/3p9m_SER-LEU_C_1-8/3p9m_SER-LEU_C_1-8_complex_UNL.pdb'), ('3pab_GLU-LEU_C_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3pab/3pab_GLU-LEU_C_1-8/3pab_GLU-LEU_C_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3pab/3pab_GLU-LEU_C_1-8/3pab_GLU-LEU_C_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3pab/3pab_GLU-LEU_C_1-8/3pab_GLU-LEU_C_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3pab/3pab_GLU-LEU_C_1-8/3pab_GLU-LEU_C_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3pab/3pab_GLU-LEU_C_1-8/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3pab/3pab_GLU-LEU_C_1-8/3pab_GLU-LEU_C_1-8_complex_UNL.pdb'), ('3qfd_ALA-VAL_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3qfd/3qfd_ALA-VAL_C_1-9/3qfd_ALA-VAL_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3qfd/3qfd_ALA-VAL_C_1-9/3qfd_ALA-VAL_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3qfd/3qfd_ALA-VAL_C_1-9/3qfd_ALA-VAL_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3qfd/3qfd_ALA-VAL_C_1-9/3qfd_ALA-VAL_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3qfd/3qfd_ALA-VAL_C_1-9/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3qfd/3qfd_ALA-VAL_C_1-9/3qfd_ALA-VAL_C_1-9_complex_UNL.pdb'), ('3r0h_ALA-VAL_c_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3r0h/3r0h_ALA-VAL_c_1-9/3r0h_ALA-VAL_c_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3r0h/3r0h_ALA-VAL_c_1-9/3r0h_ALA-VAL_c_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3r0h/3r0h_ALA-VAL_c_1-9/3r0h_ALA-VAL_c_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3r0h/3r0h_ALA-VAL_c_1-9/3r0h_ALA-VAL_c_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3r0h/3r0h_ALA-VAL_c_1-9/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3r0h/3r0h_ALA-VAL_c_1-9/3r0h_ALA-VAL_c_1-9_complex_UNL.pdb'), ('3rwc_ILE-TRP_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwc/3rwc_ILE-TRP_C_1-9/3rwc_ILE-TRP_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwc/3rwc_ILE-TRP_C_1-9/3rwc_ILE-TRP_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwc/3rwc_ILE-TRP_C_1-9/3rwc_ILE-TRP_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwc/3rwc_ILE-TRP_C_1-9/3rwc_ILE-TRP_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwc/3rwc_ILE-TRP_C_1-9/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwc/3rwc_ILE-TRP_C_1-9/3rwc_ILE-TRP_C_1-9_complex_UNL.pdb'), ('3rwd_ILE-TRP_F_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwd/3rwd_ILE-TRP_F_1-11/3rwd_ILE-TRP_F_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwd/3rwd_ILE-TRP_F_1-11/3rwd_ILE-TRP_F_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwd/3rwd_ILE-TRP_F_1-11/3rwd_ILE-TRP_F_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwd/3rwd_ILE-TRP_F_1-11/3rwd_ILE-TRP_F_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwd/3rwd_ILE-TRP_F_1-11/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwd/3rwd_ILE-TRP_F_1-11/3rwd_ILE-TRP_F_1-11_complex_UNL.pdb'), ('3rwe_PHE-TRP_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwe/3rwe_PHE-TRP_C_1-9/3rwe_PHE-TRP_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwe/3rwe_PHE-TRP_C_1-9/3rwe_PHE-TRP_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwe/3rwe_PHE-TRP_C_1-9/3rwe_PHE-TRP_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwe/3rwe_PHE-TRP_C_1-9/3rwe_PHE-TRP_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwe/3rwe_PHE-TRP_C_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwe/3rwe_PHE-TRP_C_1-9/3rwe_PHE-TRP_C_1-9_complex_UNL.pdb'), ('3rwf_GLN-TRP_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwf/3rwf_GLN-TRP_C_1-9/3rwf_GLN-TRP_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwf/3rwf_GLN-TRP_C_1-9/3rwf_GLN-TRP_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwf/3rwf_GLN-TRP_C_1-9/3rwf_GLN-TRP_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwf/3rwf_GLN-TRP_C_1-9/3rwf_GLN-TRP_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwf/3rwf_GLN-TRP_C_1-9/interaction.xml', 'GLN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwf/3rwf_GLN-TRP_C_1-9/3rwf_GLN-TRP_C_1-9_complex_UNL.pdb'), ('3rwg_MET-TRP_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwg/3rwg_MET-TRP_C_1-9/3rwg_MET-TRP_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwg/3rwg_MET-TRP_C_1-9/3rwg_MET-TRP_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwg/3rwg_MET-TRP_C_1-9/3rwg_MET-TRP_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwg/3rwg_MET-TRP_C_1-9/3rwg_MET-TRP_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwg/3rwg_MET-TRP_C_1-9/interaction.xml', 'MET', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwg/3rwg_MET-TRP_C_1-9/3rwg_MET-TRP_C_1-9_complex_UNL.pdb'), ('3rwh_MET-PHE_C_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwh/3rwh_MET-PHE_C_1-8/3rwh_MET-PHE_C_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwh/3rwh_MET-PHE_C_1-8/3rwh_MET-PHE_C_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwh/3rwh_MET-PHE_C_1-8/3rwh_MET-PHE_C_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwh/3rwh_MET-PHE_C_1-8/3rwh_MET-PHE_C_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwh/3rwh_MET-PHE_C_1-8/interaction.xml', 'MET', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwh/3rwh_MET-PHE_C_1-8/3rwh_MET-PHE_C_1-8_complex_UNL.pdb'), ('3rwi_GLY-TRP_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwi/3rwi_GLY-TRP_C_1-10/3rwi_GLY-TRP_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwi/3rwi_GLY-TRP_C_1-10/3rwi_GLY-TRP_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwi/3rwi_GLY-TRP_C_1-10/3rwi_GLY-TRP_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwi/3rwi_GLY-TRP_C_1-10/3rwi_GLY-TRP_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwi/3rwi_GLY-TRP_C_1-10/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwi/3rwi_GLY-TRP_C_1-10/3rwi_GLY-TRP_C_1-10_complex_UNL.pdb'), ('3rwj_HIS-TRP_C_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwj/3rwj_HIS-TRP_C_1-8/3rwj_HIS-TRP_C_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwj/3rwj_HIS-TRP_C_1-8/3rwj_HIS-TRP_C_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwj/3rwj_HIS-TRP_C_1-8/3rwj_HIS-TRP_C_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwj/3rwj_HIS-TRP_C_1-8/3rwj_HIS-TRP_C_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwj/3rwj_HIS-TRP_C_1-8/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3rwj/3rwj_HIS-TRP_C_1-8/3rwj_HIS-TRP_C_1-8_complex_UNL.pdb'), ('3tfk_GLN-LEU_B_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tfk/3tfk_GLN-LEU_B_1-9/3tfk_GLN-LEU_B_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tfk/3tfk_GLN-LEU_B_1-9/3tfk_GLN-LEU_B_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tfk/3tfk_GLN-LEU_B_1-9/3tfk_GLN-LEU_B_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tfk/3tfk_GLN-LEU_B_1-9/3tfk_GLN-LEU_B_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tfk/3tfk_GLN-LEU_B_1-9/interaction.xml', 'GLN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tfk/3tfk_GLN-LEU_B_1-9/3tfk_GLN-LEU_B_1-9_complex_UNL.pdb'), ('3tjh_SER-ILE_B_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tjh/3tjh_SER-ILE_B_1-9/3tjh_SER-ILE_B_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tjh/3tjh_SER-ILE_B_1-9/3tjh_SER-ILE_B_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tjh/3tjh_SER-ILE_B_1-9/3tjh_SER-ILE_B_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tjh/3tjh_SER-ILE_B_1-9/3tjh_SER-ILE_B_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tjh/3tjh_SER-ILE_B_1-9/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tjh/3tjh_SER-ILE_B_1-9/3tjh_SER-ILE_B_1-9_complex_UNL.pdb'), ('3tpu_PHE-ILE_L_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tpu/3tpu_PHE-ILE_L_1-9/3tpu_PHE-ILE_L_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tpu/3tpu_PHE-ILE_L_1-9/3tpu_PHE-ILE_L_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tpu/3tpu_PHE-ILE_L_1-9/3tpu_PHE-ILE_L_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tpu/3tpu_PHE-ILE_L_1-9/3tpu_PHE-ILE_L_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tpu/3tpu_PHE-ILE_L_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3tpu/3tpu_PHE-ILE_L_1-9/3tpu_PHE-ILE_L_1-9_complex_UNL.pdb'), ('3u0t_ALA-VAL_E_30-40', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3u0t/3u0t_ALA-VAL_E_30-40/3u0t_ALA-VAL_E_30-40_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3u0t/3u0t_ALA-VAL_E_30-40/3u0t_ALA-VAL_E_30-40_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3u0t/3u0t_ALA-VAL_E_30-40/3u0t_ALA-VAL_E_30-40_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3u0t/3u0t_ALA-VAL_E_30-40/3u0t_ALA-VAL_E_30-40_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3u0t/3u0t_ALA-VAL_E_30-40/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3u0t/3u0t_ALA-VAL_E_30-40/3u0t_ALA-VAL_E_30-40_complex_UNL.pdb'), ('3unn_MET-ASP_B_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3unn/3unn_MET-ASP_B_1-8/3unn_MET-ASP_B_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3unn/3unn_MET-ASP_B_1-8/3unn_MET-ASP_B_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3unn/3unn_MET-ASP_B_1-8/3unn_MET-ASP_B_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3unn/3unn_MET-ASP_B_1-8/3unn_MET-ASP_B_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3unn/3unn_MET-ASP_B_1-8/interaction.xml', 'MET', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3unn/3unn_MET-ASP_B_1-8/3unn_MET-ASP_B_1-8_complex_UNL.pdb'), ('3v2x_LEU-LEU_B_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v2x/3v2x_LEU-LEU_B_1-11/3v2x_LEU-LEU_B_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v2x/3v2x_LEU-LEU_B_1-11/3v2x_LEU-LEU_B_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v2x/3v2x_LEU-LEU_B_1-11/3v2x_LEU-LEU_B_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v2x/3v2x_LEU-LEU_B_1-11/3v2x_LEU-LEU_B_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v2x/3v2x_LEU-LEU_B_1-11/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v2x/3v2x_LEU-LEU_B_1-11/3v2x_LEU-LEU_B_1-11_complex_UNL.pdb'), ('3v31_ACE-PRO_B_0-17', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v31/3v31_ACE-PRO_B_0-17/3v31_ACE-PRO_B_0-17_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v31/3v31_ACE-PRO_B_0-17/3v31_ACE-PRO_B_0-17_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v31/3v31_ACE-PRO_B_0-17/3v31_ACE-PRO_B_0-17_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v31/3v31_ACE-PRO_B_0-17/3v31_ACE-PRO_B_0-17_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v31/3v31_ACE-PRO_B_0-17/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/3v31/3v31_ACE-PRO_B_0-17/3v31_ACE-PRO_B_0-17_complex_UNL.pdb'), ('4bd3_ALA-TYR_B_31-41', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bd3/4bd3_ALA-TYR_B_31-41/4bd3_ALA-TYR_B_31-41_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bd3/4bd3_ALA-TYR_B_31-41/4bd3_ALA-TYR_B_31-41_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bd3/4bd3_ALA-TYR_B_31-41/4bd3_ALA-TYR_B_31-41_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bd3/4bd3_ALA-TYR_B_31-41/4bd3_ALA-TYR_B_31-41_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bd3/4bd3_ALA-TYR_B_31-41/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bd3/4bd3_ALA-TYR_B_31-41/4bd3_ALA-TYR_B_31-41_complex_UNL.pdb'), ('4bt9_PRO-GLY_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bt9/4bt9_PRO-GLY_C_1-9/4bt9_PRO-GLY_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bt9/4bt9_PRO-GLY_C_1-9/4bt9_PRO-GLY_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bt9/4bt9_PRO-GLY_C_1-9/4bt9_PRO-GLY_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bt9/4bt9_PRO-GLY_C_1-9/4bt9_PRO-GLY_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bt9/4bt9_PRO-GLY_C_1-9/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4bt9/4bt9_PRO-GLY_C_1-9/4bt9_PRO-GLY_C_1-9_complex_UNL.pdb'), ('4btb_PRO-PRO_C_0-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4btb/4btb_PRO-PRO_C_0-8/4btb_PRO-PRO_C_0-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4btb/4btb_PRO-PRO_C_0-8/4btb_PRO-PRO_C_0-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4btb/4btb_PRO-PRO_C_0-8/4btb_PRO-PRO_C_0-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4btb/4btb_PRO-PRO_C_0-8/4btb_PRO-PRO_C_0-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4btb/4btb_PRO-PRO_C_0-8/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4btb/4btb_PRO-PRO_C_0-8/4btb_PRO-PRO_C_0-8_complex_UNL.pdb'), ('4cc3_PRO-ALA_H_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4cc3/4cc3_PRO-ALA_H_1-12/4cc3_PRO-ALA_H_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4cc3/4cc3_PRO-ALA_H_1-12/4cc3_PRO-ALA_H_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4cc3/4cc3_PRO-ALA_H_1-12/4cc3_PRO-ALA_H_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4cc3/4cc3_PRO-ALA_H_1-12/4cc3_PRO-ALA_H_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4cc3/4cc3_PRO-ALA_H_1-12/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4cc3/4cc3_PRO-ALA_H_1-12/4cc3_PRO-ALA_H_1-12_complex_UNL.pdb'), ('4e34_ALA-ILE_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4e34/4e34_ALA-ILE_C_1-10/4e34_ALA-ILE_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4e34/4e34_ALA-ILE_C_1-10/4e34_ALA-ILE_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4e34/4e34_ALA-ILE_C_1-10/4e34_ALA-ILE_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4e34/4e34_ALA-ILE_C_1-10/4e34_ALA-ILE_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4e34/4e34_ALA-ILE_C_1-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4e34/4e34_ALA-ILE_C_1-10/4e34_ALA-ILE_C_1-10_complex_UNL.pdb'), ('4eoy_ASN-TYR_D_103-110', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4eoy/4eoy_ASN-TYR_D_103-110/4eoy_ASN-TYR_D_103-110_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4eoy/4eoy_ASN-TYR_D_103-110/4eoy_ASN-TYR_D_103-110_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4eoy/4eoy_ASN-TYR_D_103-110/4eoy_ASN-TYR_D_103-110_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4eoy/4eoy_ASN-TYR_D_103-110/4eoy_ASN-TYR_D_103-110_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4eoy/4eoy_ASN-TYR_D_103-110/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4eoy/4eoy_ASN-TYR_D_103-110/4eoy_ASN-TYR_D_103-110_complex_UNL.pdb'), ('4ezr_SER-VAL_B_12-19', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ezr/4ezr_SER-VAL_B_12-19/4ezr_SER-VAL_B_12-19_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ezr/4ezr_SER-VAL_B_12-19/4ezr_SER-VAL_B_12-19_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ezr/4ezr_SER-VAL_B_12-19/4ezr_SER-VAL_B_12-19_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ezr/4ezr_SER-VAL_B_12-19/4ezr_SER-VAL_B_12-19_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ezr/4ezr_SER-VAL_B_12-19/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ezr/4ezr_SER-VAL_B_12-19/4ezr_SER-VAL_B_12-19_complex_UNL.pdb'), ('4fi9_PHE-THR_B_6872-6885', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4fi9/4fi9_PHE-THR_B_6872-6885/4fi9_PHE-THR_B_6872-6885_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4fi9/4fi9_PHE-THR_B_6872-6885/4fi9_PHE-THR_B_6872-6885_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4fi9/4fi9_PHE-THR_B_6872-6885/4fi9_PHE-THR_B_6872-6885_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4fi9/4fi9_PHE-THR_B_6872-6885/4fi9_PHE-THR_B_6872-6885_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4fi9/4fi9_PHE-THR_B_6872-6885/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4fi9/4fi9_PHE-THR_B_6872-6885/4fi9_PHE-THR_B_6872-6885_complex_UNL.pdb'), ('4gpk_SER-GLY_O_25-32', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4gpk/4gpk_SER-GLY_O_25-32/4gpk_SER-GLY_O_25-32_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4gpk/4gpk_SER-GLY_O_25-32/4gpk_SER-GLY_O_25-32_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4gpk/4gpk_SER-GLY_O_25-32/4gpk_SER-GLY_O_25-32_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4gpk/4gpk_SER-GLY_O_25-32/4gpk_SER-GLY_O_25-32_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4gpk/4gpk_SER-GLY_O_25-32/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4gpk/4gpk_SER-GLY_O_25-32/4gpk_SER-GLY_O_25-32_complex_UNL.pdb'), ('4h75_ALA-ARG_B_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4h75/4h75_ALA-ARG_B_1-8/4h75_ALA-ARG_B_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4h75/4h75_ALA-ARG_B_1-8/4h75_ALA-ARG_B_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4h75/4h75_ALA-ARG_B_1-8/4h75_ALA-ARG_B_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4h75/4h75_ALA-ARG_B_1-8/4h75_ALA-ARG_B_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4h75/4h75_ALA-ARG_B_1-8/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4h75/4h75_ALA-ARG_B_1-8/4h75_ALA-ARG_B_1-8_complex_UNL.pdb'), ('4ht6_ILE-GLN_B_75-85', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ht6/4ht6_ILE-GLN_B_75-85/4ht6_ILE-GLN_B_75-85_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ht6/4ht6_ILE-GLN_B_75-85/4ht6_ILE-GLN_B_75-85_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ht6/4ht6_ILE-GLN_B_75-85/4ht6_ILE-GLN_B_75-85_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ht6/4ht6_ILE-GLN_B_75-85/4ht6_ILE-GLN_B_75-85_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ht6/4ht6_ILE-GLN_B_75-85/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4ht6/4ht6_ILE-GLN_B_75-85/4ht6_ILE-GLN_B_75-85_complex_UNL.pdb'), ('4i2w_ALA-ASP_B_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4i2w/4i2w_ALA-ASP_B_1-10/4i2w_ALA-ASP_B_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4i2w/4i2w_ALA-ASP_B_1-10/4i2w_ALA-ASP_B_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4i2w/4i2w_ALA-ASP_B_1-10/4i2w_ALA-ASP_B_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4i2w/4i2w_ALA-ASP_B_1-10/4i2w_ALA-ASP_B_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4i2w/4i2w_ALA-ASP_B_1-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4i2w/4i2w_ALA-ASP_B_1-10/4i2w_ALA-ASP_B_1-10_complex_UNL.pdb'), ('4iho_GLU-LEU_F_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4iho/4iho_GLU-LEU_F_1-9/4iho_GLU-LEU_F_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4iho/4iho_GLU-LEU_F_1-9/4iho_GLU-LEU_F_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4iho/4iho_GLU-LEU_F_1-9/4iho_GLU-LEU_F_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4iho/4iho_GLU-LEU_F_1-9/4iho_GLU-LEU_F_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4iho/4iho_GLU-LEU_F_1-9/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4iho/4iho_GLU-LEU_F_1-9/4iho_GLU-LEU_F_1-9_complex_UNL.pdb'), ('4jfd_GLU-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfd/4jfd_GLU-VAL_C_1-10/4jfd_GLU-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfd/4jfd_GLU-VAL_C_1-10/4jfd_GLU-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfd/4jfd_GLU-VAL_C_1-10/4jfd_GLU-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfd/4jfd_GLU-VAL_C_1-10/4jfd_GLU-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfd/4jfd_GLU-VAL_C_1-10/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfd/4jfd_GLU-VAL_C_1-10/4jfd_GLU-VAL_C_1-10_complex_UNL.pdb'), ('4jfe_GLU-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfe/4jfe_GLU-VAL_C_1-10/4jfe_GLU-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfe/4jfe_GLU-VAL_C_1-10/4jfe_GLU-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfe/4jfe_GLU-VAL_C_1-10/4jfe_GLU-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfe/4jfe_GLU-VAL_C_1-10/4jfe_GLU-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfe/4jfe_GLU-VAL_C_1-10/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jfe/4jfe_GLU-VAL_C_1-10/4jfe_GLU-VAL_C_1-10_complex_UNL.pdb'), ('4jff_GLU-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jff/4jff_GLU-VAL_C_1-10/4jff_GLU-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jff/4jff_GLU-VAL_C_1-10/4jff_GLU-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jff/4jff_GLU-VAL_C_1-10/4jff_GLU-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jff/4jff_GLU-VAL_C_1-10/4jff_GLU-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jff/4jff_GLU-VAL_C_1-10/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jff/4jff_GLU-VAL_C_1-10/4jff_GLU-VAL_C_1-10_complex_UNL.pdb'), ('4jk5_ALA-ZBR_B_1-101', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk5/4jk5_ALA-ZBR_B_1-101/4jk5_ALA-ZBR_B_1-101_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk5/4jk5_ALA-ZBR_B_1-101/4jk5_ALA-ZBR_B_1-101_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk5/4jk5_ALA-ZBR_B_1-101/4jk5_ALA-ZBR_B_1-101_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk5/4jk5_ALA-ZBR_B_1-101/4jk5_ALA-ZBR_B_1-101_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk5/4jk5_ALA-ZBR_B_1-101/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk5/4jk5_ALA-ZBR_B_1-101/4jk5_ALA-ZBR_B_1-101_complex_UNL.pdb'), ('4jk6_ALA-ZBR_B_1-101', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk6/4jk6_ALA-ZBR_B_1-101/4jk6_ALA-ZBR_B_1-101_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk6/4jk6_ALA-ZBR_B_1-101/4jk6_ALA-ZBR_B_1-101_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk6/4jk6_ALA-ZBR_B_1-101/4jk6_ALA-ZBR_B_1-101_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk6/4jk6_ALA-ZBR_B_1-101/4jk6_ALA-ZBR_B_1-101_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk6/4jk6_ALA-ZBR_B_1-101/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4jk6/4jk6_ALA-ZBR_B_1-101/4jk6_ALA-ZBR_B_1-101_complex_UNL.pdb'), ('4joe_ALA-ILE_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4joe/4joe_ALA-ILE_C_1-10/4joe_ALA-ILE_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4joe/4joe_ALA-ILE_C_1-10/4joe_ALA-ILE_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4joe/4joe_ALA-ILE_C_1-10/4joe_ALA-ILE_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4joe/4joe_ALA-ILE_C_1-10/4joe_ALA-ILE_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4joe/4joe_ALA-ILE_C_1-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4joe/4joe_ALA-ILE_C_1-10/4joe_ALA-ILE_C_1-10_complex_UNL.pdb'), ('4k1e_GLY-ASP_B_1-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k1e/4k1e_GLY-ASP_B_1-14/4k1e_GLY-ASP_B_1-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k1e/4k1e_GLY-ASP_B_1-14/4k1e_GLY-ASP_B_1-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k1e/4k1e_GLY-ASP_B_1-14/4k1e_GLY-ASP_B_1-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k1e/4k1e_GLY-ASP_B_1-14/4k1e_GLY-ASP_B_1-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k1e/4k1e_GLY-ASP_B_1-14/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k1e/4k1e_GLY-ASP_B_1-14/4k1e_GLY-ASP_B_1-14_complex_UNL.pdb'), ('4k78_ALA-LEU_B_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k78/4k78_ALA-LEU_B_1-10/4k78_ALA-LEU_B_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k78/4k78_ALA-LEU_B_1-10/4k78_ALA-LEU_B_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k78/4k78_ALA-LEU_B_1-10/4k78_ALA-LEU_B_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k78/4k78_ALA-LEU_B_1-10/4k78_ALA-LEU_B_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k78/4k78_ALA-LEU_B_1-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4k78/4k78_ALA-LEU_B_1-10/4k78_ALA-LEU_B_1-10_complex_UNL.pdb'), ('4m7c_SER-TRP_D_497-509', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4m7c/4m7c_SER-TRP_D_497-509/4m7c_SER-TRP_D_497-509_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4m7c/4m7c_SER-TRP_D_497-509/4m7c_SER-TRP_D_497-509_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4m7c/4m7c_SER-TRP_D_497-509/4m7c_SER-TRP_D_497-509_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4m7c/4m7c_SER-TRP_D_497-509/4m7c_SER-TRP_D_497-509_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4m7c/4m7c_SER-TRP_D_497-509/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4m7c/4m7c_SER-TRP_D_497-509/4m7c_SER-TRP_D_497-509_complex_UNL.pdb'), ('4mji_THR-ILE_H_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mji/4mji_THR-ILE_H_1-8/4mji_THR-ILE_H_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mji/4mji_THR-ILE_H_1-8/4mji_THR-ILE_H_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mji/4mji_THR-ILE_H_1-8/4mji_THR-ILE_H_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mji/4mji_THR-ILE_H_1-8/4mji_THR-ILE_H_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mji/4mji_THR-ILE_H_1-8/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mji/4mji_THR-ILE_H_1-8/4mji_THR-ILE_H_1-8_complex_UNL.pdb'), ('4mnq_ILE-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnq/4mnq_ILE-LEU_C_1-9/4mnq_ILE-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnq/4mnq_ILE-LEU_C_1-9/4mnq_ILE-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnq/4mnq_ILE-LEU_C_1-9/4mnq_ILE-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnq/4mnq_ILE-LEU_C_1-9/4mnq_ILE-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnq/4mnq_ILE-LEU_C_1-9/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnq/4mnq_ILE-LEU_C_1-9/4mnq_ILE-LEU_C_1-9_complex_UNL.pdb'), ('4mnx_LEU-29N_B_1-101', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnx/4mnx_LEU-29N_B_1-101/4mnx_LEU-29N_B_1-101_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnx/4mnx_LEU-29N_B_1-101/4mnx_LEU-29N_B_1-101_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnx/4mnx_LEU-29N_B_1-101/4mnx_LEU-29N_B_1-101_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnx/4mnx_LEU-29N_B_1-101/4mnx_LEU-29N_B_1-101_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnx/4mnx_LEU-29N_B_1-101/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4mnx/4mnx_LEU-29N_B_1-101/4mnx_LEU-29N_B_1-101_complex_UNL.pdb'), ('4nxr_ASN-VAL_B_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4nxr/4nxr_ASN-VAL_B_1-8/4nxr_ASN-VAL_B_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4nxr/4nxr_ASN-VAL_B_1-8/4nxr_ASN-VAL_B_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4nxr/4nxr_ASN-VAL_B_1-8/4nxr_ASN-VAL_B_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4nxr/4nxr_ASN-VAL_B_1-8/4nxr_ASN-VAL_B_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4nxr/4nxr_ASN-VAL_B_1-8/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4nxr/4nxr_ASN-VAL_B_1-8/4nxr_ASN-VAL_B_1-8_complex_UNL.pdb'), ('4o2c_ACE-LEU_C_0-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2c/4o2c_ACE-LEU_C_0-9/4o2c_ACE-LEU_C_0-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2c/4o2c_ACE-LEU_C_0-9/4o2c_ACE-LEU_C_0-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2c/4o2c_ACE-LEU_C_0-9/4o2c_ACE-LEU_C_0-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2c/4o2c_ACE-LEU_C_0-9/4o2c_ACE-LEU_C_0-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2c/4o2c_ACE-LEU_C_0-9/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2c/4o2c_ACE-LEU_C_0-9/4o2c_ACE-LEU_C_0-9_complex_UNL.pdb'), ('4o2e_SER-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2e/4o2e_SER-LEU_C_1-9/4o2e_SER-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2e/4o2e_SER-LEU_C_1-9/4o2e_SER-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2e/4o2e_SER-LEU_C_1-9/4o2e_SER-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2e/4o2e_SER-LEU_C_1-9/4o2e_SER-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2e/4o2e_SER-LEU_C_1-9/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2e/4o2e_SER-LEU_C_1-9/4o2e_SER-LEU_C_1-9_complex_UNL.pdb'), ('4o2f_HIS-LEU_C_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2f/4o2f_HIS-LEU_C_1-8/4o2f_HIS-LEU_C_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2f/4o2f_HIS-LEU_C_1-8/4o2f_HIS-LEU_C_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2f/4o2f_HIS-LEU_C_1-8/4o2f_HIS-LEU_C_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2f/4o2f_HIS-LEU_C_1-8/4o2f_HIS-LEU_C_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2f/4o2f_HIS-LEU_C_1-8/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o2f/4o2f_HIS-LEU_C_1-8/4o2f_HIS-LEU_C_1-8_complex_UNL.pdb'), ('4o37_LYS-SER_A_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o37/4o37_LYS-SER_A_1-15/4o37_LYS-SER_A_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o37/4o37_LYS-SER_A_1-15/4o37_LYS-SER_A_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o37/4o37_LYS-SER_A_1-15/4o37_LYS-SER_A_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o37/4o37_LYS-SER_A_1-15/4o37_LYS-SER_A_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o37/4o37_LYS-SER_A_1-15/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4o37/4o37_LYS-SER_A_1-15/4o37_LYS-SER_A_1-15_complex_UNL.pdb'), ('4pg9_PHE-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pg9/4pg9_PHE-LEU_C_1-9/4pg9_PHE-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pg9/4pg9_PHE-LEU_C_1-9/4pg9_PHE-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pg9/4pg9_PHE-LEU_C_1-9/4pg9_PHE-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pg9/4pg9_PHE-LEU_C_1-9/4pg9_PHE-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pg9/4pg9_PHE-LEU_C_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pg9/4pg9_PHE-LEU_C_1-9/4pg9_PHE-LEU_C_1-9_complex_UNL.pdb'), ('4pgb_PHE-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgb/4pgb_PHE-LEU_C_1-9/4pgb_PHE-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgb/4pgb_PHE-LEU_C_1-9/4pgb_PHE-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgb/4pgb_PHE-LEU_C_1-9/4pgb_PHE-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgb/4pgb_PHE-LEU_C_1-9/4pgb_PHE-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgb/4pgb_PHE-LEU_C_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgb/4pgb_PHE-LEU_C_1-9/4pgb_PHE-LEU_C_1-9_complex_UNL.pdb'), ('4pgd_PHE-PHE_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgd/4pgd_PHE-PHE_C_1-9/4pgd_PHE-PHE_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgd/4pgd_PHE-PHE_C_1-9/4pgd_PHE-PHE_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgd/4pgd_PHE-PHE_C_1-9/4pgd_PHE-PHE_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgd/4pgd_PHE-PHE_C_1-9/4pgd_PHE-PHE_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgd/4pgd_PHE-PHE_C_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pgd/4pgd_PHE-PHE_C_1-9/4pgd_PHE-PHE_C_1-9_complex_UNL.pdb'), ('4pge_PHE-TRP_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pge/4pge_PHE-TRP_C_1-9/4pge_PHE-TRP_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pge/4pge_PHE-TRP_C_1-9/4pge_PHE-TRP_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pge/4pge_PHE-TRP_C_1-9/4pge_PHE-TRP_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pge/4pge_PHE-TRP_C_1-9/4pge_PHE-TRP_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pge/4pge_PHE-TRP_C_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pge/4pge_PHE-TRP_C_1-9/4pge_PHE-TRP_C_1-9_complex_UNL.pdb'), ('4pr5_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pr5/4pr5_HIS-TYR_C_1-11/4pr5_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pr5/4pr5_HIS-TYR_C_1-11/4pr5_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pr5/4pr5_HIS-TYR_C_1-11/4pr5_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pr5/4pr5_HIS-TYR_C_1-11/4pr5_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pr5/4pr5_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pr5/4pr5_HIS-TYR_C_1-11/4pr5_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4pra_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pra/4pra_HIS-TYR_C_1-11/4pra_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pra/4pra_HIS-TYR_C_1-11/4pra_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pra/4pra_HIS-TYR_C_1-11/4pra_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pra/4pra_HIS-TYR_C_1-11/4pra_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pra/4pra_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pra/4pra_HIS-TYR_C_1-11/4pra_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4prb_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prb/4prb_HIS-TYR_C_1-11/4prb_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prb/4prb_HIS-TYR_C_1-11/4prb_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prb/4prb_HIS-TYR_C_1-11/4prb_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prb/4prb_HIS-TYR_C_1-11/4prb_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prb/4prb_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prb/4prb_HIS-TYR_C_1-11/4prb_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4prd_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prd/4prd_HIS-TYR_C_1-11/4prd_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prd/4prd_HIS-TYR_C_1-11/4prd_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prd/4prd_HIS-TYR_C_1-11/4prd_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prd/4prd_HIS-TYR_C_1-11/4prd_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prd/4prd_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prd/4prd_HIS-TYR_C_1-11/4prd_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4pre_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pre/4pre_HIS-TYR_C_1-11/4pre_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pre/4pre_HIS-TYR_C_1-11/4pre_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pre/4pre_HIS-TYR_C_1-11/4pre_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pre/4pre_HIS-TYR_C_1-11/4pre_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pre/4pre_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pre/4pre_HIS-TYR_C_1-11/4pre_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4prh_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prh/4prh_HIS-TYR_C_1-11/4prh_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prh/4prh_HIS-TYR_C_1-11/4prh_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prh/4prh_HIS-TYR_C_1-11/4prh_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prh/4prh_HIS-TYR_C_1-11/4prh_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prh/4prh_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prh/4prh_HIS-TYR_C_1-11/4prh_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4pri_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pri/4pri_HIS-TYR_C_1-11/4pri_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pri/4pri_HIS-TYR_C_1-11/4pri_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pri/4pri_HIS-TYR_C_1-11/4pri_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pri/4pri_HIS-TYR_C_1-11/4pri_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pri/4pri_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4pri/4pri_HIS-TYR_C_1-11/4pri_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4prn_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prn/4prn_HIS-TYR_C_1-11/4prn_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prn/4prn_HIS-TYR_C_1-11/4prn_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prn/4prn_HIS-TYR_C_1-11/4prn_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prn/4prn_HIS-TYR_C_1-11/4prn_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prn/4prn_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prn/4prn_HIS-TYR_C_1-11/4prn_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4prp_HIS-TYR_C_1-11', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prp/4prp_HIS-TYR_C_1-11/4prp_HIS-TYR_C_1-11_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prp/4prp_HIS-TYR_C_1-11/4prp_HIS-TYR_C_1-11_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prp/4prp_HIS-TYR_C_1-11/4prp_HIS-TYR_C_1-11_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prp/4prp_HIS-TYR_C_1-11/4prp_HIS-TYR_C_1-11_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prp/4prp_HIS-TYR_C_1-11/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4prp/4prp_HIS-TYR_C_1-11/4prp_HIS-TYR_C_1-11_complex_UNL.pdb'), ('4qbm_GLY-LEU_D_14-22', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qbm/4qbm_GLY-LEU_D_14-22/4qbm_GLY-LEU_D_14-22_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qbm/4qbm_GLY-LEU_D_14-22/4qbm_GLY-LEU_D_14-22_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qbm/4qbm_GLY-LEU_D_14-22/4qbm_GLY-LEU_D_14-22_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qbm/4qbm_GLY-LEU_D_14-22/4qbm_GLY-LEU_D_14-22_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qbm/4qbm_GLY-LEU_D_14-22/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qbm/4qbm_GLY-LEU_D_14-22/4qbm_GLY-LEU_D_14-22_complex_UNL.pdb'), ('4qok_GLU-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qok/4qok_GLU-VAL_C_1-10/4qok_GLU-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qok/4qok_GLU-VAL_C_1-10/4qok_GLU-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qok/4qok_GLU-VAL_C_1-10/4qok_GLU-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qok/4qok_GLU-VAL_C_1-10/4qok_GLU-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qok/4qok_GLU-VAL_C_1-10/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4qok/4qok_GLU-VAL_C_1-10/4qok_GLU-VAL_C_1-10_complex_UNL.pdb'), ('4u6y_PHE-MET_P_1-15', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u6y/4u6y_PHE-MET_P_1-15/4u6y_PHE-MET_P_1-15_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u6y/4u6y_PHE-MET_P_1-15/4u6y_PHE-MET_P_1-15_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u6y/4u6y_PHE-MET_P_1-15/4u6y_PHE-MET_P_1-15_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u6y/4u6y_PHE-MET_P_1-15/4u6y_PHE-MET_P_1-15_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u6y/4u6y_PHE-MET_P_1-15/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u6y/4u6y_PHE-MET_P_1-15/4u6y_PHE-MET_P_1-15_complex_UNL.pdb'), ('4u91_PHE-CYS_E_398-408', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u91/4u91_PHE-CYS_E_398-408/4u91_PHE-CYS_E_398-408_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u91/4u91_PHE-CYS_E_398-408/4u91_PHE-CYS_E_398-408_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u91/4u91_PHE-CYS_E_398-408/4u91_PHE-CYS_E_398-408_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u91/4u91_PHE-CYS_E_398-408/4u91_PHE-CYS_E_398-408_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u91/4u91_PHE-CYS_E_398-408/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4u91/4u91_PHE-CYS_E_398-408/4u91_PHE-CYS_E_398-408_complex_UNL.pdb'), ('4v11_SER-ASP_B_81-90', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4v11/4v11_SER-ASP_B_81-90/4v11_SER-ASP_B_81-90_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4v11/4v11_SER-ASP_B_81-90/4v11_SER-ASP_B_81-90_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4v11/4v11_SER-ASP_B_81-90/4v11_SER-ASP_B_81-90_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4v11/4v11_SER-ASP_B_81-90/4v11_SER-ASP_B_81-90_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4v11/4v11_SER-ASP_B_81-90/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4v11/4v11_SER-ASP_B_81-90/4v11_SER-ASP_B_81-90_complex_UNL.pdb'), ('4wj5_GRN-LPH_F_0-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wj5/4wj5_GRN-LPH_F_0-9/4wj5_GRN-LPH_F_0-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wj5/4wj5_GRN-LPH_F_0-9/4wj5_GRN-LPH_F_0-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wj5/4wj5_GRN-LPH_F_0-9/4wj5_GRN-LPH_F_0-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wj5/4wj5_GRN-LPH_F_0-9/4wj5_GRN-LPH_F_0-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wj5/4wj5_GRN-LPH_F_0-9/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wj5/4wj5_GRN-LPH_F_0-9/4wj5_GRN-LPH_F_0-9_complex_UNL.pdb'), ('4wvt_MAA-3V7_C_1-6', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvt/4wvt_MAA-3V7_C_1-6/4wvt_MAA-3V7_C_1-6_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvt/4wvt_MAA-3V7_C_1-6/4wvt_MAA-3V7_C_1-6_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvt/4wvt_MAA-3V7_C_1-6/4wvt_MAA-3V7_C_1-6_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvt/4wvt_MAA-3V7_C_1-6/4wvt_MAA-3V7_C_1-6_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvt/4wvt_MAA-3V7_C_1-6/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvt/4wvt_MAA-3V7_C_1-6/4wvt_MAA-3V7_C_1-6_complex_UNL.pdb'), ('4wvu_MAA-HOX_B_1-5', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvu/4wvu_MAA-HOX_B_1-5/4wvu_MAA-HOX_B_1-5_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvu/4wvu_MAA-HOX_B_1-5/4wvu_MAA-HOX_B_1-5_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvu/4wvu_MAA-HOX_B_1-5/4wvu_MAA-HOX_B_1-5_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvu/4wvu_MAA-HOX_B_1-5/4wvu_MAA-HOX_B_1-5_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvu/4wvu_MAA-HOX_B_1-5/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4wvu/4wvu_MAA-HOX_B_1-5/4wvu_MAA-HOX_B_1-5_complex_UNL.pdb'), ('4xhv_ASP-VAL_B_1831-1840', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4xhv/4xhv_ASP-VAL_B_1831-1840/4xhv_ASP-VAL_B_1831-1840_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4xhv/4xhv_ASP-VAL_B_1831-1840/4xhv_ASP-VAL_B_1831-1840_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4xhv/4xhv_ASP-VAL_B_1831-1840/4xhv_ASP-VAL_B_1831-1840_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4xhv/4xhv_ASP-VAL_B_1831-1840/4xhv_ASP-VAL_B_1831-1840_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4xhv/4xhv_ASP-VAL_B_1831-1840/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4xhv/4xhv_ASP-VAL_B_1831-1840/4xhv_ASP-VAL_B_1831-1840_complex_UNL.pdb'), ('4y7r_GLU-VAL_B_260-267', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4y7r/4y7r_GLU-VAL_B_260-267/4y7r_GLU-VAL_B_260-267_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4y7r/4y7r_GLU-VAL_B_260-267/4y7r_GLU-VAL_B_260-267_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4y7r/4y7r_GLU-VAL_B_260-267/4y7r_GLU-VAL_B_260-267_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4y7r/4y7r_GLU-VAL_B_260-267/4y7r_GLU-VAL_B_260-267_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4y7r/4y7r_GLU-VAL_B_260-267/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4y7r/4y7r_GLU-VAL_B_260-267/4y7r_GLU-VAL_B_260-267_complex_UNL.pdb'), ('4yk6_TYR-PRO_B_365-375', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4yk6/4yk6_TYR-PRO_B_365-375/4yk6_TYR-PRO_B_365-375_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4yk6/4yk6_TYR-PRO_B_365-375/4yk6_TYR-PRO_B_365-375_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4yk6/4yk6_TYR-PRO_B_365-375/4yk6_TYR-PRO_B_365-375_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4yk6/4yk6_TYR-PRO_B_365-375/4yk6_TYR-PRO_B_365-375_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4yk6/4yk6_TYR-PRO_B_365-375/interaction.xml', 'TYR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4yk6/4yk6_TYR-PRO_B_365-375/4yk6_TYR-PRO_B_365-375_complex_UNL.pdb'), ('4z8a_ACE-NH2_B_1684-1700', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4z8a/4z8a_ACE-NH2_B_1684-1700/4z8a_ACE-NH2_B_1684-1700_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4z8a/4z8a_ACE-NH2_B_1684-1700/4z8a_ACE-NH2_B_1684-1700_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4z8a/4z8a_ACE-NH2_B_1684-1700/4z8a_ACE-NH2_B_1684-1700_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4z8a/4z8a_ACE-NH2_B_1684-1700/4z8a_ACE-NH2_B_1684-1700_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4z8a/4z8a_ACE-NH2_B_1684-1700/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/4z8a/4z8a_ACE-NH2_B_1684-1700/4z8a_ACE-NH2_B_1684-1700_complex_UNL.pdb'), ('5aei_LYS-ARG_F_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aei/5aei_LYS-ARG_F_1-10/5aei_LYS-ARG_F_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aei/5aei_LYS-ARG_F_1-10/5aei_LYS-ARG_F_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aei/5aei_LYS-ARG_F_1-10/5aei_LYS-ARG_F_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aei/5aei_LYS-ARG_F_1-10/5aei_LYS-ARG_F_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aei/5aei_LYS-ARG_F_1-10/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aei/5aei_LYS-ARG_F_1-10/5aei_LYS-ARG_F_1-10_complex_UNL.pdb'), ('5aer_ASN-SER_B_430-443', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aer/5aer_ASN-SER_B_430-443/5aer_ASN-SER_B_430-443_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aer/5aer_ASN-SER_B_430-443/5aer_ASN-SER_B_430-443_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aer/5aer_ASN-SER_B_430-443/5aer_ASN-SER_B_430-443_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aer/5aer_ASN-SER_B_430-443/5aer_ASN-SER_B_430-443_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aer/5aer_ASN-SER_B_430-443/interaction.xml', 'ASN', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5aer/5aer_ASN-SER_B_430-443/5aer_ASN-SER_B_430-443_complex_UNL.pdb'), ('5b6c_PHE-LEU_B_225-235', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5b6c/5b6c_PHE-LEU_B_225-235/5b6c_PHE-LEU_B_225-235_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5b6c/5b6c_PHE-LEU_B_225-235/5b6c_PHE-LEU_B_225-235_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5b6c/5b6c_PHE-LEU_B_225-235/5b6c_PHE-LEU_B_225-235_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5b6c/5b6c_PHE-LEU_B_225-235/5b6c_PHE-LEU_B_225-235_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5b6c/5b6c_PHE-LEU_B_225-235/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5b6c/5b6c_PHE-LEU_B_225-235/5b6c_PHE-LEU_B_225-235_complex_UNL.pdb'), ('5brz_GLU-TYR_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5brz/5brz_GLU-TYR_C_1-9/5brz_GLU-TYR_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5brz/5brz_GLU-TYR_C_1-9/5brz_GLU-TYR_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5brz/5brz_GLU-TYR_C_1-9/5brz_GLU-TYR_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5brz/5brz_GLU-TYR_C_1-9/5brz_GLU-TYR_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5brz/5brz_GLU-TYR_C_1-9/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5brz/5brz_GLU-TYR_C_1-9/5brz_GLU-TYR_C_1-9_complex_UNL.pdb'), ('5bs0_GLU-TYR_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5bs0/5bs0_GLU-TYR_C_1-9/5bs0_GLU-TYR_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5bs0/5bs0_GLU-TYR_C_1-9/5bs0_GLU-TYR_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5bs0/5bs0_GLU-TYR_C_1-9/5bs0_GLU-TYR_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5bs0/5bs0_GLU-TYR_C_1-9/5bs0_GLU-TYR_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5bs0/5bs0_GLU-TYR_C_1-9/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5bs0/5bs0_GLU-TYR_C_1-9/5bs0_GLU-TYR_C_1-9_complex_UNL.pdb'), ('5dd0_ACE-NH2_P_659-671', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5dd0/5dd0_ACE-NH2_P_659-671/5dd0_ACE-NH2_P_659-671_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5dd0/5dd0_ACE-NH2_P_659-671/5dd0_ACE-NH2_P_659-671_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5dd0/5dd0_ACE-NH2_P_659-671/5dd0_ACE-NH2_P_659-671_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5dd0/5dd0_ACE-NH2_P_659-671/5dd0_ACE-NH2_P_659-671_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5dd0/5dd0_ACE-NH2_P_659-671/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5dd0/5dd0_ACE-NH2_P_659-671/5dd0_ACE-NH2_P_659-671_complex_UNL.pdb'), ('5e2q_ASP-PHE_B_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5e2q/5e2q_ASP-PHE_B_1-8/5e2q_ASP-PHE_B_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5e2q/5e2q_ASP-PHE_B_1-8/5e2q_ASP-PHE_B_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5e2q/5e2q_ASP-PHE_B_1-8/5e2q_ASP-PHE_B_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5e2q/5e2q_ASP-PHE_B_1-8/5e2q_ASP-PHE_B_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5e2q/5e2q_ASP-PHE_B_1-8/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5e2q/5e2q_ASP-PHE_B_1-8/5e2q_ASP-PHE_B_1-8_complex_UNL.pdb'), ('5gow_TYR-ASP_A_392-410', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gow/5gow_TYR-ASP_A_392-410/5gow_TYR-ASP_A_392-410_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gow/5gow_TYR-ASP_A_392-410/5gow_TYR-ASP_A_392-410_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gow/5gow_TYR-ASP_A_392-410/5gow_TYR-ASP_A_392-410_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gow/5gow_TYR-ASP_A_392-410/5gow_TYR-ASP_A_392-410_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gow/5gow_TYR-ASP_A_392-410/interaction.xml', 'TYR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gow/5gow_TYR-ASP_A_392-410/5gow_TYR-ASP_A_392-410_complex_UNL.pdb'), ('5gr9_HIS-ASN_C_93-104', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gr9/5gr9_HIS-ASN_C_93-104/5gr9_HIS-ASN_C_93-104_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gr9/5gr9_HIS-ASN_C_93-104/5gr9_HIS-ASN_C_93-104_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gr9/5gr9_HIS-ASN_C_93-104/5gr9_HIS-ASN_C_93-104_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gr9/5gr9_HIS-ASN_C_93-104/5gr9_HIS-ASN_C_93-104_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gr9/5gr9_HIS-ASN_C_93-104/interaction.xml', 'HIS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5gr9/5gr9_HIS-ASN_C_93-104/5gr9_HIS-ASN_C_93-104_complex_UNL.pdb'), ('5h7g_LEU-PRO_D_801-813', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5h7g/5h7g_LEU-PRO_D_801-813/5h7g_LEU-PRO_D_801-813_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5h7g/5h7g_LEU-PRO_D_801-813/5h7g_LEU-PRO_D_801-813_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5h7g/5h7g_LEU-PRO_D_801-813/5h7g_LEU-PRO_D_801-813_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5h7g/5h7g_LEU-PRO_D_801-813/5h7g_LEU-PRO_D_801-813_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5h7g/5h7g_LEU-PRO_D_801-813/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5h7g/5h7g_LEU-PRO_D_801-813/5h7g_LEU-PRO_D_801-813_complex_UNL.pdb'), ('5hyx_ASP-ASN_A_1-13', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5hyx/5hyx_ASP-ASN_A_1-13/5hyx_ASP-ASN_A_1-13_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5hyx/5hyx_ASP-ASN_A_1-13/5hyx_ASP-ASN_A_1-13_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5hyx/5hyx_ASP-ASN_A_1-13/5hyx_ASP-ASN_A_1-13_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5hyx/5hyx_ASP-ASN_A_1-13/5hyx_ASP-ASN_A_1-13_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5hyx/5hyx_ASP-ASN_A_1-13/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5hyx/5hyx_ASP-ASN_A_1-13/5hyx_ASP-ASN_A_1-13_complex_UNL.pdb'), ('5i22_SER-THR_B_1728-1744', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5i22/5i22_SER-THR_B_1728-1744/5i22_SER-THR_B_1728-1744_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5i22/5i22_SER-THR_B_1728-1744/5i22_SER-THR_B_1728-1744_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5i22/5i22_SER-THR_B_1728-1744/5i22_SER-THR_B_1728-1744_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5i22/5i22_SER-THR_B_1728-1744/5i22_SER-THR_B_1728-1744_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5i22/5i22_SER-THR_B_1728-1744/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5i22/5i22_SER-THR_B_1728-1744/5i22_SER-THR_B_1728-1744_complex_UNL.pdb'), ('5iay_THR-SER_B_642-657', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iay/5iay_THR-SER_B_642-657/5iay_THR-SER_B_642-657_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iay/5iay_THR-SER_B_642-657/5iay_THR-SER_B_642-657_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iay/5iay_THR-SER_B_642-657/5iay_THR-SER_B_642-657_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iay/5iay_THR-SER_B_642-657/5iay_THR-SER_B_642-657_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iay/5iay_THR-SER_B_642-657/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iay/5iay_THR-SER_B_642-657/5iay_THR-SER_B_642-657_complex_UNL.pdb'), ('5iql_ALA-ALA_B_24-31', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iql/5iql_ALA-ALA_B_24-31/5iql_ALA-ALA_B_24-31_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iql/5iql_ALA-ALA_B_24-31/5iql_ALA-ALA_B_24-31_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iql/5iql_ALA-ALA_B_24-31/5iql_ALA-ALA_B_24-31_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iql/5iql_ALA-ALA_B_24-31/5iql_ALA-ALA_B_24-31_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iql/5iql_ALA-ALA_B_24-31/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5iql/5iql_ALA-ALA_B_24-31/5iql_ALA-ALA_B_24-31_complex_UNL.pdb'), ('5isz_GLY-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5isz/5isz_GLY-LEU_C_1-9/5isz_GLY-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5isz/5isz_GLY-LEU_C_1-9/5isz_GLY-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5isz/5isz_GLY-LEU_C_1-9/5isz_GLY-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5isz/5isz_GLY-LEU_C_1-9/5isz_GLY-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5isz/5isz_GLY-LEU_C_1-9/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5isz/5isz_GLY-LEU_C_1-9/5isz_GLY-LEU_C_1-9_complex_UNL.pdb'), ('5ixf_ALA-ALA_B_1-14', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixf/5ixf_ALA-ALA_B_1-14/5ixf_ALA-ALA_B_1-14_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixf/5ixf_ALA-ALA_B_1-14/5ixf_ALA-ALA_B_1-14_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixf/5ixf_ALA-ALA_B_1-14/5ixf_ALA-ALA_B_1-14_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixf/5ixf_ALA-ALA_B_1-14/5ixf_ALA-ALA_B_1-14_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixf/5ixf_ALA-ALA_B_1-14/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixf/5ixf_ALA-ALA_B_1-14/5ixf_ALA-ALA_B_1-14_complex_UNL.pdb'), ('5ixq_PRO-ASN_B_58-69', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixq/5ixq_PRO-ASN_B_58-69/5ixq_PRO-ASN_B_58-69_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixq/5ixq_PRO-ASN_B_58-69/5ixq_PRO-ASN_B_58-69_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixq/5ixq_PRO-ASN_B_58-69/5ixq_PRO-ASN_B_58-69_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixq/5ixq_PRO-ASN_B_58-69/5ixq_PRO-ASN_B_58-69_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixq/5ixq_PRO-ASN_B_58-69/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ixq/5ixq_PRO-ASN_B_58-69/5ixq_PRO-ASN_B_58-69_complex_UNL.pdb'), ('5izu_ALA-LEU_D_963-977', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5izu/5izu_ALA-LEU_D_963-977/5izu_ALA-LEU_D_963-977_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5izu/5izu_ALA-LEU_D_963-977/5izu_ALA-LEU_D_963-977_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5izu/5izu_ALA-LEU_D_963-977/5izu_ALA-LEU_D_963-977_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5izu/5izu_ALA-LEU_D_963-977/5izu_ALA-LEU_D_963-977_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5izu/5izu_ALA-LEU_D_963-977/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5izu/5izu_ALA-LEU_D_963-977/5izu_ALA-LEU_D_963-977_complex_UNL.pdb'), ('5jhd_GLY-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jhd/5jhd_GLY-LEU_C_1-9/5jhd_GLY-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jhd/5jhd_GLY-LEU_C_1-9/5jhd_GLY-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jhd/5jhd_GLY-LEU_C_1-9/5jhd_GLY-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jhd/5jhd_GLY-LEU_C_1-9/5jhd_GLY-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jhd/5jhd_GLY-LEU_C_1-9/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jhd/5jhd_GLY-LEU_C_1-9/5jhd_GLY-LEU_C_1-9_complex_UNL.pdb'), ('5jzi_LYS-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jzi/5jzi_LYS-VAL_C_1-10/5jzi_LYS-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jzi/5jzi_LYS-VAL_C_1-10/5jzi_LYS-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jzi/5jzi_LYS-VAL_C_1-10/5jzi_LYS-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jzi/5jzi_LYS-VAL_C_1-10/5jzi_LYS-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jzi/5jzi_LYS-VAL_C_1-10/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5jzi/5jzi_LYS-VAL_C_1-10/5jzi_LYS-VAL_C_1-10_complex_UNL.pdb'), ('5lvf_PRO-SER_B_143-158', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5lvf/5lvf_PRO-SER_B_143-158/5lvf_PRO-SER_B_143-158_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5lvf/5lvf_PRO-SER_B_143-158/5lvf_PRO-SER_B_143-158_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5lvf/5lvf_PRO-SER_B_143-158/5lvf_PRO-SER_B_143-158_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5lvf/5lvf_PRO-SER_B_143-158/5lvf_PRO-SER_B_143-158_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5lvf/5lvf_PRO-SER_B_143-158/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5lvf/5lvf_PRO-SER_B_143-158/5lvf_PRO-SER_B_143-158_complex_UNL.pdb'), ('5m9d_THR-SER_B_139-154', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5m9d/5m9d_THR-SER_B_139-154/5m9d_THR-SER_B_139-154_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5m9d/5m9d_THR-SER_B_139-154/5m9d_THR-SER_B_139-154_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5m9d/5m9d_THR-SER_B_139-154/5m9d_THR-SER_B_139-154_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5m9d/5m9d_THR-SER_B_139-154/5m9d_THR-SER_B_139-154_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5m9d/5m9d_THR-SER_B_139-154/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5m9d/5m9d_THR-SER_B_139-154/5m9d_THR-SER_B_139-154_complex_UNL.pdb'), ('5moc_LYS-ASP_P_382-393', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5moc/5moc_LYS-ASP_P_382-393/5moc_LYS-ASP_P_382-393_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5moc/5moc_LYS-ASP_P_382-393/5moc_LYS-ASP_P_382-393_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5moc/5moc_LYS-ASP_P_382-393/5moc_LYS-ASP_P_382-393_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5moc/5moc_LYS-ASP_P_382-393/5moc_LYS-ASP_P_382-393_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5moc/5moc_LYS-ASP_P_382-393/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5moc/5moc_LYS-ASP_P_382-393/5moc_LYS-ASP_P_382-393_complex_UNL.pdb'), ('5n1y_MET-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n1y/5n1y_MET-VAL_C_1-10/5n1y_MET-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n1y/5n1y_MET-VAL_C_1-10/5n1y_MET-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n1y/5n1y_MET-VAL_C_1-10/5n1y_MET-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n1y/5n1y_MET-VAL_C_1-10/5n1y_MET-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n1y/5n1y_MET-VAL_C_1-10/interaction.xml', 'MET', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n1y/5n1y_MET-VAL_C_1-10/5n1y_MET-VAL_C_1-10_complex_UNL.pdb'), ('5n70_ALA-ARG_B_5-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n70/5n70_ALA-ARG_B_5-12/5n70_ALA-ARG_B_5-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n70/5n70_ALA-ARG_B_5-12/5n70_ALA-ARG_B_5-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n70/5n70_ALA-ARG_B_5-12/5n70_ALA-ARG_B_5-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n70/5n70_ALA-ARG_B_5-12/5n70_ALA-ARG_B_5-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n70/5n70_ALA-ARG_B_5-12/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5n70/5n70_ALA-ARG_B_5-12/5n70_ALA-ARG_B_5-12_complex_UNL.pdb'), ('5nme_SER-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nme/5nme_SER-LEU_C_1-9/5nme_SER-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nme/5nme_SER-LEU_C_1-9/5nme_SER-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nme/5nme_SER-LEU_C_1-9/5nme_SER-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nme/5nme_SER-LEU_C_1-9/5nme_SER-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nme/5nme_SER-LEU_C_1-9/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nme/5nme_SER-LEU_C_1-9/5nme_SER-LEU_C_1-9_complex_UNL.pdb'), ('5nmf_SER-LEU_H_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmf/5nmf_SER-LEU_H_1-9/5nmf_SER-LEU_H_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmf/5nmf_SER-LEU_H_1-9/5nmf_SER-LEU_H_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmf/5nmf_SER-LEU_H_1-9/5nmf_SER-LEU_H_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmf/5nmf_SER-LEU_H_1-9/5nmf_SER-LEU_H_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmf/5nmf_SER-LEU_H_1-9/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmf/5nmf_SER-LEU_H_1-9/5nmf_SER-LEU_H_1-9_complex_UNL.pdb'), ('5nmg_SER-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmg/5nmg_SER-LEU_C_1-9/5nmg_SER-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmg/5nmg_SER-LEU_C_1-9/5nmg_SER-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmg/5nmg_SER-LEU_C_1-9/5nmg_SER-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmg/5nmg_SER-LEU_C_1-9/5nmg_SER-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmg/5nmg_SER-LEU_C_1-9/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5nmg/5nmg_SER-LEU_C_1-9/5nmg_SER-LEU_C_1-9_complex_UNL.pdb'), ('5o0e_THR-ILE_C_9-26', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5o0e/5o0e_THR-ILE_C_9-26/5o0e_THR-ILE_C_9-26_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5o0e/5o0e_THR-ILE_C_9-26/5o0e_THR-ILE_C_9-26_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5o0e/5o0e_THR-ILE_C_9-26/5o0e_THR-ILE_C_9-26_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5o0e/5o0e_THR-ILE_C_9-26/5o0e_THR-ILE_C_9-26_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5o0e/5o0e_THR-ILE_C_9-26/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5o0e/5o0e_THR-ILE_C_9-26/5o0e_THR-ILE_C_9-26_complex_UNL.pdb'), ('5ok3_THR-ILE_D_8-25', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ok3/5ok3_THR-ILE_D_8-25/5ok3_THR-ILE_D_8-25_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ok3/5ok3_THR-ILE_D_8-25/5ok3_THR-ILE_D_8-25_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ok3/5ok3_THR-ILE_D_8-25/5ok3_THR-ILE_D_8-25_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ok3/5ok3_THR-ILE_D_8-25/5ok3_THR-ILE_D_8-25_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ok3/5ok3_THR-ILE_D_8-25/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ok3/5ok3_THR-ILE_D_8-25/5ok3_THR-ILE_D_8-25_complex_UNL.pdb'), ('5ol3_THR-ILE_D_8-25', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ol3/5ol3_THR-ILE_D_8-25/5ol3_THR-ILE_D_8-25_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ol3/5ol3_THR-ILE_D_8-25/5ol3_THR-ILE_D_8-25_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ol3/5ol3_THR-ILE_D_8-25/5ol3_THR-ILE_D_8-25_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ol3/5ol3_THR-ILE_D_8-25/5ol3_THR-ILE_D_8-25_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ol3/5ol3_THR-ILE_D_8-25/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ol3/5ol3_THR-ILE_D_8-25/5ol3_THR-ILE_D_8-25_complex_UNL.pdb'), ('5t6z_THR-TRP_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t6z/5t6z_THR-TRP_C_1-10/5t6z_THR-TRP_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t6z/5t6z_THR-TRP_C_1-10/5t6z_THR-TRP_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t6z/5t6z_THR-TRP_C_1-10/5t6z_THR-TRP_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t6z/5t6z_THR-TRP_C_1-10/5t6z_THR-TRP_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t6z/5t6z_THR-TRP_C_1-10/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t6z/5t6z_THR-TRP_C_1-10/5t6z_THR-TRP_C_1-10_complex_UNL.pdb'), ('5t70_THR-TRP_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t70/5t70_THR-TRP_C_1-10/5t70_THR-TRP_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t70/5t70_THR-TRP_C_1-10/5t70_THR-TRP_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t70/5t70_THR-TRP_C_1-10/5t70_THR-TRP_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t70/5t70_THR-TRP_C_1-10/5t70_THR-TRP_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t70/5t70_THR-TRP_C_1-10/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5t70/5t70_THR-TRP_C_1-10/5t70_THR-TRP_C_1-10_complex_UNL.pdb'), ('5tbn_ALA-NH2_C_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5tbn/5tbn_ALA-NH2_C_1-12/5tbn_ALA-NH2_C_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5tbn/5tbn_ALA-NH2_C_1-12/5tbn_ALA-NH2_C_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5tbn/5tbn_ALA-NH2_C_1-12/5tbn_ALA-NH2_C_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5tbn/5tbn_ALA-NH2_C_1-12/5tbn_ALA-NH2_C_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5tbn/5tbn_ALA-NH2_C_1-12/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5tbn/5tbn_ALA-NH2_C_1-12/5tbn_ALA-NH2_C_1-12_complex_UNL.pdb'), ('5u98_VAL-VAL_F_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5u98/5u98_VAL-VAL_F_1-9/5u98_VAL-VAL_F_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5u98/5u98_VAL-VAL_F_1-9/5u98_VAL-VAL_F_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5u98/5u98_VAL-VAL_F_1-9/5u98_VAL-VAL_F_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5u98/5u98_VAL-VAL_F_1-9/5u98_VAL-VAL_F_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5u98/5u98_VAL-VAL_F_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5u98/5u98_VAL-VAL_F_1-9/5u98_VAL-VAL_F_1-9_complex_UNL.pdb'), ('5vwi_PRO-LEU_D_10-17', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwi/5vwi_PRO-LEU_D_10-17/5vwi_PRO-LEU_D_10-17_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwi/5vwi_PRO-LEU_D_10-17/5vwi_PRO-LEU_D_10-17_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwi/5vwi_PRO-LEU_D_10-17/5vwi_PRO-LEU_D_10-17_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwi/5vwi_PRO-LEU_D_10-17/5vwi_PRO-LEU_D_10-17_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwi/5vwi_PRO-LEU_D_10-17/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwi/5vwi_PRO-LEU_D_10-17/5vwi_PRO-LEU_D_10-17_complex_UNL.pdb'), ('5vwk_PRO-LEU_G_156-163', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwk/5vwk_PRO-LEU_G_156-163/5vwk_PRO-LEU_G_156-163_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwk/5vwk_PRO-LEU_G_156-163/5vwk_PRO-LEU_G_156-163_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwk/5vwk_PRO-LEU_G_156-163/5vwk_PRO-LEU_G_156-163_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwk/5vwk_PRO-LEU_G_156-163/5vwk_PRO-LEU_G_156-163_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwk/5vwk_PRO-LEU_G_156-163/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5vwk/5vwk_PRO-LEU_G_156-163/5vwk_PRO-LEU_G_156-163_complex_UNL.pdb'), ('5w1v_VAL-LEU_M_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1v/5w1v_VAL-LEU_M_1-9/5w1v_VAL-LEU_M_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1v/5w1v_VAL-LEU_M_1-9/5w1v_VAL-LEU_M_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1v/5w1v_VAL-LEU_M_1-9/5w1v_VAL-LEU_M_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1v/5w1v_VAL-LEU_M_1-9/5w1v_VAL-LEU_M_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1v/5w1v_VAL-LEU_M_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1v/5w1v_VAL-LEU_M_1-9/5w1v_VAL-LEU_M_1-9_complex_UNL.pdb'), ('5w1w_VAL-LEU_M_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1w/5w1w_VAL-LEU_M_1-9/5w1w_VAL-LEU_M_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1w/5w1w_VAL-LEU_M_1-9/5w1w_VAL-LEU_M_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1w/5w1w_VAL-LEU_M_1-9/5w1w_VAL-LEU_M_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1w/5w1w_VAL-LEU_M_1-9/5w1w_VAL-LEU_M_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1w/5w1w_VAL-LEU_M_1-9/interaction.xml', 'VAL', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5w1w/5w1w_VAL-LEU_M_1-9/5w1w_VAL-LEU_M_1-9_complex_UNL.pdb'), ('5wkf_GLY-ARG_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkf/5wkf_GLY-ARG_C_1-10/5wkf_GLY-ARG_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkf/5wkf_GLY-ARG_C_1-10/5wkf_GLY-ARG_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkf/5wkf_GLY-ARG_C_1-10/5wkf_GLY-ARG_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkf/5wkf_GLY-ARG_C_1-10/5wkf_GLY-ARG_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkf/5wkf_GLY-ARG_C_1-10/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkf/5wkf_GLY-ARG_C_1-10/5wkf_GLY-ARG_C_1-10_complex_UNL.pdb'), ('5wkh_GLY-ARG_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkh/5wkh_GLY-ARG_C_1-10/5wkh_GLY-ARG_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkh/5wkh_GLY-ARG_C_1-10/5wkh_GLY-ARG_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkh/5wkh_GLY-ARG_C_1-10/5wkh_GLY-ARG_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkh/5wkh_GLY-ARG_C_1-10/5wkh_GLY-ARG_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkh/5wkh_GLY-ARG_C_1-10/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wkh/5wkh_GLY-ARG_C_1-10/5wkh_GLY-ARG_C_1-10_complex_UNL.pdb'), ('5wlg_SER-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wlg/5wlg_SER-LEU_C_1-9/5wlg_SER-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wlg/5wlg_SER-LEU_C_1-9/5wlg_SER-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wlg/5wlg_SER-LEU_C_1-9/5wlg_SER-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wlg/5wlg_SER-LEU_C_1-9/5wlg_SER-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wlg/5wlg_SER-LEU_C_1-9/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wlg/5wlg_SER-LEU_C_1-9/5wlg_SER-LEU_C_1-9_complex_UNL.pdb'), ('5wou_LEU-LEU_V_98-105', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wou/5wou_LEU-LEU_V_98-105/5wou_LEU-LEU_V_98-105_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wou/5wou_LEU-LEU_V_98-105/5wou_LEU-LEU_V_98-105_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wou/5wou_LEU-LEU_V_98-105/5wou_LEU-LEU_V_98-105_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wou/5wou_LEU-LEU_V_98-105/5wou_LEU-LEU_V_98-105_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wou/5wou_LEU-LEU_V_98-105/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5wou/5wou_LEU-LEU_V_98-105/5wou_LEU-LEU_V_98-105_complex_UNL.pdb'), ('5xjm_SAR-ILE_B_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xjm/5xjm_SAR-ILE_B_1-8/5xjm_SAR-ILE_B_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xjm/5xjm_SAR-ILE_B_1-8/5xjm_SAR-ILE_B_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xjm/5xjm_SAR-ILE_B_1-8/5xjm_SAR-ILE_B_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xjm/5xjm_SAR-ILE_B_1-8/5xjm_SAR-ILE_B_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xjm/5xjm_SAR-ILE_B_1-8/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xjm/5xjm_SAR-ILE_B_1-8/5xjm_SAR-ILE_B_1-8_complex_UNL.pdb'), ('5xn3_ARG-VAL_B_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xn3/5xn3_ARG-VAL_B_1-8/5xn3_ARG-VAL_B_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xn3/5xn3_ARG-VAL_B_1-8/5xn3_ARG-VAL_B_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xn3/5xn3_ARG-VAL_B_1-8/5xn3_ARG-VAL_B_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xn3/5xn3_ARG-VAL_B_1-8/5xn3_ARG-VAL_B_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xn3/5xn3_ARG-VAL_B_1-8/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5xn3/5xn3_ARG-VAL_B_1-8/5xn3_ARG-VAL_B_1-8_complex_UNL.pdb'), ('5ybe_PRO-ASP_B_1138-1156', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ybe/5ybe_PRO-ASP_B_1138-1156/5ybe_PRO-ASP_B_1138-1156_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ybe/5ybe_PRO-ASP_B_1138-1156/5ybe_PRO-ASP_B_1138-1156_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ybe/5ybe_PRO-ASP_B_1138-1156/5ybe_PRO-ASP_B_1138-1156_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ybe/5ybe_PRO-ASP_B_1138-1156/5ybe_PRO-ASP_B_1138-1156_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ybe/5ybe_PRO-ASP_B_1138-1156/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5ybe/5ybe_PRO-ASP_B_1138-1156/5ybe_PRO-ASP_B_1138-1156_complex_UNL.pdb'), ('5zys_LEU-VAL_B_1247-1256', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5zys/5zys_LEU-VAL_B_1247-1256/5zys_LEU-VAL_B_1247-1256_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5zys/5zys_LEU-VAL_B_1247-1256/5zys_LEU-VAL_B_1247-1256_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5zys/5zys_LEU-VAL_B_1247-1256/5zys_LEU-VAL_B_1247-1256_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5zys/5zys_LEU-VAL_B_1247-1256/5zys_LEU-VAL_B_1247-1256_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5zys/5zys_LEU-VAL_B_1247-1256/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/5zys/5zys_LEU-VAL_B_1247-1256/5zys_LEU-VAL_B_1247-1256_complex_UNL.pdb'), ('6aee_ARG-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6aee/6aee_ARG-LEU_C_1-9/6aee_ARG-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6aee/6aee_ARG-LEU_C_1-9/6aee_ARG-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6aee/6aee_ARG-LEU_C_1-9/6aee_ARG-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6aee/6aee_ARG-LEU_C_1-9/6aee_ARG-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6aee/6aee_ARG-LEU_C_1-9/interaction.xml', 'ARG', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6aee/6aee_ARG-LEU_C_1-9/6aee_ARG-LEU_C_1-9_complex_UNL.pdb'), ('6bj3_ILE-LEU_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bj3/6bj3_ILE-LEU_C_1-9/6bj3_ILE-LEU_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bj3/6bj3_ILE-LEU_C_1-9/6bj3_ILE-LEU_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bj3/6bj3_ILE-LEU_C_1-9/6bj3_ILE-LEU_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bj3/6bj3_ILE-LEU_C_1-9/6bj3_ILE-LEU_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bj3/6bj3_ILE-LEU_C_1-9/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bj3/6bj3_ILE-LEU_C_1-9/6bj3_ILE-LEU_C_1-9_complex_UNL.pdb'), ('6bnh_LYS-ASN_B_201-213', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bnh/6bnh_LYS-ASN_B_201-213/6bnh_LYS-ASN_B_201-213_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bnh/6bnh_LYS-ASN_B_201-213/6bnh_LYS-ASN_B_201-213_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bnh/6bnh_LYS-ASN_B_201-213/6bnh_LYS-ASN_B_201-213_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bnh/6bnh_LYS-ASN_B_201-213/6bnh_LYS-ASN_B_201-213_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bnh/6bnh_LYS-ASN_B_201-213/interaction.xml', 'LYS', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6bnh/6bnh_LYS-ASN_B_201-213/6bnh_LYS-ASN_B_201-213_complex_UNL.pdb'), ('6ct7_ACE-LYS_S_0-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ct7/6ct7_ACE-LYS_S_0-10/6ct7_ACE-LYS_S_0-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ct7/6ct7_ACE-LYS_S_0-10/6ct7_ACE-LYS_S_0-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ct7/6ct7_ACE-LYS_S_0-10/6ct7_ACE-LYS_S_0-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ct7/6ct7_ACE-LYS_S_0-10/6ct7_ACE-LYS_S_0-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ct7/6ct7_ACE-LYS_S_0-10/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ct7/6ct7_ACE-LYS_S_0-10/6ct7_ACE-LYS_S_0-10_complex_UNL.pdb'), ('6d78_ALA-VAL_C_2-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6d78/6d78_ALA-VAL_C_2-10/6d78_ALA-VAL_C_2-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6d78/6d78_ALA-VAL_C_2-10/6d78_ALA-VAL_C_2-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6d78/6d78_ALA-VAL_C_2-10/6d78_ALA-VAL_C_2-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6d78/6d78_ALA-VAL_C_2-10/6d78_ALA-VAL_C_2-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6d78/6d78_ALA-VAL_C_2-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6d78/6d78_ALA-VAL_C_2-10/6d78_ALA-VAL_C_2-10_complex_UNL.pdb'), ('6dkp_GLU-VAL_C_1-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dkp/6dkp_GLU-VAL_C_1-10/6dkp_GLU-VAL_C_1-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dkp/6dkp_GLU-VAL_C_1-10/6dkp_GLU-VAL_C_1-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dkp/6dkp_GLU-VAL_C_1-10/6dkp_GLU-VAL_C_1-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dkp/6dkp_GLU-VAL_C_1-10/6dkp_GLU-VAL_C_1-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dkp/6dkp_GLU-VAL_C_1-10/interaction.xml', 'GLU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dkp/6dkp_GLU-VAL_C_1-10/6dkp_GLU-VAL_C_1-10_complex_UNL.pdb'), ('6do1_SAR-ILE_H_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6do1/6do1_SAR-ILE_H_1-8/6do1_SAR-ILE_H_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6do1/6do1_SAR-ILE_H_1-8/6do1_SAR-ILE_H_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6do1/6do1_SAR-ILE_H_1-8/6do1_SAR-ILE_H_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6do1/6do1_SAR-ILE_H_1-8/6do1_SAR-ILE_H_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6do1/6do1_SAR-ILE_H_1-8/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6do1/6do1_SAR-ILE_H_1-8/6do1_SAR-ILE_H_1-8_complex_UNL.pdb'), ('6dql_MET-LEU_C_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dql/6dql_MET-LEU_C_1-8/6dql_MET-LEU_C_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dql/6dql_MET-LEU_C_1-8/6dql_MET-LEU_C_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dql/6dql_MET-LEU_C_1-8/6dql_MET-LEU_C_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dql/6dql_MET-LEU_C_1-8/6dql_MET-LEU_C_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dql/6dql_MET-LEU_C_1-8/interaction.xml', 'MET', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6dql/6dql_MET-LEU_C_1-8/6dql_MET-LEU_C_1-8_complex_UNL.pdb'), ('6e83_ALA-GLY_A_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e83/6e83_ALA-GLY_A_1-12/6e83_ALA-GLY_A_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e83/6e83_ALA-GLY_A_1-12/6e83_ALA-GLY_A_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e83/6e83_ALA-GLY_A_1-12/6e83_ALA-GLY_A_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e83/6e83_ALA-GLY_A_1-12/6e83_ALA-GLY_A_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e83/6e83_ALA-GLY_A_1-12/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e83/6e83_ALA-GLY_A_1-12/6e83_ALA-GLY_A_1-12_complex_UNL.pdb'), ('6e86_ALA-ARG_A_1-8', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e86/6e86_ALA-ARG_A_1-8/6e86_ALA-ARG_A_1-8_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e86/6e86_ALA-ARG_A_1-8/6e86_ALA-ARG_A_1-8_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e86/6e86_ALA-ARG_A_1-8/6e86_ALA-ARG_A_1-8_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e86/6e86_ALA-ARG_A_1-8/6e86_ALA-ARG_A_1-8_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e86/6e86_ALA-ARG_A_1-8/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6e86/6e86_ALA-ARG_A_1-8/6e86_ALA-ARG_A_1-8_complex_UNL.pdb'), ('6eh2_THR-ILE_B_10-27', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eh2/6eh2_THR-ILE_B_10-27/6eh2_THR-ILE_B_10-27_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eh2/6eh2_THR-ILE_B_10-27/6eh2_THR-ILE_B_10-27_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eh2/6eh2_THR-ILE_B_10-27/6eh2_THR-ILE_B_10-27_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eh2/6eh2_THR-ILE_B_10-27/6eh2_THR-ILE_B_10-27_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eh2/6eh2_THR-ILE_B_10-27/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eh2/6eh2_THR-ILE_B_10-27/6eh2_THR-ILE_B_10-27_complex_UNL.pdb'), ('6eqa_ALA-VAL_C_2-10', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eqa/6eqa_ALA-VAL_C_2-10/6eqa_ALA-VAL_C_2-10_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eqa/6eqa_ALA-VAL_C_2-10/6eqa_ALA-VAL_C_2-10_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eqa/6eqa_ALA-VAL_C_2-10/6eqa_ALA-VAL_C_2-10_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eqa/6eqa_ALA-VAL_C_2-10/6eqa_ALA-VAL_C_2-10_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eqa/6eqa_ALA-VAL_C_2-10/interaction.xml', 'ALA', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6eqa/6eqa_ALA-VAL_C_2-10/6eqa_ALA-VAL_C_2-10_complex_UNL.pdb'), ('6erv_THR-ILE_B_10-27', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erv/6erv_THR-ILE_B_10-27/6erv_THR-ILE_B_10-27_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erv/6erv_THR-ILE_B_10-27/6erv_THR-ILE_B_10-27_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erv/6erv_THR-ILE_B_10-27/6erv_THR-ILE_B_10-27_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erv/6erv_THR-ILE_B_10-27/6erv_THR-ILE_B_10-27_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erv/6erv_THR-ILE_B_10-27/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erv/6erv_THR-ILE_B_10-27/6erv_THR-ILE_B_10-27_complex_UNL.pdb'), ('6erw_THR-ILE_D_9-26', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erw/6erw_THR-ILE_D_9-26/6erw_THR-ILE_D_9-26_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erw/6erw_THR-ILE_D_9-26/6erw_THR-ILE_D_9-26_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erw/6erw_THR-ILE_D_9-26/6erw_THR-ILE_D_9-26_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erw/6erw_THR-ILE_D_9-26/6erw_THR-ILE_D_9-26_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erw/6erw_THR-ILE_D_9-26/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6erw/6erw_THR-ILE_D_9-26/6erw_THR-ILE_D_9-26_complex_UNL.pdb'), ('6evm_PRO-PRO_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evm/6evm_PRO-PRO_C_1-9/6evm_PRO-PRO_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evm/6evm_PRO-PRO_C_1-9/6evm_PRO-PRO_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evm/6evm_PRO-PRO_C_1-9/6evm_PRO-PRO_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evm/6evm_PRO-PRO_C_1-9/6evm_PRO-PRO_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evm/6evm_PRO-PRO_C_1-9/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evm/6evm_PRO-PRO_C_1-9/6evm_PRO-PRO_C_1-9_complex_UNL.pdb'), ('6evn_PRO-GLY_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evn/6evn_PRO-GLY_C_1-9/6evn_PRO-GLY_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evn/6evn_PRO-GLY_C_1-9/6evn_PRO-GLY_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evn/6evn_PRO-GLY_C_1-9/6evn_PRO-GLY_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evn/6evn_PRO-GLY_C_1-9/6evn_PRO-GLY_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evn/6evn_PRO-GLY_C_1-9/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evn/6evn_PRO-GLY_C_1-9/6evn_PRO-GLY_C_1-9_complex_UNL.pdb'), ('6evo_PRO-GLY_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evo/6evo_PRO-GLY_C_1-9/6evo_PRO-GLY_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evo/6evo_PRO-GLY_C_1-9/6evo_PRO-GLY_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evo/6evo_PRO-GLY_C_1-9/6evo_PRO-GLY_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evo/6evo_PRO-GLY_C_1-9/6evo_PRO-GLY_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evo/6evo_PRO-GLY_C_1-9/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6evo/6evo_PRO-GLY_C_1-9/6evo_PRO-GLY_C_1-9_complex_UNL.pdb'), ('6f0y_LEU-LEU_B_419-433', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6f0y/6f0y_LEU-LEU_B_419-433/6f0y_LEU-LEU_B_419-433_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6f0y/6f0y_LEU-LEU_B_419-433/6f0y_LEU-LEU_B_419-433_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6f0y/6f0y_LEU-LEU_B_419-433/6f0y_LEU-LEU_B_419-433_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6f0y/6f0y_LEU-LEU_B_419-433/6f0y_LEU-LEU_B_419-433_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6f0y/6f0y_LEU-LEU_B_419-433/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6f0y/6f0y_LEU-LEU_B_419-433/6f0y_LEU-LEU_B_419-433_complex_UNL.pdb'), ('6fdp_ASP-ASP_B_724-732', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdp/6fdp_ASP-ASP_B_724-732/6fdp_ASP-ASP_B_724-732_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdp/6fdp_ASP-ASP_B_724-732/6fdp_ASP-ASP_B_724-732_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdp/6fdp_ASP-ASP_B_724-732/6fdp_ASP-ASP_B_724-732_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdp/6fdp_ASP-ASP_B_724-732/6fdp_ASP-ASP_B_724-732_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdp/6fdp_ASP-ASP_B_724-732/interaction.xml', 'ASP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdp/6fdp_ASP-ASP_B_724-732/6fdp_ASP-ASP_B_724-732_complex_UNL.pdb'), ('6fdt_SER-ASP_B_638-646', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdt/6fdt_SER-ASP_B_638-646/6fdt_SER-ASP_B_638-646_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdt/6fdt_SER-ASP_B_638-646/6fdt_SER-ASP_B_638-646_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdt/6fdt_SER-ASP_B_638-646/6fdt_SER-ASP_B_638-646_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdt/6fdt_SER-ASP_B_638-646/6fdt_SER-ASP_B_638-646_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdt/6fdt_SER-ASP_B_638-646/interaction.xml', 'SER', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6fdt/6fdt_SER-ASP_B_638-646/6fdt_SER-ASP_B_638-646_complex_UNL.pdb'), ('6gbe_MET-CYS_B_120-131', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gbe/6gbe_MET-CYS_B_120-131/6gbe_MET-CYS_B_120-131_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gbe/6gbe_MET-CYS_B_120-131/6gbe_MET-CYS_B_120-131_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gbe/6gbe_MET-CYS_B_120-131/6gbe_MET-CYS_B_120-131_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gbe/6gbe_MET-CYS_B_120-131/6gbe_MET-CYS_B_120-131_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gbe/6gbe_MET-CYS_B_120-131/interaction.xml', 'MET', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gbe/6gbe_MET-CYS_B_120-131/6gbe_MET-CYS_B_120-131_complex_UNL.pdb'), ('6gy5_LEU-ASN_U_1334-1344', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gy5/6gy5_LEU-ASN_U_1334-1344/6gy5_LEU-ASN_U_1334-1344_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gy5/6gy5_LEU-ASN_U_1334-1344/6gy5_LEU-ASN_U_1334-1344_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gy5/6gy5_LEU-ASN_U_1334-1344/6gy5_LEU-ASN_U_1334-1344_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gy5/6gy5_LEU-ASN_U_1334-1344/6gy5_LEU-ASN_U_1334-1344_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gy5/6gy5_LEU-ASN_U_1334-1344/interaction.xml', 'LEU', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6gy5/6gy5_LEU-ASN_U_1334-1344/6gy5_LEU-ASN_U_1334-1344_complex_UNL.pdb'), ('6hy2_TRP-ARG_A_1-12', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6hy2/6hy2_TRP-ARG_A_1-12/6hy2_TRP-ARG_A_1-12_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6hy2/6hy2_TRP-ARG_A_1-12/6hy2_TRP-ARG_A_1-12_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6hy2/6hy2_TRP-ARG_A_1-12/6hy2_TRP-ARG_A_1-12_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6hy2/6hy2_TRP-ARG_A_1-12/6hy2_TRP-ARG_A_1-12_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6hy2/6hy2_TRP-ARG_A_1-12/interaction.xml', 'TRP', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6hy2/6hy2_TRP-ARG_A_1-12/6hy2_TRP-ARG_A_1-12_complex_UNL.pdb'), ('6ms1_GLY-VAL_D_145-152', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ms1/6ms1_GLY-VAL_D_145-152/6ms1_GLY-VAL_D_145-152_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ms1/6ms1_GLY-VAL_D_145-152/6ms1_GLY-VAL_D_145-152_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ms1/6ms1_GLY-VAL_D_145-152/6ms1_GLY-VAL_D_145-152_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ms1/6ms1_GLY-VAL_D_145-152/6ms1_GLY-VAL_D_145-152_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ms1/6ms1_GLY-VAL_D_145-152/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6ms1/6ms1_GLY-VAL_D_145-152/6ms1_GLY-VAL_D_145-152_complex_UNL.pdb'), ('6mt4_PHE-PHE_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt4/6mt4_PHE-PHE_C_1-9/6mt4_PHE-PHE_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt4/6mt4_PHE-PHE_C_1-9/6mt4_PHE-PHE_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt4/6mt4_PHE-PHE_C_1-9/6mt4_PHE-PHE_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt4/6mt4_PHE-PHE_C_1-9/6mt4_PHE-PHE_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt4/6mt4_PHE-PHE_C_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt4/6mt4_PHE-PHE_C_1-9/6mt4_PHE-PHE_C_1-9_complex_UNL.pdb'), ('6mt5_PHE-PHE_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt5/6mt5_PHE-PHE_C_1-9/6mt5_PHE-PHE_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt5/6mt5_PHE-PHE_C_1-9/6mt5_PHE-PHE_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt5/6mt5_PHE-PHE_C_1-9/6mt5_PHE-PHE_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt5/6mt5_PHE-PHE_C_1-9/6mt5_PHE-PHE_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt5/6mt5_PHE-PHE_C_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt5/6mt5_PHE-PHE_C_1-9/6mt5_PHE-PHE_C_1-9_complex_UNL.pdb'), ('6mt6_PHE-PHE_B_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt6/6mt6_PHE-PHE_B_1-9/6mt6_PHE-PHE_B_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt6/6mt6_PHE-PHE_B_1-9/6mt6_PHE-PHE_B_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt6/6mt6_PHE-PHE_B_1-9/6mt6_PHE-PHE_B_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt6/6mt6_PHE-PHE_B_1-9/6mt6_PHE-PHE_B_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt6/6mt6_PHE-PHE_B_1-9/interaction.xml', 'PHE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mt6/6mt6_PHE-PHE_B_1-9/6mt6_PHE-PHE_B_1-9_complex_UNL.pdb'), ('6mtu_PRO-LEU_D_822-829', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mtu/6mtu_PRO-LEU_D_822-829/6mtu_PRO-LEU_D_822-829_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mtu/6mtu_PRO-LEU_D_822-829/6mtu_PRO-LEU_D_822-829_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mtu/6mtu_PRO-LEU_D_822-829/6mtu_PRO-LEU_D_822-829_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mtu/6mtu_PRO-LEU_D_822-829/6mtu_PRO-LEU_D_822-829_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mtu/6mtu_PRO-LEU_D_822-829/interaction.xml', 'PRO', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6mtu/6mtu_PRO-LEU_D_822-829/6mtu_PRO-LEU_D_822-829_complex_UNL.pdb'), ('6o9b_THR-LYS_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9b/6o9b_THR-LYS_C_1-9/6o9b_THR-LYS_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9b/6o9b_THR-LYS_C_1-9/6o9b_THR-LYS_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9b/6o9b_THR-LYS_C_1-9/6o9b_THR-LYS_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9b/6o9b_THR-LYS_C_1-9/6o9b_THR-LYS_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9b/6o9b_THR-LYS_C_1-9/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9b/6o9b_THR-LYS_C_1-9/6o9b_THR-LYS_C_1-9_complex_UNL.pdb'), ('6o9c_THR-LYS_C_1-9', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9c/6o9c_THR-LYS_C_1-9/6o9c_THR-LYS_C_1-9_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9c/6o9c_THR-LYS_C_1-9/6o9c_THR-LYS_C_1-9_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9c/6o9c_THR-LYS_C_1-9/6o9c_THR-LYS_C_1-9_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9c/6o9c_THR-LYS_C_1-9/6o9c_THR-LYS_C_1-9_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9c/6o9c_THR-LYS_C_1-9/interaction.xml', 'THR', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6o9c/6o9c_THR-LYS_C_1-9/6o9c_THR-LYS_C_1-9_complex_UNL.pdb'), ('6q36_ACE-NH2_D_1-17', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q36/6q36_ACE-NH2_D_1-17/6q36_ACE-NH2_D_1-17_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q36/6q36_ACE-NH2_D_1-17/6q36_ACE-NH2_D_1-17_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q36/6q36_ACE-NH2_D_1-17/6q36_ACE-NH2_D_1-17_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q36/6q36_ACE-NH2_D_1-17/6q36_ACE-NH2_D_1-17_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q36/6q36_ACE-NH2_D_1-17/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q36/6q36_ACE-NH2_D_1-17/6q36_ACE-NH2_D_1-17_complex_UNL.pdb'), ('6q3q_GLY-ASP_b_0-7', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q3q/6q3q_GLY-ASP_b_0-7/6q3q_GLY-ASP_b_0-7_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q3q/6q3q_GLY-ASP_b_0-7/6q3q_GLY-ASP_b_0-7_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q3q/6q3q_GLY-ASP_b_0-7/6q3q_GLY-ASP_b_0-7_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q3q/6q3q_GLY-ASP_b_0-7/6q3q_GLY-ASP_b_0-7_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q3q/6q3q_GLY-ASP_b_0-7/interaction.xml', 'GLY', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6q3q/6q3q_GLY-ASP_b_0-7/6q3q_GLY-ASP_b_0-7_complex_UNL.pdb'), ('6seo_ACE-NH2_L_219-236', 'pdbbind_opt_poly/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6seo/6seo_ACE-NH2_L_219-236/6seo_ACE-NH2_L_219-236_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6seo/6seo_ACE-NH2_L_219-236/6seo_ACE-NH2_L_219-236_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6seo/6seo_ACE-NH2_L_219-236/6seo_ACE-NH2_L_219-236_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6seo/6seo_ACE-NH2_L_219-236/6seo_ACE-NH2_L_219-236_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6seo/6seo_ACE-NH2_L_219-236/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_poly/6seo/6seo_ACE-NH2_L_219-236/6seo_ACE-NH2_L_219-236_complex_UNL.pdb'), ('1htf_G26_A_100', 'pdbbind_opt_sm/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/1htf/1htf_G26_A_100/1htf_G26_A_100_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/1htf/1htf_G26_A_100/1htf_G26_A_100_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/1htf/1htf_G26_A_100/1htf_G26_A_100_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/1htf/1htf_G26_A_100/1htf_G26_A_100_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/1htf/1htf_G26_A_100/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/1htf/1htf_G26_A_100/1htf_G26_A_100_complex_UNL.pdb'), ('2fv9_002_B_2', 'pdbbind_opt_sm/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2fv9/2fv9_002_B_2/2fv9_002_B_2_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2fv9/2fv9_002_B_2/2fv9_002_B_2_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2fv9/2fv9_002_B_2/2fv9_002_B_2_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2fv9/2fv9_002_B_2/2fv9_002_B_2_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2fv9/2fv9_002_B_2/interaction.xml', 'ILE', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2fv9/2fv9_002_B_2/2fv9_002_B_2_complex_UNL.pdb'), ('2opy_CO9_A_101', 'pdbbind_opt_sm/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2opy/2opy_CO9_A_101/2opy_CO9_A_101_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2opy/2opy_CO9_A_101/2opy_CO9_A_101_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2opy/2opy_CO9_A_101/2opy_CO9_A_101_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2opy/2opy_CO9_A_101/2opy_CO9_A_101_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2opy/2opy_CO9_A_101/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/2opy/2opy_CO9_A_101/2opy_CO9_A_101_complex_UNL.pdb'), ('3l3n_LSW_A_1', 'pdbbind_opt_sm/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3l3n/3l3n_LSW_A_1/3l3n_LSW_A_1_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3l3n/3l3n_LSW_A_1/3l3n_LSW_A_1_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3l3n/3l3n_LSW_A_1/3l3n_LSW_A_1_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3l3n/3l3n_LSW_A_1/3l3n_LSW_A_1_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3l3n/3l3n_LSW_A_1/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3l3n/3l3n_LSW_A_1/3l3n_LSW_A_1_complex_UNL.pdb'), ('3o9p_MHI_A_516', 'pdbbind_opt_sm/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3o9p/3o9p_MHI_A_516/3o9p_MHI_A_516_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3o9p/3o9p_MHI_A_516/3o9p_MHI_A_516_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3o9p/3o9p_MHI_A_516/3o9p_MHI_A_516_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3o9p/3o9p_MHI_A_516/3o9p_MHI_A_516_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3o9p/3o9p_MHI_A_516/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/3o9p/3o9p_MHI_A_516/3o9p_MHI_A_516_complex_UNL.pdb'), ('5d7r_57Y_A_306', 'pdbbind_opt_sm/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/5d7r/5d7r_57Y_A_306/5d7r_57Y_A_306_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/5d7r/5d7r_57Y_A_306/5d7r_57Y_A_306_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/5d7r/5d7r_57Y_A_306/5d7r_57Y_A_306_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/5d7r/5d7r_57Y_A_306/5d7r_57Y_A_306_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/5d7r/5d7r_57Y_A_306/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/5d7r/5d7r_57Y_A_306/5d7r_57Y_A_306_complex_UNL.pdb'), ('6qxj_JKQ_A_401', 'pdbbind_opt_sm/', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/6qxj/6qxj_JKQ_A_401/6qxj_JKQ_A_401_ligand_refined.sdf', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/6qxj/6qxj_JKQ_A_401/6qxj_JKQ_A_401_ligand_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/6qxj/6qxj_JKQ_A_401/6qxj_JKQ_A_401_protein_refined.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/6qxj/6qxj_JKQ_A_401/6qxj_JKQ_A_401_complex.pdb', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/6qxj/6qxj_JKQ_A_401/interaction.xml', 'UNK', '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset/pdbbind_opt_sm/6qxj/6qxj_JKQ_A_401/6qxj_JKQ_A_401_complex_UNL.pdb')]

## Fragment Protein

### Load Halogen Interactions, Generate Halogen Directory Args

In [2]:
halogen_dir = os.getcwd()
dataset_dir = '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset'
subdirs = ['biolip2_opt/',
            'pdbbind_opt_poly/',
            'pdbbind_opt_sm/']

df = pd.read_csv("halogen_bond.csv", index_col=0)

halogens = []

for index, row in tqdm(df.iterrows()):
    halogens.append((
        row['PDBID'], 
        row['full_PDBID'],
        row['subdir'],
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/{row['full_PDBID']}_complex.pdb"),
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/interaction.xml"),
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/HalogenProteinFragment"),
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/HalogenLigandFragment"), 
        os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/{row['full_PDBID']}_ligand_refined.sdf"),
        row['don_idx'], 
        index
    ))

1318it [00:00, 21339.62it/s]


### Protein Fragmentation

In [63]:
FRAG_SIDE_CHAIN = {
    'ASP': ['OD1', 'OD2', 'CG', 'CB', 'HB2', 'HB3'],
    'GLU': ['OE1', 'OE2', 'CD', 'CG', 'HG2', 'HG3'],
    'LYS': ['NZ', 'HZ1', 'HZ2', 'HZ3', 'CE', 'HE2', 'HE3'],
    'ARG': ['NH1', 'HH11', 'HH12', 'NH2', 'HH21', 'HH22', 'CZ', 'NE', 'HE1', 'CD', 'HD2', 'HD3'],
    'TYR': ['OH', 'HH', 'CZ', 'CE1', 'HE1', 'CE2', 'HE2', 'CD1', 'HD1', 'CD2', 'HD2', 'CG'],
    'TRP': ['CH2', 'HH2', 'CZ2', 'HZ2', 'CZ3', 'HZ3', 'NE1', 'HE1', 'CE2', 'CE3', 'HE3', 'CD1', 'HD1', 'CD2', 'CG'],
    'SER': ['OG', 'HG', 'CB', 'HB2', 'HB3'],
    'THR': ['OG1', 'HG1', 'CB', 'HB'],
    'HIS': ['CE1', 'HE1', 'NE2', 'HE2', 'ND1', 'HD1', 'CD2', 'HD2', 'CG'],
    'HIE': ['CE1', 'HE1', 'NE2', 'HE2', 'ND1', 'CD2', 'HD2', 'CG'],
    'HID': ['CE1', 'HE1', 'NE2', 'ND1', 'HD1', 'CD2', 'HD2', 'CG'],
    'ASN': ['OD1', 'ND2', 'HD21', 'HD22', 'CG', 'CB', 'HB2', 'HB3'],
    'GLN': ['OE1', 'NE2', 'HE21', 'HE22', 'CD', 'CG', 'HG2', 'HG3'],
    'CYS': ['SG', 'HG', 'CB', 'HB2', 'HB3']
}
# check if CYS code is correct using pymol


def get_residue_fragment_mask(
    struct, resnr, restype, reschain, sidechain, acceptortype=None
):
    residues = struct[f':{resnr}'].residues
    assert len(residues) == 1, "More than one residues are selected"
    assert residues[0].name == restype, f"Selected residue is not {restype}"
    assert residues[0].chain == reschain, f"Selected residue does not in chain {reschain}"

    # data type conversion
    if sidechain == 'False':
        sidechain = False

    resnr = int(resnr)

    if sidechain:
        mask = f':{resnr}@{",".join(FRAG_SIDE_CHAIN[restype])}'
    else:
        # Ng+ exclusive for Arg SC
        # N3+ exclusice for LYS SC
        assert acceptortype in ['O2', 'O3', 'O.co2', 'Nam']
        if acceptortype == 'Nam':
            mask = f':{resnr-1}@C,O,CA,HA* | :{resnr}@N,H,CA,HA*' # check what H is
        else:
            if acceptortype == 'O.co2':
                # Terminal O
                mask = f':{resnr}@C,O,OXT,CA,HA*' 
            else:
                if struct.residues[resnr].name == 'PRO': # next residue is a Proline
                    mask = f':{resnr}@C,O,CA,HA* | :{resnr+1}@N,CD,HD2,HD3,CA,HA*' 
                else:
                    mask = f':{resnr}@C,O,CA,HA* | :{resnr+1}@N,H,CA,HA*'
    return mask


def struct_to_pdb_string(struct):
    strio = StringIO()
    struct.save(strio, format='pdb')
    strio.seek(0)
    return strio.read()


def add_hydrogen(struct, output_pdb):
    obConversion = openbabel.OBConversion()
    obConversion.SetInAndOutFormats("pdb", "pdb")
    
    ob_mol = openbabel.OBMol()
    obConversion.ReadString(ob_mol, struct_to_pdb_string(struct))
    
    resnames = set()
    atnames = []
    for atom in openbabel.OBMolAtomIter(ob_mol):
        residue = atom.GetResidue()
        resname = residue.GetName()
        atname = residue.GetAtomID(atom).strip()
        atnames.append(atname)
        resnames.add(resname)
    resnames = list(resnames)
    
    if len(resnames) == 1 and resnames[0] in ['GLU', 'ASP']:
        for atom in openbabel.OBMolAtomIter(ob_mol):
            atname = residue.GetAtomID(atom).strip()
            if atname == 'OD1' or atname == 'OE1':
                atom.SetFormalCharge(0)
                atom.SetImplicitHCount(0)
            if atname == 'OD2' or atname == 'OE2':
                atom.SetFormalCharge(-1)
                atom.SetImplicitHCount(0)
    elif 'OXT' in atnames:
        for atom in openbabel.OBMolAtomIter(ob_mol):
            atname = residue.GetAtomID(atom).strip()
            if atname == 'O':
                atom.SetFormalCharge(0)
                atom.SetImplicitHCount(0)
            if atname == 'OXT':
                atom.SetFormalCharge(-1)
                atom.SetImplicitHCount(0)
        
    ob_mol.AddHydrogens()
    obConversion.WriteFile(ob_mol, output_pdb)

In [79]:
warnings.filterwarnings("ignore")

def wrapper_prot_frag(halogens):
    struct = parmed.load_file(halogens[3])
    infos = extract_halogen(halogens[4])
    
    errs = []
    for interaction in infos:
        try:
            mask = get_residue_fragment_mask(
                struct,                              
                interaction['resnr'], interaction['restype'], interaction['reschain'], 
                interaction['sidechain'], interaction['acceptortype']
            )
            if eval(interaction['sidechain']):
                ifsc = 'SC'
            else:
                ifsc = 'BB'
                
            frag_name = '{}_{}_{}_{}_{}_{}.pdb'.format(
                halogens[0],
                interaction['resnr'], interaction['restype'],
                ifsc,
                interaction['don_idx'], interaction['acc_idx']
            )
            frag_path = os.path.join(halogens[5], frag_name)
            add_hydrogen(struct[mask], frag_path)
        except Exception as e:
            errs.append((halogens[1], halogens[2], interaction, str(e)))
    return errs
    


if __name__ == '__main__':
    
    reset = True
    for arg in tqdm(halogens):
        if reset and os.path.isdir(arg[5]):
            shutil.rmtree(arg[5])
            
        if not os.path.isdir(arg[5]):
            os.mkdir(arg[5])

    errs = []
    with mp.Pool(cpu_count()) as pool:
        errs = list(tqdm(pool.imap(wrapper_prot_frag, halogens), total=len(halogens)))
    
    for err in errs:
        if len(err) > 0:
            print(err)
    
    # will get [('5ltn', 'Bad backbone N type: O3')]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1320/1320 [01:45<00:00, 12.50it/s]

[('2cf8_ESH_H_1246', 'pdbbind_opt_sm/', {'resnr': '0', 'restype': 'GLU', 'reschain': 'H', 'resnr_lig': '1', 'restype_lig': 'UNL', 'reschain_lig': 'Z', 'sidechain': 'False', 'dist': '3.22', 'don_angle': '148.14', 'acc_angle': '128.77', 'don_idx': '32', 'donortype': 'Cl', 'acc_idx': '1633', 'acceptortype': 'O2', 'ligcoo_x': '25.673', 'ligcoo_y': '-14.097', 'ligcoo_z': '27.312', 'protcoo_x': '22.648', 'protcoo_y': '-13.883', 'protcoo_z': '26.223'}, 'More than one residues are selected')]
[('2cn0_F25_H_1246', 'pdbbind_opt_sm/', {'resnr': '0', 'restype': 'GLU', 'reschain': 'H', 'resnr_lig': '1', 'restype_lig': 'UNL', 'reschain_lig': 'Z', 'sidechain': 'False', 'dist': '3.17', 'don_angle': '137.97', 'acc_angle': '123.33', 'don_idx': '9', 'donortype': 'F', 'acc_idx': '1627', 'acceptortype': 'O2', 'ligcoo_x': '25.377', 'ligcoo_y': '-14.086', 'ligcoo_z': '27.092', 'protcoo_x': '22.366', 'protcoo_y': '-13.566', 'protcoo_z': '26.257'}, 'More than one residues are selected')]


In [141]:
print(f"Number of PDBs having halogen bonds: {len(df['PDBID'].unique())}")

Number of PDBs having halogen bonds: 1190


## Fragment Ligands

In [4]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from tqdm import tqdm
import sys
sys.path.insert(0, '../')
from edautils.fragment import get_fg_set, select_fragment, write_sdf

In [6]:
frags = []
for arg in tqdm(halogens):
    mol = Chem.SDMolSupplier(arg[7], removeHs=False, sanitize=True)[0]
    idx = mol.GetAtomWithIdx(int(arg[8]) - 1).GetNeighbors()[0].GetIdx()
    fgs = get_fg_set(mol, False)
    frag = select_fragment(mol, fgs[idx])
    frag.SetProp("identifier", arg[9])

    if not os.path.isdir(arg[6]):
        os.mkdir(arg[6])
    write_sdf(frag, os.path.join(arg[6], f'{arg[9]}.sdf'))

    frag_no_h = Chem.RemoveHs(frag)
    AllChem.Compute2DCoords(frag_no_h)
    frags.append(frag_no_h)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1318/1318 [00:09<00:00, 140.40it/s]


### Combine Similiar Ligand Frag with Fingerprints

In [11]:
from rdkit.Chem import DataStructs
from rdkit.Chem.Fingerprints import FingerprintMols
from rdkit.DataStructs import TanimotoSimilarity

fingerprints = [FingerprintMols.FingerprintMol(frag) for frag in frags]

similarity = []
for i, fp1 in tqdm(enumerate(fingerprints), total=len(fingerprints)):
    row = []
    for j, fp2 in enumerate(fingerprints):
        sim = TanimotoSimilarity(fp1, fp2)
        row.append(sim)
    similarity.append(row)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1318/1318 [00:08<00:00, 148.81it/s]


In [39]:
threshold = 1.0

uniqueIndices = []
seen = set()

for i in range(len(fingerprints)):
    if i not in seen:
        uniqueIndices.append(i)
        seen.update(j for j, sim in enumerate(similarity[i]) if sim >= threshold)

uniquefps = [fingerprints[i] for i in selected_indices]
uniqueFragsFP = [frags[i] for i in selected_indices]

print(len(uniqueFragsFP))

# Draw.MolsToGridImage(uniqueFrags, legends=[f.GetProp('identifier') for f in uniqueFrags], subImgSize=(300, 300), molsPerRow=8, maxMols=600)

593


### Combine Identical Ligands Frag with Smiles String

In [40]:
smiles = {index: Chem.MolToSmiles(frag, canonical=True) for index, frag in enumerate(frags)}

uniqueSmiles = set(smiles.values())
uniqueIndex = {smiles: index for index, smiles in smiles.items()}
uniqueFragsSM = [frags[index] for index in uniqueIndex.values()]
print(len(uniqueFragsSM))

Draw.MolsToGridImage(uniqueFrags, legends=[f.GetProp('identifier') for f in uniqueFragsSM], subImgSize=(300, 300), molsPerRow=8, maxMols=600)

598


# Collect to output

In [67]:
from edautils.fragment import read_pdb, extract_info

In [97]:
errPDBID = []

for err in errs:
    if len(err) > 0:
        errPDBID.append(err[0][0])

print(errPDBID)

valid_prot_smiles = {
    'C[NH2+]C',
    'C1=CC2=C(C=C1)NC=C2',
    'C1=CNC=N1',
    'CC(=O)N(C)C',
    'CC(=O)[O-]',
    'CC(N)=O',
    'CNC(C)=O',
    'CNC(N)=[NH2+]',
    'CO',
    'C[NH3+]',
    'OC1=CC=CC=C1',
    "CS"
}

for index, row in df.iterrows():
    PDBID = row['full_PDBID']
    if PDBID in errPDBID:
        print(PDBID)
        df.drop(index, inplace=True)
    else:
        prot_frag_dir = os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/HalogenProteinFragment")
        lig_frag_dir = os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/HalogenLigandFragment")
            
        prot_frag_file = os.path.join(prot_frag_dir, f'{index}.pdb')
        lig_frag_file = os.path.join(lig_frag_dir, f'{index}.sdf')
    
        lig_frag = Chem.SDMolSupplier(lig_frag_file, removeHs=False)[0]
        prot_frag = read_pdb(prot_frag_file)
    
        lig_info = extract_info(lig_frag)
        prot_info = extract_info(prot_frag)
        
        assert prot_info['smiles'] in valid_prot_smiles, f'Invalid protin fragment: {prot_info["smiles"]}'
    
        df.loc[index, 'natoms0'] = lig_info['natoms']
        df.loc[index, 'charge0'] = lig_info['charge']
        df.loc[index, 'smiles0'] = lig_info['smiles']
        df.loc[index, 'natoms1'] = prot_info['natoms']
        df.loc[index, 'charge1'] = prot_info['charge']
        df.loc[index, 'smiles1'] = prot_info['smiles']
        df.loc[index, 'elements'] = ' '.join(lig_info['elements'] + prot_info['elements'])
        df.loc[index, 'xyz'] = ' '.join([f'{x:.3f}' for x in lig_info['xyz'].flatten()] + \
            [f'{x:.3f}' for x in prot_info['xyz'].flatten()])

df.to_csv('halogen_bond.csv')

['2cf8_ESH_H_1246', '2cn0_F25_H_1246']
2cf8_ESH_H_1246
2cn0_F25_H_1246
